# VAE DLR Done

In [1]:
#!pip install optuna
# pip install torchmetrics
# torchrun --nproc_per_node=N train.py

import optuna
#from V2_0_BasicCode_papermill import *
from functools import partial




import os, glob, random
import numpy as np
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
import cv2


import os, numpy as np
from tqdm import tqdm
import optuna

import time
import functools

## Folders

In [2]:
from pathlib import Path
import os


data_source = Path(
    r'C:\Users\kec994\OneDrive - The University of Texas-Rio Grande Valley\PhD\Data_Source'
)


train_time = {} # Data from time decorator
fruits_train = os.path.join(data_source , 'Fruits' , 'train')
fruits_val   = os.path.join(data_source , 'Fruits' , 'val')


fruits_train = Path(fruits_train).resolve()
fruits_val   = Path(fruits_val).resolve()

train_dir_default = fruits_train
val_dir_default = fruits_val

#train_dir_default = train_y_dir_default
#val_dir_default = val_y_dir_default


folders = ['DIV2K_train_LR_bicubic_X4', 'DIV2K_train_LR_unknown_X4', 'DIV2K_valid_LR_bicubic_X4', 'DIV2K_valid_LR_unknown_X4', ]

for index, val in enumerate(folders):
    folders[index] = os.path.join(data_source , 'Div2k' , val)
    


train_x_dir_default = folders[1]
train_y_dir_default = folders[0] 
val_x_dir_default = folders[3] 
val_y_dir_default = folders[2] 




# DLR
dlr_folders = ['optuna_studies_dlr_vae', 'optuna_studies_dlr_fcn',]
db_files = ['study_charbonnier.db','study_ms_ssim.db','study_psnr.db','study_mse.db']
dlr_vae_models = ['best_dlr_VAE_charbonnier','best_dlr_VAE_ms_ssim','best_dlr_VAE_psnr','best_dlr_VAE_mse']
dlr_fcn_models = ['best_dlr_FCN_charbonnier', 'best_dlr_FCN_ms_ssim']


dlr_models = []
for i in dlr_vae_models:
    folder_path_ = os.path.join(dlr_folders[0],i)
    dlr_models.append(folder_path_)


for i in dlr_fcn_models:
    folder_path_ = os.path.join(dlr_folders[1],i)
    dlr_models.append(folder_path_)

for index,val in enumerate(dlr_models):
    dlr_models[index] = os.path.join(val,'best.pt')


dlr_optuna_db = []
for name in db_files:
    file_ = os.path.join(dlr_folders[0],name)
    dlr_optuna_db.append(file_)


for index, name in enumerate(db_files):
    file_ = os.path.join(dlr_folders[1],name)
    dlr_optuna_db.append(file_)
    if index==1:
        break


#print(dlr_models,'\n')








# Model_With_DLR ***************************************************************************
# DLR
model_folders = ['optuna_studies_vae_fruits', 'optuna_studies_fcn_fruits',
                 'optuna_studies_vae_div2k', 'optuna_studies_fcn_div2k',
                 'optuna_studies_vae_div2k_dlr', 'optuna_studies_fcn_div2k_dlr',]

db_files = ['study_charbonnier.db','study_ms_ssim.db','study_psnr.db','study_mse.db', 'study_charbonnier_ms_ssim.db', 'study_mse_ms_ssim.db']
study_names = ['study_charbonnier','study_ms_ssim','study_psnr','study_mse', 'study_charbonnier_ms_ssim', 'study_mse_ms_ssim']
best_model_name = 'best.pt'
#best_model_folders = ['study_charbonnier','study_ms_ssim','study_psnr','study_mse']
#study_names = ['charbonnier_study', 'ms_ssim_study', 'psnr_study', 'mse_study' ]
#best_model_folders = ['charbonnier_study', 'ms_ssim_study', 'psnr_study', 'mse_study' ]
model_w_dlr_optuna_db = []
best_models = []


# VAE Fruits
j=0
for index, name in enumerate(db_files):
    if index in [0,1,2,3,4,5]:
        file_ = os.path.join(model_folders[j],name)
        model_w_dlr_optuna_db.append(file_)
        file_ = os.path.join(model_folders[j],name[:-3],best_model_name)
        best_models.append(file_)

# FCN Fruits
j+=1
for index, name in enumerate(db_files):
    if index in [0,1,2,3,4,5]:
        file_ = os.path.join(model_folders[j],name)
        model_w_dlr_optuna_db.append(file_)
        file_ = os.path.join(model_folders[j],name[:-3],best_model_name)
        best_models.append(file_)


# VAE Div2k
j+=1
for index,name in enumerate(db_files):
    if index in [0,1,2,3,4,5]:
        file_ = os.path.join(model_folders[j],name)
        model_w_dlr_optuna_db.append(file_)
        file_ = os.path.join(model_folders[j],name[:-3],best_model_name)
        best_models.append(file_)

# FCN Div2k
j+=1
for index, name in enumerate(db_files):
    if index in [0,1,2,3,4,5]:
        file_ = os.path.join(model_folders[j],name)
        model_w_dlr_optuna_db.append(file_)
        file_ = os.path.join(model_folders[j],name[:-3],best_model_name)
        best_models.append(file_)


# VAE Div2k DLR
j+=1
for index, name in enumerate(db_files):
    if index in [0,1,2,3,4,5]:
        file_ = os.path.join(model_folders[j],name)
        model_w_dlr_optuna_db.append(file_)
        file_ = os.path.join(model_folders[j],name[:-3],best_model_name)
        best_models.append(file_)



# FCN Div2k DLR
j+=1
for index, name in enumerate(db_files):
    if index in [0,1,2,3,4,5]:
        file_ = os.path.join(model_folders[j],name)
        model_w_dlr_optuna_db.append(file_)
        file_ = os.path.join(model_folders[j],name[:-3],best_model_name)
        best_models.append(file_)



# Best Models


text_ ='sqlite:///'
# Convert to Optuna-compatible format:
model_w_dlr_optuna_db = [f"{text_}{path.replace(os.sep, '/')}" for path in model_w_dlr_optuna_db]
best_models = [f"{path.replace(os.sep, '/')}" for path in best_models]

#print(model_w_dlr_optuna_db)  
#print(best_models)
#_______________________________________________________________________________________________________



In [3]:
def get_loss_fn_coeff(std_n, dlr_model):
    """
    Returns loss coefficient dictionary for a given study index.

    std_n:
      0 → Charbonnier
      1 → MS-SSIM
      2 → PSNR
      3 → MSE
      4 → Charbonnier + MS-SSIM (Optuna-tuned)
      5 → MS-SSIM + MSE (Optuna-tuned)
    """

    # default: all losses off
    loss_fn_coeff = {
        'a': 0.0,   # Charbonnier
        'b': 0.0,   # 1-SSIM
        'c': 0.0,   # 1-MS-SSIM
        'd': 0.0,   # -PSNR
        'e': 0.0,   # MSE
        'dlr_model': dlr_model,
    }

    if std_n == 0:          # Charbonnier
        loss_fn_coeff['a'] = 1.0

    elif std_n == 1:        # MS-SSIM
        loss_fn_coeff['c'] = 1.0

    elif std_n == 2:        # PSNR
        loss_fn_coeff['d'] = 1.0

    elif std_n == 3:        # MSE
        loss_fn_coeff['e'] = 1.0

    elif std_n == 4:        # Charbonnier + MS-SSIM (Optuna)
        loss_fn_coeff['a'] = None
        loss_fn_coeff['c'] = None

    elif std_n == 5:        # MS-SSIM + MSE (Optuna)
        loss_fn_coeff['c'] = None
        loss_fn_coeff['e'] = None

    else:
        raise ValueError(f"Unknown std_n={std_n}")

    return loss_fn_coeff


# Training Parameters _______________

In [4]:
import argparse
import torch

def parse_args():
    parser = argparse.ArgumentParser(description="Training configuration")

    # --------------------------
    # Training schedule
    # --------------------------
    parser.add_argument("--epochs", type=int, default=2,
                        help="Epochs per Optuna trial (e.g., 100, 2)")
    parser.add_argument("--n-trials", type=int, default=3,
                        help="Number of Optuna trials (e.g., 50, 3)")
    parser.add_argument("--n-trials-main-model", type=int, default=2,
                        help="Trials for main model stage (e.g., 100, 2)")
    parser.add_argument("--pbar-interval", type=int, default=2,
                        help="Progress bar print interval (e.g., 50, 2)")
    parser.add_argument("--train-epochs", type=int, default=5,
                        help="Final training epochs (e.g., 500, 5)")

    # --------------------------
    # Runtime / compilation
    # --------------------------
    parser.add_argument("--compile-train", action="store_true",
                        help="Enable torch.compile for training")

    # --------------------------
    # Data handling
    # --------------------------
    parser.add_argument("--degrade-x-train", action="store_true",
                        help="Apply degradation to training inputs")
    parser.set_defaults(degrade_x_train=True)

    parser.add_argument("--num-workers", type=int, default=0,
                        help="Number of DataLoader workers")

    parser.add_argument("--keep-in-ram", action="store_true",
                        help="Keep datasets in RAM")
    parser.set_defaults(keep_in_ram=True)

    parser.add_argument("--fcn-tv-coeff", type=float, default=None,
                        help="TV coefficient for FCN (None lets Optuna tune)")

    # --------------------------
    # Pipeline control flags
    # --------------------------
    # Optuna (DLR)
    parser.add_argument("--run-optuna-vae-dlr", action="store_true")
    parser.add_argument("--run-optuna-fcn-dlr", action="store_true")

    # Training (DLR)
    parser.add_argument("--run-train-vae-dlr", action="store_true")
    parser.add_argument("--run-train-fcn-dlr", action="store_true")

    # Optuna (Main)
    parser.add_argument("--run-optuna-vae-main", action="store_true")
    parser.add_argument("--run-optuna-vae-main-dlr", action="store_true")
    parser.add_argument("--run-optuna-fcn-main", action="store_true")
    parser.add_argument("--run-optuna-fcn-main-dlr", action="store_true")

    # Training (Main)
    parser.add_argument("--run-train-vae-main", action="store_true")
    parser.add_argument("--run-train-vae-main-dlr", action="store_true")
    parser.add_argument("--run-train-fcn-main", action="store_true")
    parser.add_argument("--run-train-fcn-main-dlr", action="store_true")

    # --------------------------
    # Parallel / device mode
    # --------------------------
    parser.add_argument(
        "--run-mode",
        type=str,
        choices=["cpu", "cuda", "dp", "ddp"],
        default="cuda",
        help="Execution mode: cpu | cuda | dp | ddp",
    )

    parser.add_argument(
        "--gpu-device-ids",
        type=int,
        nargs="*",
        default=[0],
        help="GPU device IDs for DataParallel",
    )


    parser.add_argument(
        "--train-time-pkl",
        type=str,
        default="train_time",
        help="Name of time pkl",
    )

    
    # Jupyter-safe
    args, _ = parser.parse_known_args()
    return args


In [5]:
# ==========================================================
# Parse arguments
# ==========================================================
args = parse_args()

# ==========================================================
# Global config (SAFE for CPU/CUDA/DP/DDP)
# ==========================================================

epochs = args.epochs
n_trials = args.n_trials
n_trials_main_model = args.n_trials_main_model
pbar_interval = args.pbar_interval
train_epochs = args.train_epochs

m_compile_train = args.compile_train
degrade_x_train = args.degrade_x_train
n_workers = args.num_workers
data_keep_in_ram = args.keep_in_ram
fcn_tv_coeff = args.fcn_tv_coeff

RUN_MODE = args.run_mode
gpu_device_ids = args.gpu_device_ids

# Mixed precision preference (checked again inside training loops)
use_amp = torch.cuda.is_available()

# Derived flags
active_dp = (RUN_MODE == "dp")
active_ddp = (RUN_MODE == "ddp")

# Convenience
device_str = "cuda" if torch.cuda.is_available() else "cpu"

# train_time
train_time_pkl = f'{args.train_time_pkl}.pkl'


# Activate Cells
run_optuna_vae_dlr = args.run_optuna_vae_dlr
run_optuna_fcn_dlr = args.run_optuna_fcn_dlr
run_train_vae_dlr = args.run_train_vae_dlr
run_train_fcn_dlr = args.run_train_fcn_dlr

run_optuna_vae_main = args.run_optuna_vae_main
run_optuna_vae_main_dlr = args.run_optuna_vae_main_dlr
run_optuna_fcn_main = args.run_optuna_fcn_main
run_optuna_fcn_main_dlr = args.run_optuna_fcn_main_dlr

run_train_vae_main = args.run_train_vae_main
run_train_vae_main_dlr = args.run_train_vae_main_dlr
run_train_fcn_main = args.run_train_fcn_main
run_train_fcn_main_dlr = args.run_train_fcn_main_dlr




print(
    f"[Config] RUN_MODE={RUN_MODE} | "
    f"CUDA available={torch.cuda.is_available()} | "
    f"use_amp={use_amp} | "
    f"compile={m_compile_train}"
)


[Config] RUN_MODE=cuda | CUDA available=True | use_amp=True | compile=False


## Activate_Cells

In [6]:
run_optuna_vae_dlr = False
run_optuna_fcn_dlr = True
run_train_vae_dlr = False
run_train_fcn_dlr = False





run_optuna_vae_main = False
run_train_vae_main = False

run_optuna_vae_main_dlr = False
run_train_vae_main_dlr = True

run_optuna_fcn_main = False
run_train_fcn_main = False

run_optuna_fcn_main_dlr = False
run_train_fcn_main_dlr = False









temp = 'train_time_vae_dlr'
train_time_pkl = f'{temp}.pkl'


In [7]:
import gc
import torch

# ==========================================================
# Global config (SAFE for CPU/CUDA/DP/DDP)
# - Do NOT reference device_t here
# - Do NOT create GradScaler here
# ==========================================================

# Mixed precision preference (will be enabled inside train/objective functions only if device_t is CUDA)
use_amp = torch.cuda.is_available()

# Update here ************************************************
epochs = 100                  # e.g., 100
n_trials = 50                # e.g., 50
n_trials_main_model = 100     # e.g., 100
pbar_interval = 50           # e.g., 50
train_epochs = 500            # e.g., 500

m_compile_train = True     # True/False
degrade_x_train = True
n_workers = 0
data_keep_in_ram = True
fcn_tv_coeff = None         # float or None (None means trial will tune it)

# ==========================================================
# Parallel mode selection
# ==========================================================
# Choose one mode: "cpu" | "cuda" | "dp" | "ddp"
RUN_MODE = "cuda"

# DP settings (only used if RUN_MODE == "dp")
gpu_device_ids = [0]        # e.g., list(range(torch.cuda.device_count()))

# Derived flags (use these everywhere)
active_dp = (RUN_MODE == "dp")
active_ddp = (RUN_MODE == "ddp")

# Convenience (optional)
device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Config] RUN_MODE={RUN_MODE} | CUDA available={torch.cuda.is_available()} | use_amp={use_amp}")


[Config] RUN_MODE=cuda | CUDA available=True | use_amp=True


In [8]:
if active_dp and active_ddp:
    raise ValueError("Choose only one: active_dp or active_ddp (not both).")

# Use Multiple GPU (DDP DistributedDataParallel)

In [9]:
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def is_ddp_run(mode: str) -> bool:
    world = int(os.environ.get("WORLD_SIZE", "1"))
    return (mode == "ddp") and (world > 1)

def ddp_cleanup():
    if dist.is_available() and dist.is_initialized():
        dist.barrier()
        dist.destroy_process_group()

def setup_device_and_model(model, mode: str):
    """
    mode: "cpu" | "cuda" | "ddp"
      - "dp" is handled OUTSIDE this function
      - "ddp" requires: torchrun --nproc_per_node=N script.py

    Returns: device_t, model, is_ddp, rank, world
    """
    world = int(os.environ.get("WORLD_SIZE", "1"))
    is_ddp = (mode == "ddp") and (world > 1)

    if is_ddp:
        want_cuda = torch.cuda.is_available()
        backend = "nccl" if want_cuda else "gloo"

        if not dist.is_initialized():
            dist.init_process_group(backend=backend)

        rank = dist.get_rank()
        world = dist.get_world_size()

        if want_cuda:
            local_rank = int(os.environ.get("LOCAL_RANK", "0"))
            torch.cuda.set_device(local_rank)
            device_t = torch.device(f"cuda:{local_rank}")
            model = model.to(device_t)
            model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)
        else:
            device_t = torch.device("cpu")
            model = model.to(device_t)
            model = DDP(model)

        return device_t, model, True, rank, world

    # Single-process (CPU or CUDA)
    rank, world = 0, 1
    device_t = torch.device("cpu") if mode == "cpu" else torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device_t)
    return device_t, model, False, rank, world


    # -----------------------------
    # Single-process (CPU or CUDA)
    # -----------------------------
    rank, world = 0, 1

    if mode == "cpu":
        device_t = torch.device("cpu")
    else:
        device_t = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = model.to(device_t)

    return device_t, model, False, rank, world




def setup_device_and_model_v0(model):
    use_ddp = int(os.environ.get("WORLD_SIZE", "1")) > 1

    if use_ddp:
        if not dist.is_initialized():
            dist.init_process_group(backend="nccl") # Initialize only once
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
        model = model.to(device)
        model = DDP(model, device_ids=[local_rank], output_device=local_rank)
        return device, model, True
    else:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        return device, model, False

# device, model, is_ddp = setup_device_and_model(model)
# torchrun --nproc_per_node=4 train.py

"""
import argparse

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--activate_ddp", action="store_true")
    return parser.parse_args()


def main(args):
    device, use_ddp, rank = setup_device(args)
    # model, dataloaders, training loop, etc.

if __name__ == "__main__":
    args = parse_args()
    main(args)

torchrun --nproc_per_node=4 train.py --epochs 50 --batch_size 8 --activate_ddp



## train_loader.sampler.set_epoch(epoch)
if False:
    for epoch in range(num_epochs):
        if use_ddp and isinstance(train_loader.sampler, DistributedSampler):
            train_loader.sampler.set_epoch(epoch)
    
        for X, Y in train_loader:
            ...
"""

'\nimport argparse\n\ndef parse_args():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--batch_size", type=int, default=8)\n    parser.add_argument("--activate_ddp", action="store_true")\n    return parser.parse_args()\n\n\ndef main(args):\n    device, use_ddp, rank = setup_device(args)\n    # model, dataloaders, training loop, etc.\n\nif __name__ == "__main__":\n    args = parse_args()\n    main(args)\n\ntorchrun --nproc_per_node=4 train.py --epochs 50 --batch_size 8 --activate_ddp\n\n\n\n## train_loader.sampler.set_epoch(epoch)\nif False:\n    for epoch in range(num_epochs):\n        if use_ddp and isinstance(train_loader.sampler, DistributedSampler):\n            train_loader.sampler.set_epoch(epoch)\n\n        for X, Y in train_loader:\n            ...\n'

# Decorators

In [10]:
# 1) Timer decorator: prints elapsed time, returns original result
def timer(func):
    """Decorator that prints the run time of a function and returns the original result."""
    @functools.wraps(func)  # keeps func.__name__ and docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start

        h, rem = divmod(elapsed, 3600)
        m, s = divmod(rem, 60)
        print(f"[TIMER] {func.__name__} finished in {int(h):02d}:{int(m):02d}:{s:05.2f} (hh:mm:ss)")

        return result
    return wrapper

In [11]:
# 2) Timer decorator: prints elapsed time AND returns elapsed
def timer_with_elapsed(func):
    """Decorator that returns (result, elapsed_seconds)."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start

        h, rem = divmod(elapsed, 3600)
        m, s = divmod(rem, 60)
        print(f"[TIMER] {func.__name__} finished in {int(h):02d}:{int(m):02d}:{s:05.2f} (hh:mm:ss)")

        return result, elapsed
    return wrapper

## ______________________+++++++++++++++++++++++____________________________________________________________

## FCN model

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import os
import math

class DenoiseFCN(nn.Module):
    def __init__(
        self,
        in_ch: int = 6,           # total input channels (e.g., RGB+edges = 3+3)
        out_ch: int = 3,          # output channels (e.g., clean RGB)
        width: int = 64,
        depth: int = 10,
        use_dilation: bool = True,
        residual: bool | str = "auto",  # True/False/"auto"
        skip_ch: bool = True,            # if True and in_ch >= out_ch: use first out_ch input channels as skip
    ):
        """
        Residual options:
          - residual=False: output = net(x)
          - residual=True and in_ch == out_ch: output = x + net(x)            (identity skip)
          - residual=True and in_ch != out_ch:
                * if skip_ch and in_ch >= out_ch: output = x[:,:out_ch] + net(x)  (first out_ch channels)
                * else:                         output = P(x) + net(x)            (learnable 1x1 projection)
        """
        super().__init__()

        # Resolve residual usage
        self.use_residual = True if residual == "auto" else bool(residual)
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.skip_ch = bool(skip_ch)

        # Create projection (used when needed)
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1) if self.use_residual and (in_ch != out_ch) else None

        # ---- Body ----
        layers = [nn.Conv2d(in_ch, width, kernel_size=3, padding=1), nn.ReLU(inplace=True)]
        for i in range(depth - 2):
            if use_dilation:
                d = [1, 2, 3, 2][i % 4]
                layers += [nn.Conv2d(width, width, kernel_size=3, padding=d, dilation=d), nn.ReLU(inplace=True)]
            else:
                layers += [nn.Conv2d(width, width, kernel_size=3, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(width, out_ch, kernel_size=3, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, in_ch, H, W] -> output: [B, out_ch, H, W]
        """
        y = self.net(x)

        if not self.use_residual:
            return y

        if self.in_ch == self.out_ch:
            # identity skip
            skip = x
        else:
            # channel-mismatch case
            if self.skip_ch and (self.in_ch >= self.out_ch):
                # use only the first out_ch channels as the skip
                skip = x[:, :self.out_ch, :, :]
            else:
                # fall back to 1x1 projection (learned mix of all input channels)
                if self.proj is None:
                    # safety: create on-the-fly if not present for some reason
                    self.proj = nn.Conv2d(self.in_ch, self.out_ch, kernel_size=1).to(x.device)
                skip = self.proj(x)

        return skip + y



#===========================================================================================================================
def create_model_fcn(width = 64, depth = 10, device='cpu' , extra_edges = (), m_compile = True):
    # ====== model ======
    # in_ch = 3 (RGB) + len(extra_edges)
    in_ch = 3 + (len(extra_edges) if extra_edges else 0)
    model = DenoiseFCN(
        in_ch=in_ch,
        out_ch=3,
        width=width,
        depth=depth,
        use_dilation=True,
        residual="auto",
        skip_ch=True
    )#.to(device)
    
    # (Optional) torch.compile for speed if available
    if m_compile:
        try:
            model = torch.compile(model, backend="eager")  # no Triton required
        except Exception:
            pass
    return model, in_ch

## VAE - Variational Autoencoder - Noise Model

In [13]:
# Conv2d Encoder Decoder (noise-predicting VAE; returns reconstruction, [mu, log_var])

import torch
import torch.nn as nn
import torch.nn.functional as F


# -------- helpers --------
def _group_norm(c: int, max_groups: int = 8):
    return nn.GroupNorm(num_groups=min(max_groups, c), num_channels=c)

def kaiming_init_(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, nonlinearity="leaky_relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)


class ConvEncoder(nn.Module):
    def __init__(self, input_channels, hidden_dims, latent_dim, img_size, use_gn: bool = True):
        super().__init__()
        if isinstance(img_size, int):
            self.img_h = self.img_w = img_size
        else:
            self.img_h, self.img_w = img_size

        modules = []
        in_channels = input_channels
        for h_dim in hidden_dims:
            norm = _group_norm(h_dim) if use_gn else nn.Identity()
            modules.append(nn.Sequential(
                nn.Conv2d(in_channels, h_dim, kernel_size=3, stride=2, padding=1),
                norm,
                nn.LeakyReLU(inplace=True),
            ))
            in_channels = h_dim
        self.encoder_cnn = nn.Sequential(*modules)

        # Compute expected feature-map shape from declared img_size
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, self.img_h, self.img_w)
            dout = self.encoder_cnn(dummy)
            self.final_feature_map_channels = dout.shape[1]
            self.final_feature_map_h = dout.shape[2]
            self.final_feature_map_w = dout.shape[3]
            assert self.final_feature_map_h == self.final_feature_map_w, (
                f"Encoder feature map is not square: "
                f"H={self.final_feature_map_h}, W={self.final_feature_map_w}."
            )
            self.final_feature_map_dim = self.final_feature_map_h
            self.flatten_size = (self.final_feature_map_channels *
                                 self.final_feature_map_h *
                                 self.final_feature_map_w)

        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)

        self.apply(kaiming_init_)

    def forward(self, x):
        assert x.dim() == 4, f"Expected NCHW tensor, got {tuple(x.shape)}"
        B, C, H, W = x.shape
        assert H == self.img_h and W == self.img_w, (
            f"Input spatial size mismatch: got {(H, W)}, expected {(self.img_h, self.img_w)}"
        )

        h = self.encoder_cnn(x)

        _, Cf, Hf, Wf = h.shape
        assert (Cf == self.final_feature_map_channels and
                Hf == self.final_feature_map_h and
                Wf == self.final_feature_map_w), (
            "Encoder feature-map size changed between init and forward.\n"
            f"Got (C,H,W)=({Cf},{Hf},{Wf}) but expected "
            f"({self.final_feature_map_channels},{self.final_feature_map_h},{self.final_feature_map_w})."
        )

        h = h.view(B, -1)
        mu = self.fc_mu(h)
        log_var = self.fc_logvar(h)
        return mu, log_var


class ConvDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dims, output_channels,
                 final_feature_map_dim, final_feature_map_channels, use_gn: bool = True):
        super().__init__()
        self.final_feature_map_dim = final_feature_map_dim
        self.final_feature_map_channels = final_feature_map_channels

        self.initial_fc = nn.Linear(
            latent_dim,
            final_feature_map_channels * final_feature_map_dim * final_feature_map_dim
        )

        modules = []
        hidden_dims_reversed = list(reversed(hidden_dims))
        in_channels = final_feature_map_channels
        for i in range(len(hidden_dims_reversed) - 1):
            out_channels = hidden_dims_reversed[i + 1]
            norm = _group_norm(out_channels) if use_gn else nn.Identity()
            modules.append(nn.Sequential(
                nn.ConvTranspose2d(in_channels, out_channels,
                                   kernel_size=3, stride=2, padding=1, output_padding=1),
                norm,
                nn.LeakyReLU(inplace=True),
            ))
            in_channels = out_channels
        self.decoder_cnn = nn.Sequential(*modules)

        # FINAL HEAD: linear (no Sigmoid) — we predict signed noise ε
        self.final_layer = nn.ConvTranspose2d(
            in_channels, output_channels,
            kernel_size=3, stride=2, padding=1, output_padding=1
        )

        self.apply(kaiming_init_)

    def forward(self, z):
        B = z.shape[0]
        h = self.initial_fc(z)
        h = h.view(B, self.final_feature_map_channels,
                   self.final_feature_map_dim, self.final_feature_map_dim)
        h = self.decoder_cnn(h)
        eps_hat = self.final_layer(h)  # signed noise
        return eps_hat


class VAE(nn.Module):
    """
    Decoder predicts epsilon (noise). Forward returns (reconstruction, [mu, log_var]).
    Reconstruction = clamp(x + tanh(eps_raw) * noise_scale, [img_min, img_max]).
    """
    def __init__(self, input_channels, hidden_dims, latent_dim, img_size,
                 noise_scale_init: float = 0.1, learnable_noise_scale: bool = False,
                 img_min: float = 0.0, img_max: float = 1.0, use_gn: bool = True):
        super(VAE, self).__init__()

        temp_encoder = ConvEncoder(input_channels, hidden_dims, latent_dim, img_size, use_gn=use_gn)
        self.encoder = temp_encoder

        self.decoder = ConvDecoder(
            latent_dim, hidden_dims, input_channels,
            temp_encoder.final_feature_map_dim,
            temp_encoder.final_feature_map_channels,
            use_gn=use_gn
        )

        # noise scale used to bound epsilon via tanh
        if learnable_noise_scale:
            self._noise_scale = nn.Parameter(torch.tensor(float(noise_scale_init)))
            self._learnable_scale = True
        else:
            self.register_buffer("_noise_scale", torch.tensor(float(noise_scale_init)))
            self._learnable_scale = False

        self.img_min = float(img_min)
        self.img_max = float(img_max)

    @staticmethod
    def _clamp_logvar(log_var, lo: float = -20.0, hi: float = 10.0):
        return log_var.clamp(min=lo, max=hi)

    def _noise_scale_value(self):
        return F.softplus(self._noise_scale) if self._learnable_scale else self._noise_scale

    def reparameterize(self, mu, log_var):
        log_var = self._clamp_logvar(log_var)
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)

        # decoder outputs raw noise; bound it with tanh * scale
        eps_raw = self.decoder(z)
        eps_bounded = torch.tanh(eps_raw) * self._noise_scale_value()

        reconstruction = torch.clamp(x + eps_bounded, self.img_min, self.img_max)
        return reconstruction, [mu, log_var]


In [14]:
def create_model_vae(
    input_channels: int = 3,
    img_size: int = 128,
    latent_dim: int = 32,
    hidden_dims: list | None = None,
    # If hidden_dims is None, we'll build it from these:
    depth: int = 2,
    base_channels: int = 32,
    growth: int = 2,
    device: str = "cpu",
    m_compile: bool = True,
):
    """
    Build a Conv VAE with configurable encoder/decoder depth.

    Args:
        input_channels: Input image channels (e.g., 1 for MNIST, 3 for RGB).
        img_size: Square image size (H=W). Must be compatible with stride-2 downsamples.
        latent_dim: Size of latent vector z.
        hidden_dims: List of channels per encoder block; length defines depth.
                     Example: [32, 64, 128]. If None, we'll generate it from depth/base_channels/growth.
        depth: Number of downsample/upsample stages when hidden_dims is None.
        base_channels: First stage channels when hidden_dims is None.
        growth: Multiplier per stage when hidden_dims is None (usually 2).
        device: "cpu" or "cuda".
        m_compile: Try torch.compile for speed (PyTorch 2+).

    Returns:
        model: An instance of VAE moved to the requested device.
    """
    # Build hidden_dims if not provided (controls depth)
    if hidden_dims is None:
        if depth < 1:
            raise ValueError("depth must be >= 1")
        hidden_dims = [base_channels * (growth ** i) for i in range(depth)]

    # Basic sanity checks
    if not isinstance(hidden_dims, (list, tuple)) or len(hidden_dims) == 0:
        raise ValueError("hidden_dims must be a non-empty list of channel sizes.")
    if any(h <= 0 for h in hidden_dims):
        raise ValueError("All entries in hidden_dims must be positive integers.")

    # Instantiate the model
    model = VAE(
        input_channels=input_channels,
        hidden_dims=hidden_dims,
        latent_dim=latent_dim,
        img_size=img_size,
    )#.to(device)

    # (Optional) torch.compile for speed if available
    if m_compile:
        try:
            model = torch.compile(model, backend="eager")  # works without Triton
        except Exception:
            pass

    return model

# CIFAR-like (3x32x32) with explicit hidden_dims (depth = len(list) = 3)
#model = create_model_vae(input_channels=3, img_size=patch_size, latent_dim=64, hidden_dims=[16, 32, 64], device="cuda", m_compile = False)
#print(model)

# Data Loader ________

In [15]:
# ======================================================================================================
# FULL COPY-PASTE READY MODULE
# - Paths (Fruits + DIV2K) + Optuna DB/model path helpers (as you provided)
# - add_noise()
# - CleanImageFolder (supports blur + multiple noise types + edges + per-worker RNG + optional RAM cache)
# - PairedImageFolder_v0 + PairedImageFolder (supports degrade_x + edges + optional RAM cache)
# - edge_from mappers
# - data_loader()
#
# REQUIREMENT:
#   - You must define n_workers somewhere (or it will default to 0 below).
#   - This file assumes you have torch, torchvision, opencv-python installed.
# ======================================================================================================


from pathlib import Path
import os
import glob
import random
from typing import Optional, Dict, Tuple, Sequence
from PIL import Image

import cv2
import math
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF
from torchvision import transforms

# For DDP
from torch.utils.data.distributed import DistributedSampler








# ======================================================================================================
# Noise helper (yours, unchanged)
# ======================================================================================================

def _colored_noise_2d_np(
    H: int,
    W: int,
    alpha: float,
    rng: np.random.RandomState,
) -> np.ndarray:
    """
    2D colored noise with PSD ~ 1/f^alpha (radial frequency).
    Returns zero-mean, unit-std float32 array of shape (H,W).
    """
    fy = np.fft.fftfreq(H)[:, None]   # (H,1)
    fx = np.fft.fftfreq(W)[None, :]   # (1,W)
    f = np.sqrt(fx * fx + fy * fy)

    # Start from real white field -> FFT -> shape spectrum
    white = rng.normal(0.0, 1.0, size=(H, W)).astype(np.float32)
    F = np.fft.fft2(white)

    amp = np.ones_like(f, dtype=np.float64)
    nonzero = f > 0
    amp[nonzero] = f[nonzero] ** (-alpha / 2.0)
    amp[~nonzero] = 0.0  # kill DC to keep mean controlled

    F *= amp
    x = np.fft.ifft2(F).real.astype(np.float32)

    x -= float(x.mean())
    s = float(x.std())
    if s > 0:
        x /= s
    return x



def add_noise(
    img01: torch.Tensor,
    noise_type: str = "gaussian",
    params: dict | None = None,
    rng: np.random.RandomState | None = None,
) -> torch.Tensor:
    """
    img01: torch tensor in [0,1], shape [C,H,W]
    noise_type:
      - existing: 'gaussian' | 'poisson' | 'random_uniform' | 'salt_pepper' | 'none'
      - new colored: 'white'|'pink'|'brown'|'blue'|'violet'|'colored'
    params:
      - gaussian: {'sigma_range': (0.0,25.0)}   in 0..255 space
      - poisson:  {'lam_range': (10.0, 60.0)}   OR {'peak_range': (10,60)}
      - random_uniform: {'amp_range': (0.0, 0.10)}  amplitude in [0,1]
      - salt_pepper: {'amount_range': (0.0, 0.05), 'salt_vs_pepper': 0.5}
      - colored:
          {'sigma_range': (0.0,25.0)} in 0..255 space
          {'alpha': 1.0}  (only for noise_type='colored')
          {'per_channel': True/False}  (RGB independent or shared)
    rng: numpy RandomState (recommended, per-worker)
    """
    if params is None:
        params = {}
    if rng is None:
        rng = np.random.RandomState(0)

    nt = (noise_type or "gaussian").lower()
    x = img01.clamp(0, 1)

    if nt in ("none", "no", "off"):
        return x

    # -------------------------
    # Standard i.i.d. noises
    # -------------------------
    if nt == "gaussian":
        sigma_range = params.get("sigma_range", (0.0, 25.0))
        sigma = float(rng.uniform(sigma_range[0], sigma_range[1])) / 255.0
        return (x + torch.randn_like(x) * sigma).clamp(0, 1)

    if nt == "poisson":
        lam_range = params.get("lam_range", None)
        peak_range = params.get("peak_range", None)

        if peak_range is not None:
            peak = float(rng.uniform(peak_range[0], peak_range[1]))
        elif lam_range is not None:
            peak = float(rng.uniform(lam_range[0], lam_range[1]))
        else:
            peak = float(rng.uniform(10.0, 60.0))

        y = torch.poisson(x * peak) / peak
        return y.clamp(0, 1)

    if nt in ("random", "random_uniform", "uniform"):
        amp_range = params.get("amp_range", (0.0, 0.10))
        amp = float(rng.uniform(amp_range[0], amp_range[1]))
        u = (torch.rand_like(x) * 2.0 - 1.0) * amp
        return (x + u).clamp(0, 1)

    if nt in ("salt_pepper", "s&p", "impulse"):
        amount_range = params.get("amount_range", (0.0, 0.05))
        amount = float(rng.uniform(amount_range[0], amount_range[1]))
        svp = float(params.get("salt_vs_pepper", 0.5))

        out = x.clone()
        mask = torch.rand_like(out[:1]) < amount
        salt = torch.rand_like(out[:1]) < svp
        maskC = mask.expand_as(out)
        saltC = salt.expand_as(out)

        out[maskC & saltC] = 1.0
        out[maskC & (~saltC)] = 0.0
        return out.clamp(0, 1)

    # -------------------------
    # Colored (spatial) noises
    # -------------------------
    color_map = {
        "white": 0.0,
        "pink": 1.0,
        "brown": 2.0,
        "blue": -1.0,
        "violet": -2.0,
    }

    if nt in color_map or nt == "colored":
        if nt == "colored":
            alpha = float(params.get("alpha", 1.0))
        else:
            alpha = float(color_map[nt])

        sigma_range = params.get("sigma_range", (0.0, 25.0))
        sigma = float(rng.uniform(sigma_range[0], sigma_range[1])) / 255.0

        per_channel = bool(params.get("per_channel", True))

        C, H, W = x.shape
        device = x.device
        dtype = x.dtype

        if (C == 1) or (not per_channel):
            n2d = _colored_noise_2d_np(H, W, alpha, rng)  # (H,W)
            n = torch.from_numpy(n2d).to(device=device, dtype=dtype).unsqueeze(0).expand(C, -1, -1)
        else:
            # independent colored field per channel
            chans = []
            for _ in range(C):
                n2d = _colored_noise_2d_np(H, W, alpha, rng)
                chans.append(torch.from_numpy(n2d))
            n = torch.stack(chans, dim=0).to(device=device, dtype=dtype)

        return (x + n * sigma).clamp(0, 1)

    raise ValueError(
        f"Unknown noise_type='{noise_type}'. "
        "Use: gaussian, poisson, random_uniform, salt_pepper, none, white, pink, brown, blue, violet, colored"
    )



# ======================================================================================================
# Basic image helpers
# ======================================================================================================
def _imread_rgb(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def _to_gray01_from_chw_rgb01(t_chw01: torch.Tensor) -> np.ndarray:
    """[0,1] CHW RGB tensor -> grayscale float32 [0,1] numpy (H,W)."""
    a = t_chw01.permute(1, 2, 0).cpu().numpy().astype(np.float32)
    gray = 0.299 * a[..., 0] + 0.587 * a[..., 1] + 0.114 * a[..., 2]
    return gray.astype(np.float32)


# ======================================================================================================
# PairedImageFolder (v0 + RAM wrapper)
# ======================================================================================================
class PairedImageFolder_v0(Dataset):
    """
    Loads paired images from two folders (X and Y) matched by filename stem.

    Returns: (Xin, Y)
      - Optionally degrades X (blur -> additive gaussian noise)
      - Optionally appends edge channels computed from 'degraded' | 'x' | 'y'
    """

    def __init__(
        self,
        x_folder: str,
        y_folder: str,
        patch_size: int = 128,

        noise_type: str = "gaussian",
        noise_params: Optional[dict] = None,

        degrade_x: bool = True,
        sigma_range: Tuple[float, float] = (0.0, 25.0),
        blur_prob: float = 0.5,
        blur_kernel_choices: Sequence[int] = (3, 5, 7, 9),
        blur_sigma_range: Tuple[float, float] = (0.5, 2.0),
        extra_edges: Sequence[str] = ("canny", "sobel", "prewitt"),
        edge_from: str = "degraded",
        canny_low: int = 200,
        canny_high: int = 255,
        sobel_ksize: int = 9,
        return_paths: bool = False,
        seed: int = 0,
    ):
        super().__init__()

        def _index(folder: str) -> Dict[str, str]:
            exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
            files = [
                p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                if p.lower().endswith(exts)
            ]
            by_stem = {}
            for p in files:
                stem = os.path.splitext(os.path.basename(p))[0].lower()
                by_stem[stem] = p
            return by_stem

        x_map = _index(x_folder)
        y_map = _index(y_folder)
        common = sorted(set(x_map.keys()) & set(y_map.keys()))
        if not common:
            raise RuntimeError(f"No paired images found between {x_folder} and {y_folder}")

        self.pairs = [(x_map[s], y_map[s]) for s in common]
        self.patch_size = patch_size if patch_size is None or patch_size > 0 else None

        self.noise_type = str(noise_type).lower()
        self.noise_params = {} if noise_params is None else dict(noise_params)
        self.np_rng = None



        self.degrade_x = bool(degrade_x)
        self.sigma_range = (float(sigma_range[0]), float(sigma_range[1]))
        self.blur_prob = float(blur_prob)
        self.blur_kernel_choices = tuple(int(k) | 1 for k in blur_kernel_choices)
        self.blur_sigma_range = (float(blur_sigma_range[0]), float(blur_sigma_range[1]))

        self.extra_edges = tuple(extra_edges) if extra_edges else ()
        self.edge_from = (edge_from or "degraded").lower()
        assert self.edge_from in ("degraded", "x", "y")

        self.canny_low = int(canny_low)
        self.canny_high = int(canny_high)
        self.sobel_ksize = int(max(3, sobel_ksize | 1))

        self.return_paths = bool(return_paths)
        self.seed = int(seed)
        self.rng = random.Random(self.seed)

    def __len__(self) -> int:
        return len(self.pairs)

    def _choose_crop(self, H1W1, H2W2, ps):
        if ps is None:
            return None
        H1, W1 = H1W1
        H2, W2 = H2W2
        if H1 < ps or W1 < ps or H2 < ps or W2 < ps:
            return None
        top = self.rng.randint(0, min(H1, H2) - ps)
        left = self.rng.randint(0, min(W1, W2) - ps)
        return top, left

    def _center_crop_if_needed(self, t: torch.Tensor, ps: Optional[int]) -> torch.Tensor:
        if ps is None:
            return t
        _, H, W = t.shape
        if H < ps or W < ps:
            return TF.center_crop(t, [ps, ps])
        return t

    def _apply_crop(self, t: torch.Tensor, crop) -> torch.Tensor:
        if crop is None or self.patch_size is None:
            return self._center_crop_if_needed(t, self.patch_size)
        top, left = crop
        ps = self.patch_size
        return t[:, top : top + ps, left : left + ps]

    def _edges_from_tensor_rgb01(self, t_rgb01: torch.Tensor):
        outs = []
        if not self.extra_edges:
            return outs

        gray01 = _to_gray01_from_chw_rgb01(t_rgb01)
        gray8 = np.clip(gray01 * 255.0, 0, 255).astype(np.uint8)

        for m in self.extra_edges:
            m = m.lower()
            if m == "canny":
                e = cv2.Canny(gray8, self.canny_low, self.canny_high).astype(np.float32) / 255.0
            elif m == "sobel":
                gx = cv2.Sobel(gray8, cv2.CV_32F, 1, 0, ksize=self.sobel_ksize)
                gy = cv2.Sobel(gray8, cv2.CV_32F, 0, 1, ksize=self.sobel_ksize)
                mag = np.sqrt(gx * gx + gy * gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            elif m == "prewitt":
                kx = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
                ky = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=np.float32)
                gx = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, kx)
                gy = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, ky)
                mag = np.sqrt(gx * gx + gy * gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            else:
                raise ValueError(f"Unsupported edge method: {m}")
            outs.append(torch.from_numpy(e).unsqueeze(0))
        return outs

    def __getitem__(self, idx: int):
        x_path, y_path = self.pairs[idx]
        X = torch.from_numpy(_imread_rgb(x_path)).permute(2, 0, 1).float() / 255.0
        Y = torch.from_numpy(_imread_rgb(y_path)).permute(2, 0, 1).float() / 255.0

        crop = self._choose_crop((X.shape[1], X.shape[2]), (Y.shape[1], Y.shape[2]), self.patch_size)
        X = self._apply_crop(X, crop)
        Y = self._apply_crop(Y, crop)

        Xin = X
        if self.degrade_x:
            if self.blur_prob > 0.0 and self.rng.random() < self.blur_prob:
                k = self.rng.choice(self.blur_kernel_choices)
                sig = self.rng.uniform(*self.blur_sigma_range)
                Xin = TF.gaussian_blur(Xin, kernel_size=[k, k], sigma=sig)
            
            # use numpy RNG for noise so it's reproducible per worker
            rng_np = self._get_np_rng() 
            p = dict(self.noise_params)
            if self.noise_type == "gaussian" and "sigma_range" not in p:
                p["sigma_range"] = self.sigma_range
            
            Xin = add_noise(Xin, noise_type=self.noise_type, params=p, rng=rng_np)



        if self.extra_edges:
            if self.edge_from == "degraded":
                src = Xin
            elif self.edge_from == "x":
                src = X
            else:
                src = Y
            edges = self._edges_from_tensor_rgb01(src)
            if edges:
                Xin = torch.cat([Xin, torch.cat(edges, dim=0)], dim=0)

        if self.return_paths:
            return Xin, Y, (x_path, y_path)
        return Xin, Y


    def _get_np_rng(self) -> np.random.RandomState:
        if self.np_rng is None:
            wi = torch.utils.data.get_worker_info()
            s = self.seed if wi is None else (self.seed + wi.id)
            self.np_rng = np.random.RandomState(s)
        return self.np_rng


class PairedImageFolder(PairedImageFolder_v0):
    """PairedImageFolder_v0 + optional RAM caching (keep_in_ram)."""

    def __init__(self, *args, keep_in_ram: bool = False, **kwargs):
        super().__init__(*args, **kwargs)

        self.keep_in_ram = bool(keep_in_ram)
        self._x_cache = None
        self._y_cache = None

        if self.keep_in_ram:
            self._x_cache = []
            self._y_cache = []
            for x_path, y_path in self.pairs:
                X = torch.from_numpy(_imread_rgb(x_path)).permute(2, 0, 1).float() / 255.0
                Y = torch.from_numpy(_imread_rgb(y_path)).permute(2, 0, 1).float() / 255.0
                self._x_cache.append(X)
                self._y_cache.append(Y)
            print(f"[PairedImageFolder] Cached {len(self.pairs)} pairs into RAM.")

    def __getitem__(self, idx: int):
        x_path, y_path = self.pairs[idx]

        if self.keep_in_ram and self._x_cache is not None:
            X = self._x_cache[idx].clone()
            Y = self._y_cache[idx].clone()
        else:
            X = torch.from_numpy(_imread_rgb(x_path)).permute(2, 0, 1).float() / 255.0
            Y = torch.from_numpy(_imread_rgb(y_path)).permute(2, 0, 1).float() / 255.0

        crop = self._choose_crop((X.shape[1], X.shape[2]), (Y.shape[1], Y.shape[2]), self.patch_size)
        X = self._apply_crop(X, crop)
        Y = self._apply_crop(Y, crop)

        Xin = X
        if self.degrade_x:
            if self.blur_prob > 0.0 and self.rng.random() < self.blur_prob:
                k = self.rng.choice(self.blur_kernel_choices)
                sig = self.rng.uniform(*self.blur_sigma_range)
                Xin = TF.gaussian_blur(Xin, kernel_size=[k, k], sigma=sig)

            # use numpy RNG for noise so it's reproducible per worker
            rng_np = self._get_np_rng()  
            p = dict(self.noise_params)
            if self.noise_type == "gaussian" and "sigma_range" not in p:
                p["sigma_range"] = self.sigma_range
            
            Xin = add_noise(Xin, noise_type=self.noise_type, params=p, rng=rng_np)


        if self.extra_edges:
            if self.edge_from == "degraded":
                src = Xin
            elif self.edge_from == "x":
                src = X
            else:
                src = Y
            edges = self._edges_from_tensor_rgb01(src)
            if edges:
                Xin = torch.cat([Xin, torch.cat(edges, dim=0)], dim=0)

        if self.return_paths:
            return Xin, Y, (x_path, y_path)
        return Xin, Y


# ======================================================================================================
# CleanImageFolder (UPDATED + FIXED)
# Fixes vs your pasted block:
#   - imports: glob/random/TF were missing in your snippet
#   - defines self.seed + self.rng (you were using self.rng but never defined it)
#   - __getitem__ had "..." placeholders; now fully implemented
#   - uses per-worker numpy RNG via _get_np_rng() (reproducible + worker-safe)
# ======================================================================================================
class CleanImageFolder(Dataset):
    def __init__(
        self,
        folder: str,
        patch_size: int = 128,
        sigma_range: Tuple[float, float] = (0.0, 25.0),
        extra_edges: Sequence[str] = ("canny", "sobel", "prewitt"),
        edge_from: str = "noisy",  # 'noisy' or 'clean'
        canny_low: int = 200,
        canny_high: int = 255,
        sobel_ksize: int = 9,
        blur_prob: float = 0.5,
        blur_kernel_choices: Sequence[int] = (3, 5, 7, 9),
        blur_sigma_range: Tuple[float, float] = (0.5, 2.0),
        keep_in_ram: bool = False,
        noise_type: str = "gaussian",
        noise_params: Optional[dict] = None,
        seed: int = 0,
    ):
        exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
        self.files = [
            p for p in glob.glob(os.path.join(str(folder), "**", "*"), recursive=True)
            if p.lower().endswith(exts)
        ]
        if not self.files:
            raise RuntimeError(f"No images found in {folder}")

        self.patch_size = patch_size if patch_size is None or patch_size > 0 else None
        self.sigma_range = (float(sigma_range[0]), float(sigma_range[1]))

        self.extra_edges = tuple(extra_edges) if extra_edges else ()
        self.edge_from = (edge_from or "noisy").lower()
        assert self.edge_from in ("noisy", "clean")

        self.canny_low = int(canny_low)
        self.canny_high = int(canny_high)
        self.sobel_ksize = int(max(3, sobel_ksize | 1))

        self.blur_prob = float(blur_prob)
        self.blur_kernel_choices = tuple(int(k) | 1 for k in blur_kernel_choices)
        self.blur_sigma_range = (float(blur_sigma_range[0]), float(blur_sigma_range[1]))

        self.noise_type = str(noise_type).lower()
        self.noise_params = {} if noise_params is None else dict(noise_params)

        self.seed = int(seed)
        self.rng = random.Random(self.seed)  # python rng for crop/blur decisions
        self.np_rng = None                   # numpy rng (per-worker)

        self.keep_in_ram = bool(keep_in_ram)
        self._img_cache = None
        if self.keep_in_ram:
            self._img_cache = []
            for path in self.files:
                img = cv2.imread(path, cv2.IMREAD_COLOR)
                if img is None:
                    raise FileNotFoundError(f"Could not read {path}")
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
                self._img_cache.append(t)
            print(f"[CleanImageFolder] Cached {len(self.files)} images into RAM.")

    def __len__(self) -> int:
        return len(self.files)

    def _random_crop(self, img: torch.Tensor, ps: Optional[int]) -> torch.Tensor:
        if ps is None:
            return img
        _, H, W = img.shape
        if H < ps or W < ps:
            return TF.center_crop(img, [ps, ps])
        top = self.rng.randint(0, H - ps)
        left = self.rng.randint(0, W - ps)
        return img[:, top : top + ps, left : left + ps]

    def _get_np_rng(self) -> np.random.RandomState:
        if self.np_rng is None:
            wi = torch.utils.data.get_worker_info()
            if wi is None:
                s = self.seed
            else:
                s = self.seed + wi.id
            self.np_rng = np.random.RandomState(s)
        return self.np_rng

    @staticmethod
    def _to_cv_gray01(t_chw01: torch.Tensor) -> np.ndarray:
        a = t_chw01.permute(1, 2, 0).cpu().numpy().astype(np.float32)
        gray = 0.299 * a[..., 0] + 0.587 * a[..., 1] + 0.114 * a[..., 2]
        return gray.astype(np.float32)

    def _edges_from_tensor_rgb01(self, t_rgb01: torch.Tensor):
        gray01 = self._to_cv_gray01(t_rgb01)
        gray8 = np.clip(gray01 * 255.0, 0, 255).astype(np.uint8)
        outs = []

        for m in self.extra_edges:
            m = m.lower()
            if m == "canny":
                e = cv2.Canny(gray8, self.canny_low, self.canny_high)
                e = (e.astype(np.float32) / 255.0)
            elif m == "sobel":
                gx = cv2.Sobel(gray8, cv2.CV_32F, 1, 0, ksize=self.sobel_ksize)
                gy = cv2.Sobel(gray8, cv2.CV_32F, 0, 1, ksize=self.sobel_ksize)
                mag = np.sqrt(gx * gx + gy * gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            elif m == "prewitt":
                kx = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
                ky = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=np.float32)
                gx = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, kx)
                gy = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, ky)
                mag = np.sqrt(gx * gx + gy * gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            else:
                raise ValueError(f"Unsupported edge method: {m}")

            outs.append(torch.from_numpy(e).unsqueeze(0))
        return outs

    def __getitem__(self, idx: int):
        rng_np = self._get_np_rng()

        # 1) Read clean image
        if self.keep_in_ram and self._img_cache is not None:
            t = self._img_cache[idx].clone()
        else:
            path = self.files[idx]
            img = cv2.imread(path, cv2.IMREAD_COLOR)
            if img is None:
                raise FileNotFoundError(f"Could not read {path}")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        # 2) Crop
        t = self._random_crop(t, self.patch_size)

        # 3) Optional blur (before noise)
        degraded = t
        if self.blur_prob > 0.0 and self.rng.random() < self.blur_prob:
            k = self.rng.choice(self.blur_kernel_choices)
            sig = self.rng.uniform(*self.blur_sigma_range)
            degraded = TF.gaussian_blur(degraded, kernel_size=[k, k], sigma=sig)

        # 4) Add noise (generalized)
        p = dict(self.noise_params)
        if self.noise_type == "gaussian" and "sigma_range" not in p:
            p["sigma_range"] = self.sigma_range

        degraded = add_noise(
            degraded,
            noise_type=self.noise_type,
            params=p,
            rng=rng_np,
        )

        # 5) Edges
        src_for_edges = degraded if self.edge_from == "noisy" else t
        edge_tensors = self._edges_from_tensor_rgb01(src_for_edges) if self.extra_edges else []

        # 6) Concatenate edges to input
        if edge_tensors:
            edges_chw = torch.cat(edge_tensors, dim=0)
            inp = torch.cat([degraded, edges_chw], dim=0)
        else:
            inp = degraded

        return inp, t


# ======================================================================================================
# Optional legacy dataset variants (kept minimal; your training should use CleanImageFolder above)
# ======================================================================================================
class CleanImageFolder_v0(Dataset):
    """Legacy version: gaussian blur + gaussian noise only."""
    def __init__(
        self,
        folder,
        patch_size=128,
        sigma_range=(0.0, 25.0),
        extra_edges=("canny", "sobel", "prewitt"),
        edge_from="noisy",
        canny_low=200,
        canny_high=255,
        sobel_ksize=9,
        blur_prob=0.5,
        blur_kernel_choices=(3, 5, 7, 9),
        blur_sigma_range=(0.5, 2.0),
        seed=0,
    ):
        self.ds = CleanImageFolder(
            folder=folder,
            patch_size=patch_size,
            sigma_range=sigma_range,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=canny_low,
            canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            blur_prob=blur_prob,
            blur_kernel_choices=blur_kernel_choices,
            blur_sigma_range=blur_sigma_range,
            keep_in_ram=False,
            noise_type="gaussian",
            noise_params={"sigma_range": sigma_range},
            seed=seed,
        )

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        return self.ds[idx]


class CleanImageFolder_onlynoise(Dataset):
    """Legacy: gaussian noise only + optional edges."""
    def __init__(
        self,
        folder,
        patch_size=128,
        sigma_range=(0.0, 25.0),
        extra_edges=("canny", "sobel", "prewitt"),
        edge_from="noisy",
        canny_low=200,
        canny_high=255,
        sobel_ksize=9,
        seed=0,
    ):
        self.ds = CleanImageFolder(
            folder=folder,
            patch_size=patch_size,
            sigma_range=sigma_range,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=canny_low,
            canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            blur_prob=0.0,
            keep_in_ram=False,
            noise_type="gaussian",
            noise_params={"sigma_range": sigma_range},
            seed=seed,
        )

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        return self.ds[idx]


# ======================================================================================================
# edge_from mappers (prevents assertions when you pass old names)
# ======================================================================================================
def _map_edge_from_for_clean(edge_from: str) -> str:
    """
    CleanImageFolder expects: 'noisy' or 'clean'
    Your code sometimes uses: 'degraded' | 'x' | 'y'
    """
    ef = (edge_from or "noisy").lower()
    m = {
        "noisy": "noisy",
        "clean": "clean",
        "degraded": "noisy",
        "x": "noisy",
        "y": "clean",
    }
    ef2 = m.get(ef, ef)
    assert ef2 in ("noisy", "clean"), f"CleanImageFolder edge_from must be 'noisy' or 'clean', got: {edge_from}"
    return ef2


def _map_edge_from_for_paired(edge_from: str) -> str:
    """
    PairedImageFolder expects: 'degraded' | 'x' | 'y'
    If user passes 'noisy'/'clean', map them sensibly.
    """
    ef = (edge_from or "degraded").lower()
    m = {
        "degraded": "degraded",
        "x": "x",
        "y": "y",
        "noisy": "degraded",
        "clean": "y",
    }
    ef2 = m.get(ef, ef)
    assert ef2 in ("degraded", "x", "y"), f"PairedImageFolder edge_from must be 'degraded'|'x'|'y', got: {edge_from}"
    return ef2


# ======================================================================================================
# data_loader()
# ======================================================================================================
def data_loader(data_loader_option: int = 0, **kwargs):
    """
    Build train/val DataLoaders.
    data_loader_option=0 -> CleanImageFolder (single folder)
    data_loader_option=1 -> PairedImageFolder (x/y folders)
    Returns (train_loader, val_loader)
    """

    # For DDP ___________
    use_ddp     = bool(kwargs.get("use_ddp", False))
    ddp_rank    = int(kwargs.get("ddp_rank", 0))
    ddp_world   = int(kwargs.get("ddp_world_size", 1))

    if use_ddp:
        assert ddp_world > 1, "use_ddp=True but ddp_world_size <= 1"
        assert 0 <= ddp_rank < ddp_world, "Invalid ddp_rank"

    # Params ___________
    batch_size      = int(kwargs.get("batch_size", 16))
    num_workers     = int(kwargs.get("num_workers", n_workers))
    pin_memory = bool(kwargs.get("pin_memory", torch.cuda.is_available()))
    drop_last_train = bool(kwargs.get("drop_last_train", True))

    # For DDP _
    dl_worker_kwargs = {}
    if num_workers > 0:
        dl_worker_kwargs["persistent_workers"] = True
        dl_worker_kwargs["prefetch_factor"] =  2


    patch_size  = kwargs.get("patch_size", 128)
    sigma_range = kwargs.get("sigma_range", (0.0, 0.0))
    blur_prob   = float(kwargs.get("blur_prob", 0.0))

    extra_edges   = kwargs.get("extra_edges", ())
    edge_from_raw = kwargs.get("edge_from", "degraded")

    blur_kernels   = kwargs.get("blur_kernels", (3, 5, 7, 9))
    blur_sigma_rng = kwargs.get("blur_sigma_rng", (0.5, 2.0))

    canny_low   = int(kwargs.get("canny_low", 200))
    canny_high  = int(kwargs.get("canny_high", 255))
    sobel_ksize = int(kwargs.get("sobel_ksize", 9))

    # noise controls (CleanImageFolder)
    noise_type   = kwargs.get("noise_type", "gaussian")
    noise_params = kwargs.get("noise_params", None)
    seed         = int(kwargs.get("seed", 0))

    # RAM caching
    keep_in_ram_train = bool(kwargs.get("keep_in_ram_train", False))
    keep_in_ram_val   = bool(kwargs.get("keep_in_ram_val", keep_in_ram_train))

    train_loader, val_loader = None, None

    if data_loader_option == 0:
        # -------------------------
        # CleanImageFolder
        # -------------------------
        train_dir = kwargs.get("train_dir", train_dir_default)
        val_dir   = kwargs.get("val_dir", val_dir_default)
        assert train_dir is not None, "data_loader_option=0 requires 'train_dir'"

        edge_from = _map_edge_from_for_clean(edge_from_raw)

        train_ds = CleanImageFolder(
            folder=str(train_dir),
            patch_size=patch_size,
            sigma_range=sigma_range,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=canny_low,
            canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            blur_prob=blur_prob,
            blur_kernel_choices=blur_kernels,
            blur_sigma_range=blur_sigma_rng,
            keep_in_ram=keep_in_ram_train,
            noise_type=noise_type,
            noise_params=noise_params,
            seed=seed,
        )

        # Updated for DDP
        train_sampler = DistributedSampler(
            train_ds, num_replicas=ddp_world, rank=ddp_rank, shuffle=True, drop_last=drop_last_train
        ) if use_ddp else None
        
        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=(train_sampler is None),
            sampler=train_sampler,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=drop_last_train if train_sampler is None else False,  # sampler handles drop_last
            **dl_worker_kwargs,
        )

        
        #train_loader = torch.utils.data.DataLoader(
        #    train_ds,
        #    batch_size=batch_size,
        #    shuffle=True,
        #    num_workers=num_workers,
        #    pin_memory=pin_memory,
        #    drop_last=drop_last_train,
        #)

        if val_dir:
            val_ds = CleanImageFolder(
                folder=str(val_dir),
                patch_size=patch_size,
                sigma_range=sigma_range,
                extra_edges=extra_edges,
                edge_from=edge_from,
                canny_low=canny_low,
                canny_high=canny_high,
                sobel_ksize=sobel_ksize,
                blur_prob=0.0,  # deterministic validation
                blur_kernel_choices=blur_kernels,
                blur_sigma_range=blur_sigma_rng,
                keep_in_ram=keep_in_ram_val,
                noise_type=noise_type,
                noise_params=noise_params,
                seed=seed + 10_000,
            )

            # Updated for DDP
            val_sampler = DistributedSampler(
                val_ds, num_replicas=ddp_world, rank=ddp_rank, shuffle=False, drop_last=False
            ) if use_ddp else None
            
            val_loader = torch.utils.data.DataLoader(
                val_ds,
                batch_size=batch_size,
                shuffle=False,
                sampler=val_sampler,
                num_workers=num_workers,
                pin_memory=pin_memory,
                drop_last=False,
                **dl_worker_kwargs,
            )

            #val_loader = torch.utils.data.DataLoader(
            #    val_ds,
            #    batch_size=batch_size,
            #    shuffle=False,
            #    num_workers=num_workers,
            #    pin_memory=pin_memory,
            #    drop_last=False,
            #)

    elif data_loader_option == 1:
        # -------------------------
        # PairedImageFolder
        # -------------------------
        train_x_dir = kwargs.get("train_x_dir", train_x_dir_default)
        train_y_dir = kwargs.get("train_y_dir", train_y_dir_default)
        val_x_dir   = kwargs.get("val_x_dir", val_x_dir_default)
        val_y_dir   = kwargs.get("val_y_dir", val_y_dir_default)

        degrade_x_train = bool(kwargs.get("degrade_x_train", False))
        assert train_x_dir and train_y_dir, "data_loader_option=1 requires 'train_x_dir' and 'train_y_dir'"

        edge_from = _map_edge_from_for_paired(edge_from_raw)

        train_ds = PairedImageFolder(
            x_folder=str(train_x_dir),
            y_folder=str(train_y_dir),
            patch_size=patch_size,
            degrade_x=degrade_x_train,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            blur_kernel_choices=blur_kernels,
            blur_sigma_range=blur_sigma_rng,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=canny_low,
            canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            seed=seed,
            keep_in_ram=keep_in_ram_train,

            noise_type=noise_type,
            noise_params=noise_params,

        )


        # Updated for DDP
        train_sampler = DistributedSampler(
            train_ds, num_replicas=ddp_world, rank=ddp_rank, shuffle=True, drop_last=drop_last_train
        ) if use_ddp else None
        
        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=(train_sampler is None),
            sampler=train_sampler,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=drop_last_train if train_sampler is None else False,  # sampler handles drop_last
            **dl_worker_kwargs,
        )
        
        #train_loader = torch.utils.data.DataLoader(
        #    train_ds,
        #    batch_size=batch_size,
        #    shuffle=True,
        #    num_workers=num_workers,
        #    pin_memory=pin_memory,
        #    drop_last=drop_last_train,
        #)

        if val_x_dir and val_y_dir:
            val_ds = PairedImageFolder(
                x_folder=str(val_x_dir),
                y_folder=str(val_y_dir),
                patch_size=patch_size,
                degrade_x=False,
                sigma_range=sigma_range,
                blur_prob=0.0,
                blur_kernel_choices=blur_kernels,
                blur_sigma_range=blur_sigma_rng,
                extra_edges=extra_edges,
                edge_from=edge_from,
                canny_low=canny_low,
                canny_high=canny_high,
                sobel_ksize=sobel_ksize,
                seed=seed + 10_000,
                keep_in_ram=keep_in_ram_val,

                noise_type=noise_type,
                noise_params=noise_params,
            )

            # Updated for DDP
            val_sampler = DistributedSampler(
                val_ds, num_replicas=ddp_world, rank=ddp_rank, shuffle=False, drop_last=False
            ) if use_ddp else None
            
            val_loader = torch.utils.data.DataLoader(
                val_ds,
                batch_size=batch_size,
                shuffle=False,
                sampler=val_sampler,
                num_workers=num_workers,
                pin_memory=pin_memory,
                drop_last=False,
                **dl_worker_kwargs,
            )
            
            #val_loader = torch.utils.data.DataLoader(
            #    val_ds,
            #    batch_size=batch_size,
            #    shuffle=False,
            #    num_workers=num_workers,
            #    pin_memory=pin_memory,
            #    drop_last=False,
            #)

    else:
        raise ValueError(f"Unknown data_loader_option={data_loader_option} (expected 0 or 1).")

    return train_loader, val_loader


# Backward-compatible alias
#data_loader_v0 = data_loader


In [16]:
import torch
from torch.utils.data import Dataset


#=======================================================================================================
#=======================================================================================================

def _imread_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#==================
def _to_gray01_from_chw_rgb01(t_chw01: torch.Tensor) -> np.ndarray:
    """[0,1] CHW RGB tensor -> grayscale float32 [0,1] numpy (H,W)."""
    a = t_chw01.permute(1, 2, 0).cpu().numpy().astype(np.float32)  # HWC
    gray = 0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]
    return gray.astype(np.float32)
#=======================================

In [17]:
#=======================================
class PairedImageFolder_v0(Dataset):
    """
    Loads paired images from two folders (X and Y) matched by filename (stem).
      - Optionally degrades X (blur -> noise)
      - Optionally appends edge channels from 'degraded' | 'x' | 'y'
      - Returns (x_in[, +edges], y_clean)

    Example pairs:
      x_folder/foo/bar/IMG_001.png  <->  y_folder/IMG_001.jpg
    Matching is by lowercase stem ('img_001'), ignoring extension and subdirs.
    """
    def __init__(self,
                 x_folder: str,
                 y_folder: str,
                 patch_size: int = 128,
                 # degradation on X
                 degrade_x: bool = True,              # Degrade ************
                 sigma_range=(0.0, 25.0),            # noise std in 0..255 space; converted to 0..1
                 blur_prob: float = 0.5,
                 blur_kernel_choices=(3,5,7,9),      # odd sizes
                 blur_sigma_range=(0.5, 2.0),        # spatial sigma in pixels
                 # edge channels
                 extra_edges=('canny', 'sobel', 'prewitt'),
                 edge_from: str = 'degraded',        # 'degraded' | 'x' | 'y'
                 canny_low: int = 200, canny_high: int = 255,
                 sobel_ksize: int = 9,
                 # misc
                 return_paths: bool = False,
                 seed: int = 0):
        super().__init__()

        def _index(folder):
            exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
            files = [p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                     if p.lower().endswith(exts)]
            by_stem = {}
            for p in files:
                stem = os.path.splitext(os.path.basename(p))[0].lower()
                by_stem[stem] = p
            return by_stem

        x_map = _index(x_folder)
        y_map = _index(y_folder)
        # intersection of stems
        common = sorted(set(x_map.keys()) & set(y_map.keys()))
        if not common:
            raise RuntimeError(f"No paired images found between {x_folder} and {y_folder}")

        self.pairs = [(x_map[s], y_map[s]) for s in common]
        self.patch_size = patch_size if patch_size is None or patch_size > 0 else None

        # degradation
        self.degrade_x = bool(degrade_x)
        self.sigma_range = tuple(float(v) for v in sigma_range)
        self.blur_prob = float(blur_prob)
        self.blur_kernel_choices = tuple(int(k) | 1 for k in blur_kernel_choices)  # force odd
        self.blur_sigma_range = (float(blur_sigma_range[0]), float(blur_sigma_range[1]))

        # edges
        self.extra_edges = tuple(extra_edges) if extra_edges else ()
        self.edge_from = edge_from.lower()
        assert self.edge_from in ('degraded', 'x', 'y')
        self.canny_low = int(canny_low)
        self.canny_high = int(canny_high)
        self.sobel_ksize = int(max(3, sobel_ksize | 1))  # odd >=3

        self.return_paths = bool(return_paths)
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.pairs)

    def _choose_crop(self, H1W1, H2W2, ps):
        """Choose a top-left crop (top,left) ensuring both can be cropped to ps."""
        if ps is None:
            return None
        H1, W1 = H1W1; H2, W2 = H2W2
        if H1 < ps or W1 < ps or H2 < ps or W2 < ps:
            # one (or both) is smaller → use center crop later
            return None
        top = self.rng.randint(0, min(H1, H2) - ps)
        left = self.rng.randint(0, min(W1, W2) - ps)
        return top, left

    def _center_crop_if_needed(self, t, ps):
        if ps is None: return t
        _, H, W = t.shape
        if H < ps or W < ps:
            return TF.center_crop(t, [ps, ps])
        return t

    def _apply_crop(self, t, crop):
        if crop is None or self.patch_size is None:
            return self._center_crop_if_needed(t, self.patch_size)
        top, left = crop
        ps = self.patch_size
        return t[:, top:top+ps, left:left+ps]

    def _edges_from_tensor_rgb01(self, t_rgb01: torch.Tensor):
        """Return list of [1,H,W] edge tensors in [0,1]."""
        outs = []
        if not self.extra_edges:
            return outs
        gray01 = _to_gray01_from_chw_rgb01(t_rgb01)
        gray8 = np.clip(gray01 * 255.0, 0, 255).astype(np.uint8)

        for m in self.extra_edges:
            m = m.lower()
            if m == 'canny':
                e = cv2.Canny(gray8, self.canny_low, self.canny_high).astype(np.float32) / 255.0
            elif m == 'sobel':
                gx = cv2.Sobel(gray8, cv2.CV_32F, 1, 0, ksize=self.sobel_ksize)
                gy = cv2.Sobel(gray8, cv2.CV_32F, 0, 1, ksize=self.sobel_ksize)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            elif m == 'prewitt':
                kx = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
                ky = np.array([[ 1,1,1],[ 0,0,0],[-1,-1,-1]], dtype=np.float32)
                gx = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, kx)
                gy = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, ky)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            else:
                raise ValueError(f"Unsupported edge method: {m}")
            outs.append(torch.from_numpy(e).unsqueeze(0))
        return outs

    def __getitem__(self, idx):
        x_path, y_path = self.pairs[idx]
        X = torch.from_numpy(_imread_rgb(x_path)).permute(2, 0, 1).float() / 255.0
        Y = torch.from_numpy(_imread_rgb(y_path)).permute(2, 0, 1).float() / 255.0

        # sample a shared crop (if possible)
        crop = self._choose_crop((X.shape[1], X.shape[2]), (Y.shape[1], Y.shape[2]), self.patch_size)
        X = self._apply_crop(X, crop)
        Y = self._apply_crop(Y, crop)

        # degrade X (blur -> noise)
        Xin = X
        if self.degrade_x:
            if self.blur_prob > 0.0 and self.rng.random() < self.blur_prob:
                k = self.rng.choice(self.blur_kernel_choices)
                sig = self.rng.uniform(*self.blur_sigma_range)
                Xin = TF.gaussian_blur(Xin, kernel_size=[k, k], sigma=sig)
            sigma = self.rng.uniform(*self.sigma_range) / 255.0
            Xin = (Xin + torch.randn_like(Xin) * sigma).clamp(0, 1)

        # edge channels (from selected source)
        if self.extra_edges:
            if self.edge_from == 'degraded':
                src = Xin
            elif self.edge_from == 'x':
                src = X
            else:
                src = Y
            edges = self._edges_from_tensor_rgb01(src)
            if edges:
                Xin = torch.cat([Xin, torch.cat(edges, dim=0)], dim=0)

        if self.return_paths:
            return Xin, Y, (x_path, y_path)
        return Xin, Y
#=======================================================================================================
#=======================================================================================================


class PairedImageFolder(PairedImageFolder_v0):
    """
    Same as PairedImageFolder_v0 but with optional RAM caching (keep_in_ram).
    """

    def __init__(
        self,
        x_folder: str,
        y_folder: str,
        patch_size: int = 128,
        degrade_x: bool = True,              # Degrade ************
        sigma_range=(0.0, 25.0),
        blur_prob: float = 0.5,              # blur_prob ************
        blur_kernel_choices=(3, 5, 7, 9),
        blur_sigma_range=(0.5, 2.0),
        extra_edges=('canny', 'sobel', 'prewitt'),
        edge_from: str = 'degraded',
        canny_low: int = 200,
        canny_high: int = 255,
        sobel_ksize: int = 9,
        return_paths: bool = False,
        seed: int = 0,
        keep_in_ram: bool = False,   # <<< NEW
    ):
        # Initialize exactly as v0 (this sets pairs, rng, etc.)
        super().__init__(
            x_folder=x_folder,
            y_folder=y_folder,
            patch_size=patch_size,
            degrade_x=degrade_x,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            blur_kernel_choices=blur_kernel_choices,
            blur_sigma_range=blur_sigma_range,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=canny_low,
            canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            return_paths=return_paths,
            seed=seed,
        )

        # RAM cache
        self.keep_in_ram = bool(keep_in_ram)
        self._x_cache = None
        self._y_cache = None

        if self.keep_in_ram:
            self._x_cache = []
            self._y_cache = []
            for x_path, y_path in self.pairs:
                X = torch.from_numpy(_imread_rgb(x_path)).permute(2, 0, 1).float() / 255.0
                Y = torch.from_numpy(_imread_rgb(y_path)).permute(2, 0, 1).float() / 255.0
                self._x_cache.append(X)
                self._y_cache.append(Y)
            print(f"[PairedImageFolder] Cached {len(self.pairs)} pairs into RAM.")

    def __getitem__(self, idx):
        x_path, y_path = self.pairs[idx]

        # Use RAM cache if available
        if self.keep_in_ram and self._x_cache is not None:
            X = self._x_cache[idx].clone()
            Y = self._y_cache[idx].clone()
        else:
            X = torch.from_numpy(_imread_rgb(x_path)).permute(2, 0, 1).float() / 255.0
            Y = torch.from_numpy(_imread_rgb(y_path)).permute(2, 0, 1).float() / 255.0

        # Everything else is exactly like v0
        crop = self._choose_crop((X.shape[1], X.shape[2]),
                                 (Y.shape[1], Y.shape[2]),
                                 self.patch_size)
        X = self._apply_crop(X, crop)
        Y = self._apply_crop(Y, crop)

        Xin = X
        if self.degrade_x:
            if self.blur_prob > 0.0 and self.rng.random() < self.blur_prob:
                k = self.rng.choice(self.blur_kernel_choices)
                sig = self.rng.uniform(*self.blur_sigma_range)
                Xin = TF.gaussian_blur(Xin, kernel_size=[k, k], sigma=sig)
            sigma = self.rng.uniform(*self.sigma_range) / 255.0
            Xin = (Xin + torch.randn_like(Xin) * sigma).clamp(0, 1)

        if self.extra_edges:
            if self.edge_from == 'degraded':
                src = Xin
            elif self.edge_from == 'x':
                src = X
            else:
                src = Y
            edges = self._edges_from_tensor_rgb01(src)
            if edges:
                Xin = torch.cat([Xin, torch.cat(edges, dim=0)], dim=0)

        if self.return_paths:
            return Xin, Y, (x_path, y_path)
        return Xin, Y

#=======================================================================================================
#=======================================================================================================

In [18]:
class CleanImageFolder(Dataset):
    def __init__(self,
                 folder,
                 patch_size=128,
                 sigma_range=(0.0, 25.0),            # noise std in 0..255 space
                 extra_edges=('canny','sobel','prewitt'),
                 edge_from='noisy',                  # 'noisy' or 'clean'
                 canny_low=200, canny_high=255,
                 sobel_ksize=9,
                 # --- blur knobs ---
                 blur_prob=0.5,
                 blur_kernel_choices=(3,5,7,9),
                 blur_sigma_range=(0.5, 2.0),
                 keep_in_ram: bool = False          # NEW
                 ):
        """
        Returns (degraded_plus_edges, clean_rgb).

        Degradation = optional Gaussian blur (prob=blur_prob) THEN additive Gaussian noise.
        extra_edges: any subset of ('canny','sobel','prewitt') or None/().
        edge_from: compute edges from 'noisy' (degraded) or 'clean'.
        """
        exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
        self.files = [p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                     if p.lower().endswith(exts)]
        #self.files = sorted([f for f in glob.glob(os.path.join(folder, "*")) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
        if not self.files:
            raise RuntimeError(f"No images found in {folder}")

        self.patch_size   = patch_size
        self.sigma_range  = sigma_range
        self.rng          = random.Random(0)

        self.extra_edges  = tuple(extra_edges) if extra_edges else ()
        self.edge_from    = edge_from.lower()
        assert self.edge_from in ('noisy', 'clean')

        # edge params
        self.canny_low    = int(canny_low)
        self.canny_high   = int(canny_high)
        self.sobel_ksize  = int(max(3, sobel_ksize | 1))  # force odd >=3

        # blur params
        self.blur_prob = float(blur_prob)
        self.blur_kernel_choices = tuple(int(k) | 1 for k in blur_kernel_choices)  # keep odd
        self.blur_sigma_range = (float(blur_sigma_range[0]), float(blur_sigma_range[1]))

        # ---- RAM cache ----
        self.keep_in_ram = bool(keep_in_ram)
        self._img_cache = None
        if self.keep_in_ram:
            self._img_cache = []
            for path in self.files:
                img = cv2.imread(path, cv2.IMREAD_COLOR)
                if img is None:
                    raise FileNotFoundError(f"Could not read {path}")
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
                self._img_cache.append(t)
            print(f"[CleanImageFolder] Cached {len(self.files)} images into RAM.")

    def __len__(self):
        return len(self.files)

    def _random_crop(self, img, ps):
        _, H, W = img.shape
        if ps is None:
            return img
        if H < ps or W < ps:
            # pad/center-crop to ps if image smaller than patch
            return TF.center_crop(img, [ps, ps])
        top  = self.rng.randint(0, H - ps)
        left = self.rng.randint(0, W - ps)
        return img[:, top:top+ps, left:left+ps]

    @staticmethod
    def _to_cv_gray01(t_chw01: torch.Tensor) -> np.ndarray:
        """[0,1] CHW RGB tensor -> grayscale float32 [0,1] numpy (H,W)."""
        a = t_chw01.permute(1, 2, 0).cpu().numpy().astype(np.float32)  # HWC
        gray = 0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]
        return gray.astype(np.float32)

    def _edges_from_tensor_rgb01(self, t_rgb01: torch.Tensor) -> list[torch.Tensor]:
        """Compute selected edge maps from [0,1] CHW RGB tensor; returns list of [1,H,W] tensors."""
        gray01 = self._to_cv_gray01(t_rgb01)
        gray8  = np.clip(gray01 * 255.0, 0, 255).astype(np.uint8)
        outs = []

        for m in self.extra_edges:
            m = m.lower()
            if m == 'canny':
                e = cv2.Canny(gray8, self.canny_low, self.canny_high)
                e = (e.astype(np.float32) / 255.0)
            elif m == 'sobel':
                gx = cv2.Sobel(gray8, cv2.CV_32F, 1, 0, ksize=self.sobel_ksize)
                gy = cv2.Sobel(gray8, cv2.CV_32F, 0, 1, ksize=self.sobel_ksize)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            elif m == 'prewitt':
                kx = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
                ky = np.array([[ 1,1,1],[ 0,0,0],[-1,-1,-1]], dtype=np.float32)
                gx = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, kx)
                gy = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, ky)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            else:
                raise ValueError(f"Unsupported edge method: {m}")

            outs.append(torch.from_numpy(e).unsqueeze(0))  # [1,H,W]

        return outs

    def __getitem__(self, idx):
        path = self.files[idx]

        # 1) Read clean image (from RAM if available)
        if self.keep_in_ram and self._img_cache is not None:
            t = self._img_cache[idx].clone()  # clone so we can safely crop / modify
        else:
            img = cv2.imread(path, cv2.IMREAD_COLOR)
            if img is None:
                raise FileNotFoundError(f"Could not read {path}")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        # 2) Random crop
        t = self._random_crop(t, self.patch_size)

        # 2.5) Optional Gaussian blur on clean t (before noise)
        degraded = t
        if self.blur_prob > 0.0 and self.rng.random() < self.blur_prob:
            k = self.rng.choice(self.blur_kernel_choices)
            sig = self.rng.uniform(*self.blur_sigma_range)
            degraded = TF.gaussian_blur(degraded, kernel_size=[k, k], sigma=sig)

        # 3) Add Gaussian noise (convert std from 0..255 to 0..1)
        sigma = self.rng.uniform(*self.sigma_range) / 255.0
        degraded = (degraded + torch.randn_like(degraded) * sigma).clamp(0, 1)

        # 4) Build edge channels from chosen source
        src_for_edges = degraded if self.edge_from == 'noisy' else t
        edge_tensors  = self._edges_from_tensor_rgb01(src_for_edges) if self.extra_edges else []

        # 5) Concatenate edges
        if edge_tensors:
            edges_chw = torch.cat(edge_tensors, dim=0)
            inp = torch.cat([degraded, edges_chw], 0)  # [3+E,H,W]
        else:
            inp = degraded

        return inp, t



class CleanImageFolder_v0(Dataset):
    def __init__(self,
                 folder,
                 patch_size=128,
                 sigma_range=(0.0, 25.0),            # noise std in 0..255 space
                 extra_edges=('canny','sobel','prewitt'),
                 edge_from='noisy',                  # 'noisy' or 'clean'
                 canny_low=200, canny_high=255,
                 sobel_ksize=9,
                 # --- NEW blur knobs ---
                 blur_prob=0.5,
                 blur_kernel_choices=(3,5,7,9),
                 blur_sigma_range=(0.5, 2.0)):
        """
        Returns (degraded_plus_edges, clean_rgb).

        Degradation = optional Gaussian blur (prob=blur_prob) THEN additive Gaussian noise.
        extra_edges: any subset of ('canny','sobel','prewitt') or None/().
        edge_from: compute edges from 'noisy' (degraded) or 'clean'.
        """
        self.files = sorted([f for f in glob.glob(os.path.join(folder, "*"))
                             if f.lower().endswith((".jpg", ".png", ".jpeg"))])
        if not self.files:
            raise RuntimeError(f"No images found in {folder}")

        self.patch_size   = patch_size
        self.sigma_range  = sigma_range
        self.rng          = random.Random(0)

        self.extra_edges  = tuple(extra_edges) if extra_edges else ()
        self.edge_from    = edge_from.lower()
        assert self.edge_from in ('noisy', 'clean')

        # edge params
        self.canny_low    = int(canny_low)
        self.canny_high   = int(canny_high)
        self.sobel_ksize  = int(max(3, sobel_ksize | 1))  # force odd >=3

        # blur params
        self.blur_prob = float(blur_prob)
        self.blur_kernel_choices = tuple(int(k) | 1 for k in blur_kernel_choices)  # keep odd
        self.blur_sigma_range = (float(blur_sigma_range[0]), float(blur_sigma_range[1]))

    def __len__(self):
        return len(self.files)

    def _random_crop(self, img, ps):
        _, H, W = img.shape
        if ps is None:
            return img
        if H < ps or W < ps:
            # pad/center-crop to ps if image smaller than patch
            return TF.center_crop(img, [ps, ps])
        top  = self.rng.randint(0, H - ps)
        left = self.rng.randint(0, W - ps)
        return img[:, top:top+ps, left:left+ps]

    @staticmethod
    def _to_cv_gray01(t_chw01: torch.Tensor) -> np.ndarray:
        """[0,1] CHW RGB tensor -> grayscale float32 [0,1] numpy (H,W)."""
        a = t_chw01.permute(1, 2, 0).cpu().numpy().astype(np.float32)  # HWC
        gray = 0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]
        return gray.astype(np.float32)

    def _edges_from_tensor_rgb01(self, t_rgb01: torch.Tensor) -> list[torch.Tensor]:
        """Compute selected edge maps from [0,1] CHW RGB tensor; returns list of [1,H,W] tensors."""
        gray01 = self._to_cv_gray01(t_rgb01)
        gray8  = np.clip(gray01 * 255.0, 0, 255).astype(np.uint8)
        outs = []

        for m in self.extra_edges:
            m = m.lower()
            if m == 'canny':
                e = cv2.Canny(gray8, self.canny_low, self.canny_high)
                e = (e.astype(np.float32) / 255.0)
            elif m == 'sobel':
                gx = cv2.Sobel(gray8, cv2.CV_32F, 1, 0, ksize=self.sobel_ksize)
                gy = cv2.Sobel(gray8, cv2.CV_32F, 0, 1, ksize=self.sobel_ksize)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            elif m == 'prewitt':
                kx = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
                ky = np.array([[ 1,1,1],[ 0,0,0],[-1,-1,-1]], dtype=np.float32)
                gx = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, kx)
                gy = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, ky)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            else:
                raise ValueError(f"Unsupported edge method: {m}")

            outs.append(torch.from_numpy(e).unsqueeze(0))  # [1,H,W]

        return outs

    def __getitem__(self, idx):
        # 1) Read clean image as RGB with OpenCV
        path = self.files[idx]
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Could not read {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 2) To tensor [3,H,W] in [0,1]
        t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        # 3) Random crop
        t = self._random_crop(t, self.patch_size)

        # 3.5) Optional Gaussian blur on clean t (before noise)
        degraded = t
        if self.blur_prob > 0.0 and self.rng.random() < self.blur_prob:
            k = self.rng.choice(self.blur_kernel_choices)
            sig = self.rng.uniform(*self.blur_sigma_range)
            degraded = TF.gaussian_blur(degraded, kernel_size=[k, k], sigma=sig)

        # 4) Add Gaussian noise (convert std from 0..255 to 0..1)
        sigma = self.rng.uniform(*self.sigma_range) / 255.0
        degraded = (degraded + torch.randn_like(degraded) * sigma).clamp(0, 1)

        # 5) Build edge channels from chosen source
        src_for_edges = degraded if self.edge_from == 'noisy' else t
        edge_tensors  = self._edges_from_tensor_rgb01(src_for_edges) if self.extra_edges else []

        # 6) Concatenate edges
        if edge_tensors:
            edges_chw = torch.cat(edge_tensors, dim=0)
            inp = torch.cat([degraded, edges_chw], 0)  # [3+E,H,W]
        else:
            inp = degraded

        # Return (degraded input, clean target)
        return inp, t

In [19]:


#=======================================
class CleanImageFolder_onlynoise(Dataset):
    def __init__(self,
                 folder,
                 patch_size=128,
                 sigma_range=(0.0, 25.0),
                 extra_edges=('canny', 'sobel', 'prewitt'),   # which edge channels to add
                 edge_from='noisy',                           # 'noisy' or 'clean'
                 canny_low=200, canny_high=255,
                 sobel_ksize=9):
        """
        Returns (noisy_plus_edges, clean_rgb).

        - extra_edges: any subset of ('canny','sobel','prewitt') or None/()
        - edge_from: compute edges from 'noisy' (default) or 'clean'
        - canny_low/high, sobel_ksize: standard defaults
        """
        self.files = sorted([f for f in glob.glob(os.path.join(folder, "*"))
                             if f.lower().endswith((".jpg", ".png", ".jpeg"))])
        if not self.files:
            raise RuntimeError(f"No images found in {folder}")

        self.patch_size   = patch_size
        self.sigma_range  = sigma_range
        self.rng          = random.Random(0)

        self.extra_edges  = tuple(extra_edges) if extra_edges else ()
        self.edge_from    = edge_from.lower()
        assert self.edge_from in ('noisy', 'clean')

        # edge params
        self.canny_low    = int(canny_low)
        self.canny_high   = int(canny_high)
        self.sobel_ksize  = int(max(3, sobel_ksize | 1))  # odd >=3

    def __len__(self):
        return len(self.files)

    def _random_crop(self, img, ps):
        _, H, W = img.shape
        if ps is None: return img
        if H < ps or W < ps:
            return TF.center_crop(img, [ps, ps])
        top  = self.rng.randint(0, H - ps)
        left = self.rng.randint(0, W - ps)
        return img[:, top:top+ps, left:left+ps]

    @staticmethod
    def _to_cv_gray01(t_chw01: torch.Tensor) -> np.ndarray:
        """Convert [0,1] RGB CHW tensor → grayscale float32 [0,1] numpy."""
        a = t_chw01.permute(1, 2, 0).cpu().numpy().astype(np.float32)  # HWC
        gray = 0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]
        return gray.astype(np.float32)

    def _edges_from_tensor_rgb01(self, t_rgb01: torch.Tensor) -> list[torch.Tensor]:
        """Compute selected edge maps from RGB tensor [0,1] (CHW)."""
        gray01 = self._to_cv_gray01(t_rgb01)
        gray8  = np.clip(gray01 * 255.0, 0, 255).astype(np.uint8)
        outs = []

        for m in self.extra_edges:
            m = m.lower()
            if m == 'canny':
                e = cv2.Canny(gray8, self.canny_low, self.canny_high)
                e = (e.astype(np.float32) / 255.0)
            elif m == 'sobel':
                gx = cv2.Sobel(gray8, cv2.CV_32F, 1, 0, ksize=self.sobel_ksize)
                gy = cv2.Sobel(gray8, cv2.CV_32F, 0, 1, ksize=self.sobel_ksize)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            elif m == 'prewitt':
                kx = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
                ky = np.array([[ 1,1,1],[ 0,0,0],[-1,-1,-1]], dtype=np.float32)
                gx = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, kx)
                gy = cv2.filter2D(gray8.astype(np.float32), cv2.CV_32F, ky)
                mag = np.sqrt(gx*gx + gy*gy)
                mmax = float(mag.max()) if mag.size and mag.max() > 0 else 1.0
                e = (mag / mmax).astype(np.float32)
            else:
                raise ValueError(f"Unsupported edge method: {m}")

            outs.append(torch.from_numpy(e).unsqueeze(0))  # [1,H,W]

        return outs

    def __getitem__(self, idx):
        # 1) Load clean image as RGB using OpenCV
        path = self.files[idx]
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Could not read {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 2) Convert to tensor [3,H,W] in [0,1]
        t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        # 3) Random crop
        t = self._random_crop(t, self.patch_size)

        # 4) Add Gaussian noise
        sigma = self.rng.uniform(*self.sigma_range) / 255.0
        noisy = (t + torch.randn_like(t) * sigma).clamp(0, 1)

        # 5) Build edge channels
        src_for_edges = noisy if self.edge_from == 'noisy' else t
        edge_tensors  = self._edges_from_tensor_rgb01(src_for_edges) if self.extra_edges else []

        # 6) Concatenate edges
        if edge_tensors:
            edges_chw = torch.cat(edge_tensors, dim=0)
            noisy_plus_edges = torch.cat([noisy, edges_chw], 0)
        else:
            noisy_plus_edges = noisy

        # Return noisy (with edges) and clean
        return noisy_plus_edges, t

#=======================================================================================================
#=======================================================================================================


# Small helper if you need to map legacy names
def _map_edge_from(old):
    old = (old or "").lower()
    if old == "noisy":
        return "degraded"
    if old == "clean":
        return "y"
    return old if old in ("degraded", "x", "y") else "degraded"

In [20]:
def data_loader(data_loader_option: int = 0, **kwargs):
    """
    Build train/val DataLoaders.
    data_option=0 -> CleanImageFolder (single folder)
    data_option=1 -> PairedImageFolder (x/y folders)
    Returns (train_loader, val_loader)
    """

    # ---- common hyperparams / knobs (with sensible defaults) ----
    batch_size      = kwargs.get("batch_size", 16)
    num_workers     = kwargs.get("num_workers", n_workers)
    pin_memory      = kwargs.get("pin_memory", True)
    drop_last_train = kwargs.get("drop_last_train", True)

    patch_size      = kwargs.get("patch_size", 128)
    sigma_range     = kwargs.get("sigma_range", (0.0, 0.0))
    blur_prob       = kwargs.get("blur_prob", 0.0)

    extra_edges     = kwargs.get("extra_edges", ())
    edge_from       = kwargs.get("edge_from", "degraded")  # 'degraded' | 'x' | 'y'

    blur_kernels    = kwargs.get("blur_kernels", (3, 5, 7, 9))
    blur_sigma_rng  = kwargs.get("blur_sigma_rng", (0.5, 2.0))

    canny_low       = kwargs.get("canny_low", 200)
    canny_high      = kwargs.get("canny_high", 255)
    sobel_ksize     = kwargs.get("sobel_ksize", 9)

    # NEW: RAM flags
    keep_in_ram_train = kwargs.get("keep_in_ram_train", False)
    keep_in_ram_val   = kwargs.get("keep_in_ram_val", keep_in_ram_train)

    train_loader, val_loader = None, None

    if data_loader_option == 0:
        # ===== CleanImageFolder =====
        train_dir = kwargs.get("train_dir", train_dir_default)
        val_dir   = kwargs.get("val_dir", val_dir_default)
        assert train_dir is not None, "data_option=0 requires 'train_dir'"

        train_ds = CleanImageFolder(
            folder=train_dir,
            patch_size=patch_size,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=canny_low, canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            keep_in_ram=keep_in_ram_train,          # <<< NEW
        )
        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=drop_last_train,
        )

        if val_dir:
            val_ds = CleanImageFolder(
                folder=val_dir,
                patch_size=patch_size,
                sigma_range=sigma_range,
                blur_prob=0.0,
                extra_edges=extra_edges,
                edge_from=edge_from,
                canny_low=canny_low, canny_high=canny_high,
                sobel_ksize=sobel_ksize,
                keep_in_ram=keep_in_ram_val,        # <<< NEW
            )
            val_loader = torch.utils.data.DataLoader(
                val_ds,
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
                pin_memory=pin_memory,
                drop_last=False,
            )

    elif data_loader_option == 1:
        # ===== PairedImageFolder =====
        train_x_dir = kwargs.get("train_x_dir",train_x_dir_default)
        train_y_dir = kwargs.get("train_y_dir",train_y_dir_default)
        val_x_dir   = kwargs.get("val_x_dir", val_x_dir_default)
        val_y_dir   = kwargs.get("val_y_dir", val_y_dir_default)
        degrade_x_train = kwargs.get("degrade_x_train", False)

        assert train_x_dir and train_y_dir, "data_option=1 requires 'train_x_dir' and 'train_y_dir'"

        train_ds = PairedImageFolder(
            x_folder=train_x_dir,
            y_folder=train_y_dir,
            patch_size=patch_size,
            degrade_x=degrade_x_train,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            blur_kernel_choices=blur_kernels,
            blur_sigma_range=blur_sigma_rng,
            extra_edges=extra_edges,
            edge_from=_map_edge_from(edge_from),
            canny_low=canny_low, canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            seed=0,
            keep_in_ram=keep_in_ram_train,         # <<< NEW
        )
        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=drop_last_train,
        )

        if val_x_dir and val_y_dir:
            val_ds = PairedImageFolder(
                x_folder=val_x_dir,
                y_folder=val_y_dir,
                patch_size=patch_size,
                degrade_x=False,
                sigma_range=sigma_range,
                blur_prob=0.0,
                blur_kernel_choices=blur_kernels,
                blur_sigma_range=blur_sigma_rng,
                extra_edges=extra_edges,
                edge_from=_map_edge_from(edge_from),
                canny_low=canny_low, canny_high=canny_high,
                sobel_ksize=sobel_ksize,
                seed=0,
                keep_in_ram=keep_in_ram_val,        # <<< NEW
            )
            val_loader = torch.utils.data.DataLoader(
                val_ds,
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
                pin_memory=pin_memory,
                drop_last=False,
            )

    else:
        raise ValueError(f"Unknown data_option={data_loader_option} (expected 0 or 1).")

    return train_loader, val_loader


def data_loader_v0(data_loader_option: int = 0, **kwargs):
    """
    Build train/val DataLoaders.
    data_option=0 -> CleanImageFolder (single folder)
    data_option=1 -> PairedImageFolder (x/y folders)
    Returns (train_loader, val_loader)
    """

    # ---- common hyperparams / knobs (with sensible defaults) ----
    batch_size      = kwargs.get("batch_size", 16)
    num_workers     = kwargs.get("num_workers", n_workers)
    pin_memory      = kwargs.get("pin_memory", True)
    drop_last_train = kwargs.get("drop_last_train", True)

    patch_size      = kwargs.get("patch_size", 128)
    sigma_range     = kwargs.get("sigma_range", (0.0, 0.0))
    blur_prob       = kwargs.get("blur_prob", 0.0)

    extra_edges     = kwargs.get("extra_edges", ())
    edge_from       = kwargs.get("edge_from", "degraded")  # 'degraded' | 'x' | 'y'

    blur_kernels    = kwargs.get("blur_kernels", (3, 5, 7, 9))
    blur_sigma_rng  = kwargs.get("blur_sigma_rng", (0.5, 2.0))

    canny_low       = kwargs.get("canny_low", 200)
    canny_high      = kwargs.get("canny_high", 255)
    sobel_ksize     = kwargs.get("sobel_ksize", 9)


    train_loader, val_loader = None, None

    if data_loader_option == 0:
        # ===== CleanImageFolder =====
        train_dir = kwargs.get("train_dir", train_dir_default)
        val_dir   = kwargs.get("val_dir", val_dir_default)
        assert train_dir is not None, "data_option=0 requires 'train_dir'"

        train_ds = CleanImageFolder(
            folder=train_dir,
            patch_size=patch_size,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=canny_low, canny_high=canny_high,
            sobel_ksize=sobel_ksize,
        )
        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=drop_last_train,
        )

        if val_dir:
            val_ds = CleanImageFolder(
                folder=val_dir,
                patch_size=patch_size,      # set None to use full images at val
                sigma_range=sigma_range,    # ignored in your dataset if not used
                blur_prob=0.0,              # usually no blur/noise in val
                extra_edges=extra_edges,
                edge_from=edge_from,
                canny_low=canny_low, canny_high=canny_high,
                sobel_ksize=sobel_ksize,
            )
            val_loader = torch.utils.data.DataLoader(
                val_ds,
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
                pin_memory=pin_memory,
                drop_last=False,
            )

    elif data_loader_option == 1:
        # ===== PairedImageFolder =====
        train_x_dir = kwargs.get("train_x_dir",train_x_dir_default)
        train_y_dir = kwargs.get("train_y_dir",train_y_dir_default)
        val_x_dir   = kwargs.get("val_x_dir", val_x_dir_default)
        val_y_dir   = kwargs.get("val_y_dir", val_y_dir_default)
        degrade_x_train = kwargs.get("degrade_x_train", False)

        assert train_x_dir and train_y_dir, "data_option=1 requires 'train_x_dir' and 'train_y_dir'"

        train_ds = PairedImageFolder(
            x_folder=train_x_dir,
            y_folder=train_y_dir,
            patch_size=patch_size,
            degrade_x=degrade_x_train,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            blur_kernel_choices=blur_kernels,
            blur_sigma_range=blur_sigma_rng,
            extra_edges=extra_edges,
            edge_from=_map_edge_from(edge_from),
            canny_low=canny_low, canny_high=canny_high,
            sobel_ksize=sobel_ksize,
            seed=0,
        )
        train_loader = torch.utils.data.DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=drop_last_train,
        )

        if val_x_dir and val_y_dir:
            val_ds = PairedImageFolder(
                x_folder=val_x_dir,
                y_folder=val_y_dir,
                patch_size=patch_size,    # or None to evaluate on full images
                degrade_x=False,          # no synthetic degradation for val
                sigma_range=sigma_range,  # ignored if degrade_x=False
                blur_prob=0.0,
                blur_kernel_choices=blur_kernels,
                blur_sigma_range=blur_sigma_rng,
                extra_edges=extra_edges,
                edge_from=_map_edge_from(edge_from),
                canny_low=canny_low, canny_high=canny_high,
                sobel_ksize=sobel_ksize,
                seed=0,
            )
            val_loader = torch.utils.data.DataLoader(
                val_ds,
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
                pin_memory=pin_memory,
                drop_last=False,
            )

    else:
        raise ValueError(f"Unknown data_option={data_option} (expected 0 or 1).")

    return train_loader, val_loader


# Loss Function ____

In [21]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ============================================================
# SSIM / MS-SSIM (fp32, device-agnostic, DP/DDP safe)
# ============================================================

def _conv2d_reflect(x: torch.Tensor, w: torch.Tensor, groups: int) -> torch.Tensor:
    k = int(w.shape[-1])
    p = k // 2
    x = F.pad(x, (p, p, p, p), mode="reflect")
    return F.conv2d(x, w, groups=groups)

def _make_gaussian_window(win: int, sigma: float, C: int, device, dtype) -> torch.Tensor:
    x = torch.arange(win, device=device, dtype=dtype) - (win - 1) / 2
    g = torch.exp(-(x ** 2) / (2 * sigma * sigma))
    g = g / g.sum()
    w2d = (g[:, None] @ g[None, :]).expand(C, 1, win, win).contiguous()
    return w2d

def _ssim_components(
    x: torch.Tensor,
    y: torch.Tensor,
    window: torch.Tensor,
    data_range: float,
    K1: float = 0.01,
    K2: float = 0.03,
):
    C = int(x.shape[1])
    conv = lambda t: _conv2d_reflect(t, window, groups=C)

    mu_x = conv(x)
    mu_y = conv(y)

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x2 = conv(x * x) - mu_x2
    sigma_y2 = conv(y * y) - mu_y2
    sigma_xy = conv(x * y) - mu_xy

    C1 = (K1 * data_range) ** 2
    C2 = (K2 * data_range) ** 2

    l = (2 * mu_xy + C1) / (mu_x2 + mu_y2 + C1)
    cs = (2 * sigma_xy + C2) / (sigma_x2 + sigma_y2 + C2)
    return l, cs

class SSIMLoss(nn.Module):
    def __init__(self, data_range=1.0, window_size=11, sigma=1.5, K1=0.01, K2=0.03):
        super().__init__()
        self.data_range = float(data_range)
        self.window_size = int(window_size) | 1  # ensure odd
        self.sigma = float(sigma)
        self.K1 = float(K1)
        self.K2 = float(K2)

        self.register_buffer("_win", torch.empty(0))  # last-built window
        self._cache = dict(C=None, dtype=None, device=None, win=None, sigma=None)

    def _get_window(self, C: int, device, dtype, win: int) -> torch.Tensor:
        stale = (
            self._cache["C"] != C
            or self._cache["device"] != device
            or self._cache["dtype"] != dtype
            or self._cache["win"] != win
            or self._cache["sigma"] != self.sigma
            or self._win.numel() == 0
        )
        if stale:
            self._win = _make_gaussian_window(win, self.sigma, C, device, dtype)
            self._cache.update(dict(C=C, device=device, dtype=dtype, win=win, sigma=self.sigma))
        return self._win

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # Force fp32 for numerical stability (works for CPU/CUDA/DP/DDP)
        x = pred.float().clamp(0, self.data_range)
        y = target.float().clamp(0, self.data_range)
        _, C, H, W = x.shape

        win = min(self.window_size, H, W)
        if win % 2 == 0:
            win -= 1
        pad = win // 2

        if min(H, W) <= pad or win < 3:
            return torch.zeros((), device=x.device, dtype=torch.float32)

        w = self._get_window(int(C), x.device, x.dtype, int(win))
        l, cs = _ssim_components(x, y, w, self.data_range, self.K1, self.K2)
        out32 = 1.0 - (l * cs).mean()
        return out32  # fp32 scalar

class MSSSIMLoss(nn.Module):
    def __init__(
        self,
        data_range=1.0,
        window_size=11,
        sigma=1.5,
        K1=0.01,
        K2=0.03,
        levels=5,
        weights=None,
    ):
        super().__init__()
        self.data_range = float(data_range)
        self.window_size = int(window_size) | 1  # ensure odd
        self.sigma = float(sigma)
        self.K1 = float(K1)
        self.K2 = float(K2)
        self.levels = int(levels)

        if weights is None:
            weights = [0.0448, 0.2856, 0.3001, 0.2363, 0.1333]

        # Ensure weight length >= levels (slice if longer, error if shorter)
        if len(weights) < self.levels:
            raise ValueError(f"msssim weights length ({len(weights)}) must be >= levels ({self.levels})")

        self.register_buffer("weights", torch.tensor(weights, dtype=torch.float32))
        self.register_buffer("_win", torch.empty(0))
        self._cache = dict(C=None, dtype=None, device=None, win=None, sigma=None)

    def _get_window(self, C: int, device, dtype, win: int) -> torch.Tensor:
        stale = (
            self._cache["C"] != C
            or self._cache["device"] != device
            or self._cache["dtype"] != dtype
            or self._cache["win"] != win
            or self._cache["sigma"] != self.sigma
            or self._win.numel() == 0
        )
        if stale:
            self._win = _make_gaussian_window(win, self.sigma, C, device, dtype)
            self._cache.update(dict(C=C, device=device, dtype=dtype, win=win, sigma=self.sigma))
        return self._win

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        x = pred.float().clamp(0, self.data_range)
        y = target.float().clamp(0, self.data_range)
        _, C, H, W = x.shape

        # Need enough pixels to downsample (levels-1) times
        if min(H, W) < 2 ** (self.levels - 1):
            return torch.zeros((), device=x.device, dtype=torch.float32)

        cs_means = []
        xi, yi = x, y

        # use only the first `levels` weights
        wts = self.weights[: self.levels].to(dtype=x.dtype, device=x.device)

        for _ in range(self.levels - 1):
            Hi, Wi = xi.shape[-2:]
            win = min(self.window_size, Hi, Wi)
            if win % 2 == 0:
                win -= 1
            pad = win // 2

            if min(Hi, Wi) <= pad or win < 3:
                return torch.zeros((), device=x.device, dtype=torch.float32)

            w = self._get_window(int(C), xi.device, xi.dtype, int(win))
            _, cs = _ssim_components(xi, yi, w, self.data_range, self.K1, self.K2)
            cs_means.append(cs.clamp(min=1e-6).mean(dim=(2, 3), keepdim=True))

            xi = F.avg_pool2d(xi, 2, 2)
            yi = F.avg_pool2d(yi, 2, 2)

        # last level luminance
        Hi, Wi = xi.shape[-2:]
        win = min(self.window_size, Hi, Wi)
        if win % 2 == 0:
            win -= 1
        pad = win // 2

        if min(Hi, Wi) <= pad or win < 3:
            return torch.zeros((), device=x.device, dtype=torch.float32)

        w = self._get_window(int(C), xi.device, xi.dtype, int(win))
        l, _ = _ssim_components(xi, yi, w, self.data_range, self.K1, self.K2)
        l = l.clamp(min=1e-6).mean(dim=(2, 3), keepdim=True)

        ms = l ** wts[-1]
        for cs_m, w_i in zip(cs_means, wts[:-1]):
            ms = ms * (cs_m ** w_i)

        out32 = 1.0 - ms.mean()
        return out32  # fp32 scalar

# ============================================================
# Pixel losses + PSNR loss
# ============================================================

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = float(eps)

    def forward(self, pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        diff = pred - target
        if mask is not None:
            while mask.dim() < diff.dim():
                mask = mask.unsqueeze(0)
            mask = mask.to(diff.dtype)
            diff = diff * mask
            denom = mask.sum().clamp_min(1.0)
            return torch.sqrt(diff * diff + self.eps**2).sum() / denom
        return torch.mean(torch.sqrt(diff * diff + self.eps**2))

class MSELoss(nn.Module):
    def forward(self, pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        diff = (pred - target) ** 2
        if mask is not None:
            while mask.dim() < diff.dim():
                mask = mask.unsqueeze(0)
            mask = mask.to(diff.dtype)
            diff = diff * mask
            denom = mask.sum().clamp_min(1.0)
            return diff.sum() / denom
        return diff.mean()

def psnr(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0, eps: float = 1e-8) -> torch.Tensor:
    mse = F.mse_loss(pred, target, reduction="mean")
    return 10 * torch.log10((data_range**2) / (mse + eps))

class PSNRLoss(nn.Module):
    def __init__(self, data_range=1.0, eps=1e-6):
        super().__init__()
        self.data_range = float(data_range)
        self.eps = float(eps)

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred = pred.float().clamp(0, self.data_range)
        target = target.float().clamp(0, self.data_range)
        mse = F.mse_loss(pred, target, reduction="mean").clamp(min=self.eps)
        ps = 10.0 * torch.log10((self.data_range**2) / mse)
        return -ps

# ============================================================
# Unified Mixed Loss (fp32 scalar)
# ============================================================

class MixedLoss(nn.Module):
    """
    L = a*Charbonnier + b*(1-SSIM) + c*(1-MS-SSIM) + d*(-PSNR) + e*MSE
    """
    def __init__(
        self,
        a=0.5, b=0.5, c=0.0, d=0.0, e=0.0,
        charbonnier_eps=1e-3,
        ssim_data_range=1.0, ssim_window=11, ssim_sigma=1.5, ssim_K1=0.01, ssim_K2=0.03,
        msssim_levels=5, msssim_weights=None,
        psnr_data_range=1.0,
    ):
        super().__init__()
        self.a, self.b, self.c, self.d, self.e = float(a), float(b), float(c), float(d), float(e)

        self.charb = CharbonnierLoss(eps=charbonnier_eps)
        self.ssim = SSIMLoss(
            data_range=ssim_data_range,
            window_size=ssim_window,
            sigma=ssim_sigma,
            K1=ssim_K1,
            K2=ssim_K2,
        )
        self.mssim = MSSSIMLoss(
            data_range=ssim_data_range,
            window_size=ssim_window,
            sigma=ssim_sigma,
            K1=ssim_K1,
            K2=ssim_K2,
            levels=msssim_levels,
            weights=msssim_weights,
        )
        self.psnr = PSNRLoss(data_range=psnr_data_range)
        self.mse = MSELoss()

    def forward(self, pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        pred32 = pred.float().clamp(0, 1)
        target32 = target.float().clamp(0, 1)

        loss = torch.zeros((), device=pred32.device, dtype=torch.float32)
        if self.a != 0.0:
            loss = loss + self.a * self.charb(pred32, target32, mask=mask)
        if self.b != 0.0:
            loss = loss + self.b * self.ssim(pred32, target32)
        if self.c != 0.0:
            loss = loss + self.c * self.mssim(pred32, target32)
        if self.d != 0.0:
            loss = loss + self.d * self.psnr(pred32, target32)
        if self.e != 0.0:
            loss = loss + self.e * self.mse(pred32, target32, mask=mask)
        return loss  # fp32 scalar

# ============================================================
# KL term for VAE
# ============================================================

def kl_standard_normal(mu: torch.Tensor, logvar: torch.Tensor, reduction: str = "batchmean") -> torch.Tensor:
    # KL[q(z|x)=N(mu, diag(exp(logvar))) || p(z)=N(0, I)]
    if reduction == "batchmean":
        return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)
    elif reduction == "sum":
        return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    elif reduction == "mean":
        return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    else:
        raise ValueError(f"Unknown reduction: {reduction}")

# ============================================================
# Lipschitz LR utilities (safer against overflow)
# ============================================================

def _layer_spectral_norm_linear(W: torch.Tensor) -> torch.Tensor:
    return torch.linalg.matrix_norm(W, ord=2)

def _layer_spectral_norm_conv2d(W: torch.Tensor) -> torch.Tensor:
    W2 = W.flatten(1)
    return torch.linalg.matrix_norm(W2, ord=2)

@torch.no_grad()
def network_L(model: nn.Module, *, depth_normalize: bool = False, max_log_sum: float = 80.0) -> torch.Tensor:
    """
    L <= Π_i ||W_i||_2 over Conv2d/Linear layers (upper bound).
    Uses log-domain accumulation; caps to avoid exp overflow.
    """
    log_sum = 0.0
    n_layers = 0

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            s = _layer_spectral_norm_conv2d(m.weight).clamp(min=1e-12)
            log_sum += float(torch.log(s))
            n_layers += 1
        elif isinstance(m, nn.Linear):
            s = _layer_spectral_norm_linear(m.weight).clamp(min=1e-12)
            log_sum += float(torch.log(s))
            n_layers += 1

    if n_layers == 0:
        return torch.tensor(1.0, dtype=torch.float32)

    if depth_normalize:
        # exp(mean log)
        val = math.exp(max(min(log_sum / n_layers, max_log_sum), -max_log_sum))
    else:
        # exp(sum log) with cap
        val = math.exp(max(min(log_sum, max_log_sum), -max_log_sum))

    return torch.tensor(val, dtype=torch.float32)



# Optimizer ____

In [22]:

class LipschitzLRUpdater:
    def __init__(
        self,
        mode: str = "spectral",
        alpha: float = 0.3,
        min_lr: float = 1e-6,
        max_lr: float = 1e-2,
        eps: float = 1e-3,
        n_samples: int = 20,
        sample_fn=None,
        update_every: int = 5,
        by: str = "epoch",  # 'epoch' or 'step'
        ema_beta: float | None = None,
        depth_normalize: bool = True,  # helps avoid tiny LR on deep nets
    ):
        self.mode = mode.lower()
        self.alpha = float(alpha)
        self.min_lr = float(min_lr)
        self.max_lr = float(max_lr)
        self.eps = float(eps)
        self.n_samples = int(n_samples)
        self.sample_fn = sample_fn

        self.update_every = max(1, int(update_every))
        self.by = by.lower()
        assert self.by in ("epoch", "step")
        self.ema_beta = None if ema_beta is None else float(ema_beta)
        self.depth_normalize = bool(depth_normalize)

        self._last_L: float | None = None
        self._last_lr: float | None = None
        self._last_epoch_update: int = -1
        self._last_step_update: int = -1
        self._ema_lr: float | None = None

    @torch.no_grad()
    def _spectral_L(self, model: nn.Module) -> float:
        L = network_L(model, depth_normalize=self.depth_normalize)
        val = float(L.detach().cpu().item())
        if not math.isfinite(val) or val <= 0:
            val = 1.0
        return val

    def _empirical_L(self, model: nn.Module, kind: str) -> float:
        if self.sample_fn is None:
            raise ValueError("sample_fn must be provided for empirical Lipschitz estimation.")
        x = self.sample_fn()
        if kind == "empirical":
            L = estimate_L_empirical(model, x, eps=self.eps, n_samples=self.n_samples)
        elif kind == "empirical0":
            L = estimate_L_empirical_mse0(model, x, eps=self.eps, n_samples=self.n_samples)
        else:
            raise ValueError(f"Unknown empirical kind: {kind}")
        val = float(L)
        if not math.isfinite(val) or val <= 0:
            val = 1.0
        return val

    def _due(self, epoch: int | None, step: int | None) -> bool:
        if self.by == "epoch":
            e = 0 if epoch is None else int(epoch)
            return (self._last_epoch_update < 0) or ((e - self._last_epoch_update) >= self.update_every)
        s = 0 if step is None else int(step)
        return (self._last_step_update < 0) or ((s - self._last_step_update) >= self.update_every)

    def _record_tick(self, epoch: int | None, step: int | None):
        if self.by == "epoch":
            self._last_epoch_update = 0 if epoch is None else int(epoch)
        else:
            self._last_step_update = 0 if step is None else int(step)

    def _apply_lr(self, optimizer: optim.Optimizer, lr: float):
        if self.ema_beta is not None:
            if self._ema_lr is None:
                self._ema_lr = lr
            else:
                self._ema_lr = self.ema_beta * self._ema_lr + (1.0 - self.ema_beta) * lr
            lr_to_set = self._ema_lr
        else:
            lr_to_set = lr

        for g in optimizer.param_groups:
            g["lr"] = float(lr_to_set)
        self._last_lr = float(lr_to_set)

    def update(
        self,
        model: nn.Module,
        optimizer: optim.Optimizer,
        epoch: int | None = None,
        step: int | None = None,
        force: bool = False,
    ):
        need_update = force or self._due(epoch, step)

        if (self._last_lr is None) or need_update:
            if self.mode == "spectral":
                L = self._spectral_L(model)
            elif self.mode == "empirical":
                L = self._empirical_L(model, "empirical")
            elif self.mode == "empirical0":
                L = self._empirical_L(model, "empirical0")
            else:
                raise ValueError(f"Unsupported Lipschitz mode: {self.mode}")

            lr = self.alpha / max(L, 1e-12)
            lr = float(min(max(lr, self.min_lr), self.max_lr))

            self._apply_lr(optimizer, lr)
            self._last_L = float(L)
            self._record_tick(epoch, step)
            return self._last_lr, self._last_L, True

        return self._last_lr, self._last_L, False




class OptimizerFactory:
    def __init__(
        self,
        name: str = "adamw",
        lr: float = 1e-3,
        weight_decay: float = 1e-4,
        betas=(0.9, 0.999),
        momentum: float = 0.9,
        scheduler: str | None = None,
        warmup_epochs: int = 0,
        max_epochs: int = 100,
        min_lr: float = 1e-6,
        T_max: int | None = None,
        lr_mode: str = "fixed",  # 'fixed' or 'lipschitz'
        lipschitz_mode: str = "spectral",
        lipschitz_alpha: float = 0.3,
        lipschitz_min_lr: float = 1e-6,
        lipschitz_max_lr: float = 1e-2,
        lipschitz_eps: float = 1e-3,
        lipschitz_n_samples: int = 20,
        lipschitz_sample_fn=None,
        lipschitz_update_every: int = 5,
        lipschitz_by: str = "epoch",
        lipschitz_ema_beta: float | None = None,
        lipschitz_depth_normalize: bool = True,
    ):
        self.name = name.lower()
        self.lr = float(lr)
        self.weight_decay = float(weight_decay)
        self.betas = betas
        self.momentum = float(momentum)
        self.scheduler_name = scheduler.lower() if scheduler else None
        self.warmup_epochs = int(warmup_epochs)
        self.max_epochs = int(max_epochs)
        self.min_lr = float(min_lr)
        self.T_max = int(T_max or max_epochs)

        self.lr_mode = lr_mode.lower()
        self.lipschitz_cfg = dict(
            mode=lipschitz_mode,
            alpha=lipschitz_alpha,
            min_lr=lipschitz_min_lr,
            max_lr=lipschitz_max_lr,
            eps=lipschitz_eps,
            n_samples=lipschitz_n_samples,
            sample_fn=lipschitz_sample_fn,
            update_every=lipschitz_update_every,
            by=lipschitz_by,
            ema_beta=lipschitz_ema_beta,
            depth_normalize=lipschitz_depth_normalize,
        )

    def make(self, model: nn.Module):
        if self.name == "adam":
            optimizer = optim.Adam(model.parameters(), lr=self.lr, betas=self.betas, weight_decay=self.weight_decay)
        elif self.name == "adamw":
            optimizer = optim.AdamW(model.parameters(), lr=self.lr, betas=self.betas, weight_decay=self.weight_decay)
        elif self.name == "sgd":
            optimizer = optim.SGD(model.parameters(), lr=self.lr, momentum=self.momentum, weight_decay=self.weight_decay)
        else:
            raise ValueError(f"Unsupported optimizer name: {self.name}")

        scheduler = None
        if self.scheduler_name:
            scheduler = self._make_scheduler(optimizer)

        lipschitz_updater = None
        if self.lr_mode == "lipschitz":
            lipschitz_updater = LipschitzLRUpdater(**self.lipschitz_cfg)
            lipschitz_updater.update(model, optimizer, force=True)

        return optimizer, scheduler, lipschitz_updater

    def _make_scheduler(self, optimizer: optim.Optimizer):
        if self.scheduler_name == "step":
            return optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)
        if self.scheduler_name == "cosine":
            return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.T_max, eta_min=self.min_lr)
        if self.scheduler_name == "onecycle":
            return optim.lr_scheduler.OneCycleLR(optimizer, max_lr=self.lr, epochs=self.max_epochs, steps_per_epoch=1)
        if self.scheduler_name == "plateau":
            return optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", factor=0.5, patience=5, min_lr=self.min_lr
            )
        raise ValueError(f"Unsupported scheduler: {self.scheduler_name}")

# Important Functions ****

In [23]:
model_width = 64
model_depth = 10
extra_edges= ()
base_lr = 1e-3
l2_weight_decay = 1e-4
#==========================================================================
#==========================================================================
def create_lipschitz():
    lips =  LipschitzLRUpdater(
        mode="spectral",
        alpha=0.3,
        min_lr=1e-6,
        max_lr=3e-3,
        update_every=5,   # recompute every 5 epochs
        by="epoch",
        ema_beta=0.8      # smooth LR changes (optional)
    )
    return lips




#==========================================================================
def get_optimizer(model, optimizer_option: int = 0, lr: float = 1e-3, weight_decay: float = l2_weight_decay, **kwargs):
    """
    Create optimizer by integer code.

    Args:
        model: PyTorch model
        optimizer_option (int):
            0 = AdamW
            1 = Adam
            2 = SGD
            3 = RMSprop
            4 = Adamax
            5 = NAdam
            6 = Adagrad
            7 = Adadelta
            8 = Lion  (PyTorch >= 2.2)
            9 = LBFGS
        lr (float): learning rate
        weight_decay (float): weight decay
        **kwargs: extra args like momentum, betas, etc.

    Returns:
        torch.optim.Optimizer
    """
    params = model.parameters()

    if optimizer_option == 0:
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay, betas=(0.9, 0.999), **kwargs)

    elif optimizer_option == 1:
        optimizer = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay, **kwargs)

    elif optimizer_option == 2:
        optimizer = torch.optim.SGD(params, lr=lr, weight_decay=weight_decay, **kwargs)

    elif optimizer_option == 3:
        optimizer = torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay, **kwargs)

    elif optimizer_option == 4:
        optimizer = torch.optim.Adamax(params, lr=lr, weight_decay=weight_decay, **kwargs)

    elif optimizer_option == 5:
        optimizer = torch.optim.NAdam(params, lr=lr, weight_decay=weight_decay, **kwargs)

    elif optimizer_option == 6:
        optimizer = torch.optim.Adagrad(params, lr=lr, weight_decay=weight_decay, **kwargs)

    elif optimizer_option == 7:
        optimizer = torch.optim.Adadelta(params, lr=lr, weight_decay=weight_decay, **kwargs)

    elif optimizer_option == 8:
        if hasattr(torch.optim, "Lion"):
            optimizer = torch.optim.Lion(params, lr=lr, weight_decay=weight_decay, **kwargs)
        else:
            raise RuntimeError("torch.optim.Lion not available in this PyTorch version.")

    elif optimizer_option == 9:
        optimizer = torch.optim.LBFGS(params, lr=lr, **kwargs)

    else:
        raise ValueError(f"Unsupported optimizer_option: {optimizer_option}")

    print(f"[Optimizer] Using option {optimizer_option}: {optimizer.__class__.__name__}")
    return optimizer


#==========================================================================
def create_optimizer(
    model,
    optimizer_option: int = 1,
    scheduler_option: int = 1,
    base_lr: float = base_lr,
    weight_decay: float = l2_weight_decay,
    *,
    total_epochs: int | None = epochs,     # for cosine / onecycle
    steps_per_epoch: int = 1,            # for onecycle
    onecycle_max_lr: float | None = None, # for onecycle (defaults to base_lr if None)
):

    optimizer = get_optimizer(model, optimizer_option, lr=base_lr, weight_decay=weight_decay, )

    # 0) no scheduler
    if scheduler_option == 0:
        return base_lr, optimizer, None

    # 1) Lipschitz updater (not a torch scheduler; you call lips.update(...) each epoch)
    if scheduler_option == 1:
        lips = create_lipschitz()
        lr0, L0, _ = lips.update(model, optimizer, epoch=0)  # apply once before training
        print(f"[Init] Spectral L={L0:.3e} | LR set to {lr0:.3e}")
        return base_lr, optimizer, lips

    # 2) Cosine decay over the whole run
    if scheduler_option == 2:
        T = total_epochs if total_epochs is not None else 250  # <-- fix: use total_epochs
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T, eta_min=1e-6)

    # 3) StepLR every 50 epochs
    elif scheduler_option == 3:
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

    # 4) ReduceLROnPlateau (step with val_loss after validation)
    elif scheduler_option == 4:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
        )

    # 5) OneCycleLR (needs total_epochs and steps_per_epoch)
    elif scheduler_option == 5:
        max_lr = onecycle_max_lr if onecycle_max_lr is not None else base_lr
        if total_epochs is None:
            raise ValueError("OneCycleLR requires total_epochs to be set.")
        if steps_per_epoch <= 0:
            raise ValueError("OneCycleLR requires steps_per_epoch > 0.")
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=max_lr,
            epochs=total_epochs,
            steps_per_epoch=steps_per_epoch,
            pct_start=0.3,
            anneal_strategy="cos",
            div_factor=25.0,          # optional: start lr = max_lr/div_factor
            final_div_factor=1e4,     # optional: final lr = max_lr/final_div_factor
        )

    else:
        raise ValueError(f"Unsupported scheduler option: {scheduler_option}")

    return base_lr, optimizer, scheduler


#==========================================================================
def sanity_check(model, loader, loss_fn, device):
    model.train()
    X, Y = next(iter(loader))
    X = X.to(device); Y = Y.to(device)
    with torch.amp.autocast('cuda', enabled=False):

        Yhat, _ = model(X)
        Yhat = Yhat.clamp(0,1)
        loss = loss_fn(Yhat, Y)
    print("probe loss:", float(loss))
    assert torch.isfinite(loss), "loss not finite in probe"
    loss.backward()
    for n,p in model.named_parameters():
        if p.grad is None: continue
        if not torch.isfinite(p.grad).all():
            raise RuntimeError(f"non-finite grads in {n}")
    model.zero_grad(set_to_none=True)
    #sanity_check(model, train_loader, loss_fn, device)
    print("✅ probe ok")




#==========================================================================
import os, numpy as np, torch
from tqdm import tqdm
from torch.cuda import amp

def train_loop_fcn(
    model, optimizer, loss_fn, train_loader, val_loader, device,
    epochs, use_amp, in_ch, model_n,
    scheduler_option,
    checkpoint_path, checkpoint_path_best, history_path, history_path_best,
    scheduler=None, lips=None
):
    best_val_loss = float('inf')
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0

        # --- optional Lipschitz LR update (only when scheduler_option == 1) ---
        if scheduler_option == 1 and lips is not None:
            lr_now, L_now, updated = lips.update(model, optimizer, epoch=epoch)
            if updated:
                print(f"[Lipschitz] epoch {epoch}: L={L_now:.3e} | LR={lr_now:.3e}")

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for X, Y in pbar:
            X = X.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                #with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp and X.device.type=='cuda'):
                with torch.autocast( device_type=device_t.type,     dtype=torch.float16,     enabled=use_amp_runtime, ): 
                    Yhat = model(X)
                    loss = loss_fn(Yhat, Y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                Yhat = model(X)
                loss = loss_fn(Yhat, Y)
                loss.backward()
                optimizer.step()

            running += float(loss.item())
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}"
            )

        avg_train = running / max(1, len(train_loader))
        history["train_loss"].append(avg_train)
        print(f"Epoch {epoch} | train loss: {avg_train:.4f}")

        # ---------- validation ----------
        val_loss = None
        if val_loader is not None:
            model.eval()
            val_running = 0.0
            with torch.inference_mode():
                for X, Y in val_loader:
                    X = X.to(device, non_blocking=True)
                    Y = Y.to(device, non_blocking=True)
                    Yhat = model(X).clamp(0, 1)
                    val_running += float(loss_fn(Yhat, Y).item())
            val_loss = val_running / max(1, len(val_loader))
            history["val_loss"].append(val_loss)
            print(f"Epoch {epoch} | val loss: {val_loss:.4f}")

            # save best by validation loss
            if checkpoint_path and val_loss < best_val_loss:
                best_val_loss = val_loss
                os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
                torch.save({
                    "epoch": epoch,
                    "state_dict": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    "in_ch": in_ch,
                }, checkpoint_path_best)
                print(f"[ckpt] saved {checkpoint_path_best} (best val loss {best_val_loss:.4f})")

                np.save(history_path_best, history)
                print(f"{history_path_best} is saved for best model: {model_n}")

        # ---- scheduler step (only when scheduler_option != 1) ----
        if scheduler_option != 1 and scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss if val_loss is not None else avg_train)
            else:
                scheduler.step()

    # ---- final checkpoint & history ----
    os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
    torch.save({
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "in_ch": in_ch,
    }, checkpoint_path)

    final_val = history["val_loss"][-1] if history["val_loss"] else float("nan")
    print(f"[ckpt] saved {checkpoint_path} (val loss {final_val if not np.isnan(final_val) else float('nan'):.4f})")

    np.save(history_path, history)
    print(f"{history_path} is saved for training of the model: {model_n}")



def train_loop_vae(
    model, optimizer, loss_fn, train_loader, val_loader, device,
    epochs, use_amp, in_ch, model_n,
    scheduler_option,
    checkpoint_path, checkpoint_path_best, history_path, history_path_best,
    scheduler=None, lips=None
):

# ---- choose safer AMP dtype (bf16 if supported; else fp16) ----
#amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    best_val_loss = float('inf')
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0

        # --- optional Lipschitz LR update (only when scheduler_option == 1) ---
        if scheduler_option == 1 and lips is not None:
            lr_now, L_now, updated = lips.update(model, optimizer, epoch=epoch)
            if updated:
                print(f"[Lipschitz] epoch {epoch}: L={L_now:.3e} | LR={lr_now:.3e}")

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for X, Y in pbar:
            X = X.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # ---- forward (mixed precision if enabled) ----
            #with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
            #with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp and X.device.type=='cuda'):
            with torch.autocast( device_type=device_t.type,     dtype=torch.float16,     enabled=use_amp_runtime, ): 
                Yhat, aux = model(X)        

            # clamp predictions; then compute loss in full precision
            Yhat = Yhat.clamp(0, 1)
            with torch.amp.autocast('cuda', enabled=False):
                loss = loss_fn(Yhat, Y)
                if loss_fn2 is not None:
                    loss = loss + loss_fn2_coeff * loss_fn2(*aux)  # <--- use aux
                    

            # ---- early finite checks (helps catch overflow fast) ----
            if not torch.isfinite(loss):
                print("Non-finite loss detected.",
                      "Yhat[min,max]=", float(Yhat.min()), float(Yhat.max()),
                      "Y[min,max]=", float(Y.min()), float(Y.max()))
                raise RuntimeError("NaN/Inf loss under AMP")

            if use_amp:
                scaler.scale(loss).backward()
                # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # <- optional
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # <- optional
                optimizer.step()

            running += float(loss.item())
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}"
            )

        avg_train = running / max(1, len(train_loader))
        history["train_loss"].append(avg_train)
        print(f"Epoch {epoch} | train loss: {avg_train:.4f}")

        # ---------- validation ----------
        val_loss = None
        if val_loader is not None:
            model.eval()
            val_running = 0.0
            with torch.inference_mode():
                for X, Y in val_loader:
                    X = X.to(device, non_blocking=True)
                    Y = Y.to(device, non_blocking=True)
        
                    # forward under the same AMP setting as train
                    #with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                    #with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp and X.device.type=='cuda'):
                    with torch.autocast( device_type=device_t.type,     dtype=torch.float16,     enabled=use_amp_runtime, ): 
                        Yhat, aux = model(X)   # <--- keep aux
        
                    Yhat = Yhat.clamp(0, 1)
        
                    # compute validation loss in full precision
                    with torch.amp.autocast('cuda', enabled=False):
                        l = loss_fn(Yhat, Y)
                        if loss_fn2 is not None:
                            l = l + loss_fn2_coeff * loss_fn2(*aux)  # <--- use aux
        
                    # optional finite check for val too
                    if not torch.isfinite(l):
                        print("Non-finite val loss detected.",
                              "Yhat[min,max]=", float(Yhat.min()), float(Yhat.max()),
                              "Y[min,max]=", float(Y.min()), float(Y.max()))
                        l = torch.nan_to_num(l, nan=0.0, posinf=0.0, neginf=0.0)
        
                    val_running += float(l.item())
        
            val_loss = val_running / max(1, len(val_loader))
            history["val_loss"].append(val_loss)
            print(f"Epoch {epoch} | val loss: {val_loss:.4f}")


            # save best by validation loss
            if checkpoint_path and val_loss < best_val_loss:
                best_val_loss = val_loss
                os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
                torch.save({
                    "epoch": epoch,
                    "state_dict": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    #"in_ch": in_ch,
                }, checkpoint_path_best)
                print(f"[ckpt] saved {checkpoint_path_best} (best val loss {best_val_loss:.4f})")

                np.save(history_path_best, history)
                print(f"{history_path_best} is saved for best model: {model_n}")

        # ---- scheduler step (only when scheduler_option != 1) ----
        if scheduler_option != 1 and scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss if val_loss is not None else avg_train)
            else:
                scheduler.step()

    # ---- final checkpoint & history ----
    os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
    torch.save({
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        #"in_ch": in_ch,
    }, checkpoint_path)

    final_val = history["val_loss"][-1] if history["val_loss"] else float("nan")
    print(f"[ckpt] saved {checkpoint_path} (val loss {final_val if not np.isnan(final_val) else float('nan'):.4f})")

    np.save(history_path, history)
    print(f"{history_path} is saved for training of the model: {model_n}")


#====================================================================================================================
#====================================================================================================================

def bce_loss(recon_x, x, reduction='mean'):
    """
    Binary Cross-Entropy reconstruction loss.
    Expects inputs in [0,1]. Casts to fp32 internally for stability.
    """
    # force fp32 for numerical stability (and to avoid AMP dtype mismatches)
    recon_x = recon_x.float()
    x       = x.float()

    # Flatten to [B, N]
    recon_x = recon_x.view(recon_x.size(0), -1)
    x       = x.view(x.size(0), -1)

    # Clamp to avoid log(0)
    recon_x = recon_x.clamp(1e-7, 1 - 1e-7)

    # Elementwise BCE
    bce = F.binary_cross_entropy(recon_x, x, reduction='none')

    if reduction == 'sum':
        return bce.sum()
    elif reduction == 'mean':
        return bce.mean()
    elif reduction == 'batchmean':
        return bce.sum(dim=1).mean()
    else:
        raise ValueError(f"Invalid reduction '{reduction}'")



def kl_standard_normal(mu, logvar, reduction='batchmean'):
    # KL Divergence loss
    # KL[q(z|x)=N(mu, diag(exp(logvar))) || p(z)=N(0, I)]
    if reduction == 'batchmean':
        return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)
    elif reduction == 'sum':
        return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    elif reduction == 'mean':
        return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    else:
        raise ValueError(reduction)


# VAE Loss Function (remains the same for image data)
def vae_loss(recon_x, x, mu, log_var):
    # Reconstruction loss (Binary Cross-Entropy for images scaled to [0, 1])
    # Using reduction='sum' sums over all elements in the batch and all pixels.
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')

    # KL Divergence loss
    KLD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())

    return BCE + KLD



def kl_beta(epoch: int, kl_warmup_ep: int, kl_beta_final) -> float:
    if kl_warmup_ep <= 0:
        return kl_beta_final
    return min(1.0, float(epoch / kl_warmup_ep)) * kl_beta_final


#====================================================================================================================
#====================================================================================================================


def total_variation_isotropic(img: torch.Tensor,
                              eps: float = 1e-3,
                              reduction: str = "mean") -> torch.Tensor:
    """
    Isotropic total variation for a 4D tensor [B, C, H, W].

    L_tv = sum_{x,y} sqrt( (∂x)^2 + (∂y)^2 + eps^2 )

    Args:
        img: predicted image tensor [B, C, H, W]
        eps: small constant to avoid sqrt(0)
        reduction: "mean", "sum", or "none"

    Returns:
        scalar loss if reduction != "none", else per-pixel TV map
    """
    # horizontal and vertical finite differences
    dx = img[:, :, :, 1:] - img[:, :, :, :-1]   # [B,C,H,W-1]
    dy = img[:, :, 1:, :] - img[:, :, :-1, :]   # [B,C,H-1,W]

    # pad to same size as img
    dx = F.pad(dx, (0, 1, 0, 0))   # pad last dim (width)
    dy = F.pad(dy, (0, 0, 0, 1))   # pad height

    tv = torch.sqrt(dx * dx + dy * dy + eps * eps)  # [B,C,H,W]

    if reduction == "mean":
        return tv.mean()
    elif reduction == "sum":
        return tv.sum()
    else:
        return tv

#====================================================================================================================



In [24]:
#====================================================================================================================
#====================================================================================================================
#====================================================================================================================
import os
import torch
import torch.nn.functional as F
import torch._dynamo as dynamo
#import torch.cuda.amp as amp

class LossFnCust:
    def __init__(self, **kwargs):
        
        #================================================================
        # 1) stash all kwargs as attributes
        for name, value in kwargs.items():
            setattr(self, name, value)
            
        #=====
        # sensible defaults if a,b,c,d,e not provided
        self.a = getattr(self, "a", 1.0)
        self.b = getattr(self, "b", 0.0)
        self.c = getattr(self, "c", 0.0)
        self.d = getattr(self, "d", 0.0)
        self.e = getattr(self, "e", 0.0)

        # make sure device exists (string is fine)
        self.device = getattr(self, "device", "cpu")

        #================================================================
        # 2) build main loss
        self.loss_fn1 = MixedLoss(a=self.a, b=self.b, c=self.c, d=self.d, e=self.e)

        #================================================================
        # 3) optionally build a model (if model_class provided)
        self.model = getattr(self, "model", None)
        #self.model = None
        model_class = getattr(self, "model_class", None)
        if model_class is not None:
            # build the model from your provided knobs
            if model_class.__name__ == "VAE":
                self.model = create_model_vae(
                    input_channels=getattr(self, "input_channels", 3),
                    img_size=getattr(self, "patch_size", 128),
                    latent_dim=getattr(self, "latent_dim", 32),
                    hidden_dims=getattr(self, "hidden_dims", [32, 64, 128]),
                    device=self.device,
                    m_compile=getattr(self, "m_compile_train", False),
                )
            elif model_class.__name__ in ("FCN", "DenoiseFCN"):
                built = create_model_fcn(
                    width=getattr(self, "model_width", 64),
                    depth=getattr(self, "model_depth", 3),
                    device=self.device,
                    extra_edges=getattr(self, "extra_edges", ()),
                    m_compile=getattr(self, "m_compile_train", False),
                )
                # handle both (model, in_ch) or model returns
                self.model = built[0] if isinstance(built, tuple) else built
                
            #================================================================
            # 4) optionally load checkpoint
            ckpt_path = getattr(self, "ckpt_path", None)
            if self.model is not None and ckpt_path:
                state = self._load_model_state(ckpt_path, self.device)
                state = self._clean_state_dict(state)
                self.model.load_state_dict(state, strict=getattr(self, "strict", True))
                self.model = self.model.to(self.device)
                #self.model = self.model.to(self.device, dtype=torch.float32) #********************
                self.model.eval()

    # ---------------- helpers ----------------
    #================================================================
    @staticmethod
    def _load_model_state(path, device):
        assert os.path.isfile(path), f"Missing checkpoint: {path}"
        ckpt = torch.load(path, map_location=device, weights_only=False)
        return ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    #================================================================
    @staticmethod
    def _clean_state_dict(state_dict, prefix="_orig_mod."):
        if any(k.startswith(prefix) for k in state_dict.keys()):
            return {k.replace(prefix, "", 1): v for k, v in state_dict.items()}
        return state_dict
    #================================================================
#    def model_eval(self, X):
#        # return the prediction so caller can use it
#        return self.model(X).clamp(0, 1)

    #================================================================

    @staticmethod
    def kl_standard_normal(mu, logvar, reduction='batchmean'):
        # clamp to avoid exp overflow
        logvar = torch.clamp(logvar, min=-15.0, max=15.0)
        kl = -0.5 * (1 + logvar - mu.pow(2) - torch.exp(logvar))
        if reduction == 'batchmean':
            return torch.sum(kl) / mu.size(0)
        if reduction == 'sum':
            return torch.sum(kl)
        if reduction == 'mean':
            return torch.mean(kl)
        raise ValueError(reduction)

    #================================================================
    # ---------------- main API ----------------
    def __call__(self, **kwargs):
        # Expect Yhat, Y, X, mu, logvar; coeffs optional
        Yhat   = kwargs.get("Yhat")
        Y      = kwargs.get("Y")
        X      = kwargs.get("X", None)
        mu     = kwargs.get("mu", None)
        logvar = kwargs.get("logvar", None)
        prnt_model_dtype = kwargs.get("prnt_model_dtype", False)
        use_amp = kwargs.get("use_amp", False)

        loss1_coeff = kwargs.get("loss1_coeff", 1.0) # Loss Func
        loss2_coeff = kwargs.get("loss2_coeff", 0.0) # dlr model
        loss3_coeff = kwargs.get("loss3_coeff", 0.0) #KL

        #===================
        loss1 = self.loss_fn1(Yhat, Y) if Yhat is not None else 0.0
        loss2 = 0.0
        loss3 = 0.0


        #============================
        # pick a working device
        if torch.is_tensor(Yhat):
            dev = Yhat.device
        elif torch.is_tensor(mu):
            dev = mu.device
        else:
            dev = torch.device(self.device)
    
        # move inputs to the same device (only if tensors)
        if torch.is_tensor(Yhat):  Yhat  = Yhat.to(dev)
        if torch.is_tensor(Y):     Y     = Y.to(dev)
        if torch.is_tensor(mu):    mu    = mu.to(dev)
        if torch.is_tensor(logvar):logvar= logvar.to(dev)


        #===================


        if (self.model is not None) and (X is not None) and (loss2_coeff > 0.0):
            # optional debug printing
            if prnt_model_dtype:
                print('DLR Model --------')
                print("X dtype:", X.dtype, "device:", X.device)
                if Y is not None:
                    print("Y dtype:", Y.dtype, "device:", Y.device)
                first_param = next(self.model.parameters())
                print("teacher model param dtype:", first_param.dtype,
                      "device:", first_param.device)
        
            # ---- prepare copies for teacher branch ----
            X_in = X.to(self.device)
            Y_in = Y.to(self.device) if Y is not None else None
        
            # infer teacher spatial size
            target_h = getattr(self.model, "img_h", None)
            target_w = getattr(self.model, "img_w", None)
        
            if target_h is not None and target_w is not None:
                if X_in.shape[-2:] != (target_h, target_w):
                    X_in = F.interpolate(
                        X_in, size=(target_h, target_w),
                        mode="bilinear", align_corners=False
                    )
                    if Y_in is not None:
                        Y_in = F.interpolate(
                            Y_in, size=(target_h, target_w),
                            mode="bilinear", align_corners=False
                        )
        
            
        
            # ---- teacher forward & loss2 ----
            self.model.eval()
            with torch.inference_mode():
                X_in = X_in.to(device, non_blocking=True)
                Y_in = Y_in.to(device, non_blocking=True)

                beta_t = getattr(self.model, "kl_beta_final", None)
                if beta_t is not None:
                    Yhat2, [mu2, logvar2] = self.model(X_in)
                    loss2 = self.loss_fn1(Yhat2, Y_in) + beta_t*self.kl_standard_normal(mu2, logvar2)
                    
                else:
                    Yhat2 = self.model(X_in)
                    loss2 = self.loss_fn1(Yhat2, Y_in)
                        
                
                

        #===================
        if (mu is not None) and (logvar is not None) and (loss3_coeff>0.0):
            loss3 = self.kl_standard_normal(mu, logvar)

        #loss= loss1_coeff * loss1 + loss2_coeff * loss2 + loss3_coeff * loss3
        #print(f'loss:{loss.item()}')
        #return loss
        return loss1_coeff * loss1 + loss2_coeff * loss2 + loss3_coeff * loss3
#====================================================================================================================
#====================================================================================================================
#====================================================================================================================


## Version 3 Functions

In [25]:
def build_final_loss_fn3(best_params, loss_fn_coeff):
    """
    Rebuild LossFnCust exactly like objective_vae3:

      - If a/b/c/d/e were fixed in loss_fn_coeff, keep those fixed.
      - Otherwise, pull the tuned values from best_params.
      - Use the same dlr_model that was passed in loss_fn_coeff['dlr_model'].
    """
    # a
    if "a" in loss_fn_coeff and loss_fn_coeff["a"] is not None:
        a = float(loss_fn_coeff["a"])
    else:
        a = float(best_params["a"])

    # b (you kept b=0.0 in objective_vae3, but support tuning if you ever change it)
    if "b" in loss_fn_coeff and loss_fn_coeff["b"] is not None:
        b = float(loss_fn_coeff["b"])
    else:
        b = float(best_params.get("b", 0.0))

    # c
    if "c" in loss_fn_coeff and loss_fn_coeff["c"] is not None:
        c = float(loss_fn_coeff["c"])
    else:
        c = float(best_params["c"])  # only present if tuned

    # d
    if "d" in loss_fn_coeff and loss_fn_coeff["d"] is not None:
        d = float(loss_fn_coeff["d"])
    else:
        d = float(best_params["d"])

    # e
    if "e" in loss_fn_coeff and loss_fn_coeff["e"] is not None:
        e = float(loss_fn_coeff["e"])
    else:
        e = float(best_params["e"])

    dlr_model = loss_fn_coeff.get("dlr_model", None)

    return LossFnCust(
        a=a,
        b=b,
        c=c,
        d=d,
        e=e,
        model=dlr_model,
    )


### Train function ______

### VAE Train

In [26]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from tqdm import tqdm

def _is_dist_ready() -> bool:
    return dist.is_available() and dist.is_initialized()

def _ddp_rank_world():
    if _is_dist_ready():
        return dist.get_rank(), dist.get_world_size()
    return 0, 1

def _reduce_sum_count(sum_x: float, count: int, device: torch.device):
    """Reduce (sum_x, count) across ranks."""
    if not _is_dist_ready():
        return float(sum_x), int(count)
    t = torch.tensor([float(sum_x), float(count)], device=device, dtype=torch.float64)
    dist.all_reduce(t, op=dist.ReduceOp.SUM)
    return float(t[0].item()), int(round(float(t[1].item())))

def _model_state_dict(m):
    return m.module.state_dict() if hasattr(m, "module") else m.state_dict()


@timer_with_elapsed
def train_dlr_vae3(
    best_params: dict,
    loss_fn,
    epochs: int,
    data_loader_option: int,
    save_dir: str = "saved_models/charbonnier_best",
    pbar_interval: int = 50,
    sigma_range=(0.0, 80.0),
    blur_prob=0.5,
    num_workers=0,
    extra_edges=(),
    edge_from="clean",
    blur_kernels=(3, 5, 7, 9),
    blur_sigma_rng=(0.5, 2.0),
    prnt_model_dtype: bool = False,
    print_new_best_model: bool = False,
    print_update: bool = False,
    # loader controls
    noise_type: str = "gaussian",
    noise_params: dict | None = None,
    # parallel controls
    active_dp: bool = False,
    gpu_device_ids: list[int] | None = None,
    active_ddp: bool = False,
    # AMP controls
    use_amp: bool = False,
    device: str = "cuda",   # "cpu" or "cuda"
    m_compile_train: bool = False,
):
    os.makedirs(save_dir, exist_ok=True)

    # -----------------------------
    # Unpack hyperparams
    # -----------------------------
    batch_size       = int(best_params["batch_size"])
    patch_size       = int(best_params["patch_size"])
    base_lr          = float(best_params["base_lr"])
    l2_weight_decay  = float(best_params["l2_weight_decay"])
    optimizer_option = int(best_params["optimizer_option"])
    scheduler_option = int(best_params["scheduler_option"])
    hidden_dims      = list(best_params["hidden_dims"])
    kl_beta_final    = float(best_params["kl_beta_final"])
    kl_warmup_ep     = int(best_params["kl_warmup_epochs"])
    loss1_coeff      = float(best_params.get("loss1_coeff", 1.0))
    loss2_coeff      = float(best_params.get("loss2_coeff", 0.0))
    latent_dim       = int(hidden_dims[-1])

    # -----------------------------
    # Build model on CPU first
    # -----------------------------
    input_channels = 3 + (len(extra_edges) if extra_edges else 0)

    model = create_model_vae(
        input_channels=input_channels,
        img_size=patch_size,
        latent_dim=latent_dim,
        hidden_dims=hidden_dims,
        device="cpu",  # important: avoid cuda:0 before DDP wrapping
        m_compile=m_compile_train,
    )

    # -----------------------------
    # Decide runtime mode & wrap BEFORE loaders (DDP needs sampler)
    # -----------------------------
    mode = "ddp" if active_ddp else ("dp" if active_dp else ("cpu" if device == "cpu" else "cuda"))

    if mode == "dp":
        if not torch.cuda.is_available():
            raise RuntimeError("active_dp=True but CUDA is not available.")
        if gpu_device_ids is None:
            gpu_device_ids = list(range(torch.cuda.device_count()))
        if len(gpu_device_ids) == 0:
            raise RuntimeError("active_dp=True but no CUDA devices found.")

        device_t = torch.device(f"cuda:{gpu_device_ids[0]}")
        model = model.to(device_t)
        if len(gpu_device_ids) > 1:
            model = nn.DataParallel(model, device_ids=gpu_device_ids)

        rank, world = 0, 1
        is_ddp = False

    else:
        # This should init process group + move to correct device + wrap for DDP if needed
        device_t, model, is_ddp, rank, world = setup_device_and_model(model, mode)

    # AMP must be decided AFTER device_t is known
    use_amp_runtime = bool(use_amp) and (device_t.type == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp_runtime)

    # -----------------------------
    # Now build loaders (DDP-aware)
    # -----------------------------
    train_loader, val_loader = data_loader(
        data_loader_option=data_loader_option,
        batch_size=batch_size,
        patch_size=patch_size,
        sigma_range=sigma_range,
        blur_prob=blur_prob,
        num_workers=num_workers,
        extra_edges=extra_edges,
        edge_from=edge_from,
        canny_low=200,
        canny_high=255,
        sobel_ksize=9,
        blur_kernels=blur_kernels,
        blur_sigma_rng=blur_sigma_rng,
        noise_type=noise_type,
        noise_params=(noise_params or {}),
        use_ddp=is_ddp,
        ddp_rank=rank,
        ddp_world_size=world,
    )

    # -----------------------------
    # Optimizer / scheduler
    # -----------------------------
    if scheduler_option == 1:
        base_lr, optimizer, lips = create_optimizer(
            model,
            optimizer_option=optimizer_option,
            scheduler_option=1,
            base_lr=base_lr,
            weight_decay=l2_weight_decay,
        )
        scheduler = None
    else:
        base_lr, optimizer, scheduler = create_optimizer(
            model,
            optimizer_option=optimizer_option,
            scheduler_option=scheduler_option,
            base_lr=base_lr,
            weight_decay=l2_weight_decay,
        )
        lips = None

    # -----------------------------
    # History (rank0 only)
    # -----------------------------
    history = {"epoch": [], "train_loss": [], "val_loss": [], "metric": [], "lr": []}
    best_metric = float("inf")
    best_ckpt_path = os.path.join(save_dir, "best.pt")

    if rank == 0:
        print("\n=== Final training with best params ===", flush=True)
        for k, v in best_params.items():
            print(f"  {k}: {v}", flush=True)
        print(f"  mode: {mode} | is_ddp={is_ddp} rank={rank}/{world} device={device_t}", flush=True)
        print("=======================================\n", flush=True)

    # -----------------------------
    # Training loop
    # -----------------------------
    for epoch in range(1, epochs + 1):
        model.train()

        if is_ddp and hasattr(train_loader, "sampler") and isinstance(
            train_loader.sampler, torch.utils.data.distributed.DistributedSampler
        ):
            train_loader.sampler.set_epoch(epoch)

        if lips is not None:
            lips.update(model, optimizer, epoch=epoch)

        loss3_coeff = kl_beta(epoch, kl_warmup_ep, kl_beta_final)

        show_epoch_info = (epoch % pbar_interval == 0) or (epoch == epochs)
        pbar_disable = (rank != 0) or (not show_epoch_info)

        running_sum = 0.0
        running_count = 0

        pbar = tqdm(train_loader, desc=f"[FINAL VAE] Epoch {epoch}/{epochs}", disable=pbar_disable)
        for i, (X, Y) in enumerate(pbar):
            X = X.to(device_t, non_blocking=True)
            Y = Y.to(device_t, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if i == 0 and prnt_model_dtype and rank == 0:
                p0 = next(model.parameters())
                print("Final Training --------", flush=True)
                print("X dtype:", X.dtype, "device:", X.device, flush=True)
                print("Y dtype:", Y.dtype, "device:", Y.device, flush=True)
                print("model param dtype:", p0.dtype, "device:", p0.device, flush=True)

            if use_amp_runtime:
                with torch.autocast(device_type=device_t.type, dtype=torch.float16, enabled=True):
                    Yhat, (mu, logvar) = model(X)
                    loss = loss_fn(
                        Yhat=Yhat, Y=Y, X=X,
                        loss1_coeff=loss1_coeff,
                        loss2_coeff=loss2_coeff,
                        loss3_coeff=loss3_coeff,
                        mu=mu, logvar=logvar,
                        prnt_model_dtype=prnt_model_dtype,
                    )
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                Yhat, (mu, logvar) = model(X)
                loss = loss_fn(
                    Yhat=Yhat, Y=Y, X=X,
                    loss1_coeff=loss1_coeff,
                    loss2_coeff=loss2_coeff,
                    loss3_coeff=loss3_coeff,
                    mu=mu, logvar=logvar,
                    prnt_model_dtype=prnt_model_dtype,
                )
                loss.backward()
                optimizer.step()

            bs = int(Y.shape[0])
            running_sum += float(loss.item()) * bs
            running_count += bs

            if not pbar_disable:
                pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        train_sum_g, train_cnt_g = _reduce_sum_count(running_sum, running_count, device=device_t)
        train_loss = train_sum_g / max(1, train_cnt_g)

        # -----------------------------
        # Validation
        # -----------------------------
        if val_loader is not None:
            model.eval()
            val_sum = 0.0
            val_cnt = 0

            with torch.inference_mode():
                for j, (X, Y) in enumerate(val_loader):
                    X = X.to(device_t, non_blocking=True)
                    Y = Y.to(device_t, non_blocking=True)

                    Yhat, (mu, logvar) = model(X)
                    vloss = loss_fn(
                        Yhat=Yhat, Y=Y, X=X,
                        loss1_coeff=loss1_coeff,
                        loss2_coeff=loss2_coeff,
                        loss3_coeff=loss3_coeff,
                        mu=mu, logvar=logvar,
                        prnt_model_dtype=prnt_model_dtype,
                    )

                    bs = int(Y.shape[0])
                    val_sum += float(vloss.item()) * bs
                    val_cnt += bs

            val_sum_g, val_cnt_g = _reduce_sum_count(val_sum, val_cnt, device=device_t)
            val_loss = val_sum_g / max(1, val_cnt_g)
            metric = val_loss
        else:
            val_loss = None
            metric = train_loss

        # -----------------------------
        # Scheduler step
        # -----------------------------
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(metric)
            else:
                scheduler.step()

        current_lr = float(optimizer.param_groups[0]["lr"])

        # -----------------------------
        # Log + checkpoint (rank0)
        # -----------------------------
        if rank == 0:
            if print_update:
                msg = f"Epoch {epoch}/{epochs} - train_loss={train_loss:.6f}"
                if val_loss is not None:
                    msg += f" val_loss={val_loss:.6f}"
                msg += f" lr={current_lr:.3e}"
                print(msg, flush=True)

            history["epoch"].append(epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["metric"].append(metric)
            history["lr"].append(current_lr)

            if metric < best_metric:
                best_metric = float(metric)
                ckpt = {
                    "epoch": epoch,
                    "model_state_dict": _model_state_dict(model),  # unwrapped weights
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                    "best_metric": best_metric,
                    "params": best_params,
                    "history": history,
                    "input_channels": input_channels,
                    "extra_edges": tuple(extra_edges),
                    "edge_from": edge_from,
                    "noise_type": noise_type,
                    "noise_params": (noise_params or {}),
                    "mode": mode,
                }
                torch.save(ckpt, best_ckpt_path)
                if print_new_best_model:
                    print(f"New best metric={best_metric:.6f} saved to {best_ckpt_path}", flush=True)

        if _is_dist_ready():
            dist.barrier()

    # -----------------------------
    # Save final (rank0)
    # -----------------------------
    if rank == 0:
        print("\n=== Final training complete ===", flush=True)
        print(f"Best metric: {best_metric:.6f}", flush=True)
        print(f"Best checkpoint saved at: {best_ckpt_path}", flush=True)

        final_ckpt = {
            "epoch": epochs,
            "model_state_dict": _model_state_dict(model),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "best_metric": best_metric,
            "params": best_params,
            "history": history,
            "input_channels": input_channels,
            "extra_edges": tuple(extra_edges),
            "edge_from": edge_from,
            "noise_type": noise_type,
            "noise_params": (noise_params or {}),
            "mode": mode,
        }
        final_ckpt_path = os.path.join(save_dir, "final.pt")
        torch.save(final_ckpt, final_ckpt_path)
        print(f"Final model saved to {final_ckpt_path}", flush=True)

        history_path = os.path.join(save_dir, "history.pt")
        torch.save(history, history_path)
        print(f"Training history saved to {history_path}", flush=True)

    return best_ckpt_path if rank == 0 else None


### FCN Train

In [27]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from tqdm import tqdm

# Assumes these exist in your codebase:
# - data_loader(...)
# - setup_device_and_model(model, mode)  # should init DDP when mode=="ddp"
# - create_model_fcn(...)
# - create_optimizer(...)
# - total_variation_isotropic(...)
# - timer_with_elapsed decorator




# ============================================
# Final training with best Optuna parameters
# ============================================
@timer_with_elapsed
def train_dlr_fcn3(
    best_params: dict,
    loss_fn,
    epochs: int,
    data_loader_option: int,
    save_dir: str = "saved_models/charbonnier_fcn_best",
    pbar_interval: int = 50,
    sigma_range=(0.0, 80.0),
    blur_prob=0.5,
    num_workers=0,
    extra_edges=(),
    edge_from="clean",
    blur_kernels=(3, 5, 7, 9),
    blur_sigma_rng=(0.5, 2.0),
    tv_coeff: float | None = None,
    prnt_model_dtype: bool = False,
    print_new_best_model: bool = False,
    print_update: bool = False,
    # loader controls
    noise_type: str = "gaussian",
    noise_params: dict | None = None,
    # parallel controls
    active_dp: bool = False,
    gpu_device_ids: list[int] | None = None,
    active_ddp: bool = False,
    # AMP controls
    use_amp: bool = False,
    device: str = "cuda",              # "cpu" or "cuda"
    m_compile_train: bool = False,
):
    """
    Re-train FCN once using best Optuna params and save:
      - best model checkpoint (by val loss or train loss fallback)
      - final model checkpoint (last epoch)
    Also saves training history.

    Key points:
      - DDP init MUST happen before building data loaders (so DistributedSampler is used).
      - Only rank 0 saves checkpoints/prints.
      - Saves clean weights using model.module.state_dict() when wrapped (DP/DDP).
    """

    os.makedirs(save_dir, exist_ok=True)

    # -----------------------------
    # Unpack hyperparams from Optuna
    # -----------------------------
    batch_size       = int(best_params["batch_size"])
    patch_size       = int(best_params["patch_size"])
    base_lr          = float(best_params["base_lr"])
    l2_weight_decay  = float(best_params["l2_weight_decay"])
    optimizer_option = int(best_params["optimizer_option"])
    scheduler_option = int(best_params["scheduler_option"])
    model_width      = int(best_params["model_width"])
    model_depth      = int(best_params["model_depth"])

    loss1_coeff = float(best_params.get("loss1_coeff", 1.0))
    loss2_coeff = float(best_params.get("loss2_coeff", 0.0))

    # TV coefficient: tuned > provided default > 0.0
    if "tv_coeff" in best_params:
        tv_coeff_final = float(best_params["tv_coeff"])
    else:
        tv_coeff_final = float(0.0 if tv_coeff is None else tv_coeff)

    # -----------------------------
    # Build model first (CPU), then decide parallel mode and wrap
    # -----------------------------
    model, in_ch = create_model_fcn(
        width=model_width,
        depth=model_depth,
        extra_edges=extra_edges,
        m_compile=m_compile_train,
    )

    mode = "ddp" if active_ddp else ("dp" if active_dp else ("cpu" if device == "cpu" else "cuda"))

    if mode == "dp":
        if not torch.cuda.is_available():
            raise RuntimeError("active_dp=True but CUDA is not available.")
        if gpu_device_ids is None:
            gpu_device_ids = list(range(torch.cuda.device_count()))
        if len(gpu_device_ids) == 0:
            raise RuntimeError("active_dp=True but no CUDA devices found.")

        device_t = torch.device(f"cuda:{gpu_device_ids[0]}")
        model = model.to(device_t)
        if len(gpu_device_ids) > 1:
            model = nn.DataParallel(model, device_ids=gpu_device_ids)

        is_ddp, rank, world = False, 0, 1

    else:
        # setup_device_and_model should:
        # - move model to correct device
        # - if mode=="ddp": init dist + wrap in DistributedDataParallel
        device_t, model, is_ddp, rank, world = setup_device_and_model(model, mode)

    # -----------------------------
    # AMP must be decided AFTER device_t is known
    # -----------------------------
    use_amp_runtime = bool(use_amp) and (device_t.type == "cuda")
    scaler = torch.amp.GradScaler('cuda',enabled=use_amp_runtime)

    # -----------------------------
    # Print params only on rank 0
    # -----------------------------
    if rank == 0:
        print("\n=== Final FCN training with best params ===", flush=True)
        for k, v in best_params.items():
            print(f"  {k}: {v}", flush=True)
        print(f"  tv_coeff_final: {tv_coeff_final}", flush=True)
        print(f"  mode: {mode} | is_ddp={is_ddp} rank={rank}/{world} device={device_t}", flush=True)
        print("===========================================\n", flush=True)

    # -----------------------------
    # Data loaders (NOW DDP-aware)
    # -----------------------------
    train_loader, val_loader = data_loader(
        data_loader_option=data_loader_option,
        batch_size=batch_size,
        patch_size=patch_size,
        sigma_range=sigma_range,
        blur_prob=blur_prob,
        num_workers=num_workers,
        extra_edges=extra_edges,
        edge_from=edge_from,
        canny_low=200,
        canny_high=255,
        sobel_ksize=9,
        blur_kernels=blur_kernels,
        blur_sigma_rng=blur_sigma_rng,
        noise_type=noise_type,
        noise_params=(noise_params or {}),
        use_ddp=is_ddp,
        ddp_rank=rank,
        ddp_world_size=world,
    )

    # -----------------------------
    # Optimizer / scheduler
    # -----------------------------
    if scheduler_option == 1:
        base_lr, optimizer, lips = create_optimizer(
            model,
            optimizer_option=optimizer_option,
            scheduler_option=1,
            base_lr=base_lr,
            weight_decay=l2_weight_decay,
        )
        scheduler = None
    else:
        base_lr, optimizer, scheduler = create_optimizer(
            model,
            optimizer_option=optimizer_option,
            scheduler_option=scheduler_option,
            base_lr=base_lr,
            weight_decay=l2_weight_decay,
        )
        lips = None

    # -----------------------------
    # History (rank 0 only)
    # -----------------------------
    history = {"epoch": [], "train_loss": [], "val_loss": [], "metric": [], "lr": []}

    best_metric = float("inf")
    best_ckpt_path = os.path.join(save_dir, "best.pt")

    # Helper: save clean weights regardless of DP/DDP
    def _model_state_dict(m):
        return m.module.state_dict() if hasattr(m, "module") else m.state_dict()

    # -----------------------------
    # Training loop
    # -----------------------------
    for epoch in range(1, epochs + 1):
        model.train()

        # DDP shuffle correctness
        if is_ddp and hasattr(train_loader, "sampler") and isinstance(
            train_loader.sampler, torch.utils.data.distributed.DistributedSampler
        ):
            train_loader.sampler.set_epoch(epoch)

        # Lipschitz scheduler update if used
        if lips is not None:
            lips.update(model, optimizer, epoch=epoch)

        show_epoch_info = (epoch % pbar_interval == 0) or (epoch == epochs)
        pbar_disable = (rank != 0) or (not show_epoch_info)

        train_sum = 0.0
        train_cnt = 0

        pbar = tqdm(train_loader, desc=f"[FINAL FCN] Epoch {epoch}/{epochs}", disable=pbar_disable)
        for i, (X, Y) in enumerate(pbar):
            X = X.to(device_t, non_blocking=True)
            Y = Y.to(device_t, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if i == 0 and prnt_model_dtype and rank == 0:
                p0 = next(model.parameters())
                print("Final FCN Training --------", flush=True)
                print("X dtype:", X.dtype, "device:", X.device, flush=True)
                print("Y dtype:", Y.dtype, "device:", Y.device, flush=True)
                print("model param dtype:", p0.dtype, "device:", p0.device, flush=True)

            if use_amp_runtime:
                with torch.autocast(device_type=device_t.type, dtype=torch.float16, enabled=True):
                    Yhat = model(X)
                    base_loss = loss_fn(
                        Yhat=Yhat,
                        Y=Y,
                        X=X,
                        loss1_coeff=loss1_coeff,
                        loss2_coeff=loss2_coeff,
                        prnt_model_dtype=prnt_model_dtype,
                    )
                    tv_loss = total_variation_isotropic(Yhat)
                    loss = base_loss + tv_coeff_final * tv_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                Yhat = model(X)
                base_loss = loss_fn(
                    Yhat=Yhat,
                    Y=Y,
                    X=X,
                    loss1_coeff=loss1_coeff,
                    loss2_coeff=loss2_coeff,
                    prnt_model_dtype=prnt_model_dtype,
                )
                tv_loss = total_variation_isotropic(Yhat)
                loss = base_loss + tv_coeff_final * tv_loss

                loss.backward()
                optimizer.step()

            bs = int(Y.shape[0])
            train_sum += float(loss.item()) * bs
            train_cnt += bs

            if not pbar_disable:
                pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        # Reduce train loss across ranks
        train_sum_g, train_cnt_g = _reduce_sum_count(train_sum, train_cnt, device=device_t)
        train_loss = train_sum_g / max(1, train_cnt_g)

        # -----------------------------
        # Validation
        # -----------------------------
        if val_loader is not None:
            model.eval()
            val_sum = 0.0
            val_cnt = 0

            with torch.inference_mode():
                for j, (X, Y) in enumerate(val_loader):
                    X = X.to(device_t, non_blocking=True)
                    Y = Y.to(device_t, non_blocking=True)

                    if j == 0 and prnt_model_dtype and rank == 0:
                        print("Final FCN Validation --------", flush=True)
                        print("X dtype:", X.dtype, "device:", X.device, flush=True)
                        print("Y dtype:", Y.dtype, "device:", Y.device, flush=True)

                    Yhat = model(X).clamp(0, 1)

                    base_vloss = loss_fn(
                        Yhat=Yhat,
                        Y=Y,
                        X=X,
                        loss1_coeff=loss1_coeff,
                        loss2_coeff=loss2_coeff,
                        prnt_model_dtype=prnt_model_dtype,
                    )
                    tv_vloss = total_variation_isotropic(Yhat)
                    vloss = base_vloss + tv_coeff_final * tv_vloss

                    bs = int(Y.shape[0])
                    val_sum += float(vloss.item()) * bs
                    val_cnt += bs

            # Reduce val loss across ranks
            val_sum_g, val_cnt_g = _reduce_sum_count(val_sum, val_cnt, device=device_t)
            val_loss = val_sum_g / max(1, val_cnt_g)
            metric = val_loss
        else:
            val_loss = None
            metric = train_loss

        # -----------------------------
        # Scheduler step
        # -----------------------------
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(metric)
            else:
                scheduler.step()

        current_lr = float(optimizer.param_groups[0]["lr"])

        # -----------------------------
        # Record history + save (rank 0 only)
        # -----------------------------
        if rank == 0:
            if print_update:
                msg = f"Epoch {epoch}/{epochs} - train_loss={train_loss:.6f}"
                if val_loss is not None:
                    msg += f" val_loss={val_loss:.6f}"
                msg += f" lr={current_lr:.3e}"
                print(msg, flush=True)

            history["epoch"].append(epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["metric"].append(metric)
            history["lr"].append(current_lr)

            if metric < best_metric:
                best_metric = float(metric)
                ckpt = {
                    "epoch": epoch,
                    "model_state_dict": _model_state_dict(model),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                    "best_metric": best_metric,
                    "params": best_params,
                    "history": history,
                    "in_ch": int(in_ch),
                    "extra_edges": tuple(extra_edges),
                    "edge_from": edge_from,
                    "noise_type": noise_type,
                    "noise_params": (noise_params or {}),
                    "tv_coeff_final": float(tv_coeff_final),
                    "mode": mode,
                }
                torch.save(ckpt, best_ckpt_path)
                if print_new_best_model:
                    print(f"New best metric={best_metric:.6f} saved to {best_ckpt_path}", flush=True)

        # Barrier only if dist is actually initialized
        if _is_dist_ready():
            dist.barrier()

    # -----------------------------
    # Save final + history (rank 0)
    # -----------------------------
    if rank == 0:
        print("\n=== Final FCN training complete ===", flush=True)
        print(f"Best metric: {best_metric:.6f}", flush=True)
        print(f"Best checkpoint saved at: {best_ckpt_path}", flush=True)

        final_ckpt = {
            "epoch": epochs,
            "model_state_dict": _model_state_dict(model),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "best_metric": best_metric,
            "params": best_params,
            "history": history,
            "in_ch": int(in_ch),
            "extra_edges": tuple(extra_edges),
            "edge_from": edge_from,
            "noise_type": noise_type,
            "noise_params": (noise_params or {}),
            "tv_coeff_final": float(tv_coeff_final),
            "mode": mode,
        }
        final_ckpt_path = os.path.join(save_dir, "final.pt")
        torch.save(final_ckpt, final_ckpt_path)
        print(f"Final model saved to {final_ckpt_path}", flush=True)

        history_path = os.path.join(save_dir, "history.pt")
        torch.save(history, history_path)
        print(f"Training history saved to {history_path}", flush=True)

    return best_ckpt_path if rank == 0 else None


### Load Function ______

### VAE

In [28]:
def load_trained_vae_dlr3(ckpt_path: str, m_compile: bool = False, input_channels: int = 3):
    assert os.path.isfile(ckpt_path), f"Checkpoint not found: {ckpt_path}"

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    best_params   = ckpt["params"]
    patch_size    = int(best_params["patch_size"])
    hidden_dims   = list(best_params["hidden_dims"])
    kl_beta_final = float(best_params.get("kl_beta_final", 0.0))
    latent_dim    = int(hidden_dims[-1])

    model = create_model_vae(
        input_channels=input_channels,
        img_size=patch_size,
        latent_dim=latent_dim,
        hidden_dims=hidden_dims,
        m_compile=False,
    )

    model.img_h = patch_size
    model.img_w = patch_size
    model.kl_beta_final = kl_beta_final

    state = ckpt["model_state_dict"]
    if any(k.startswith("_orig_mod.") for k in state.keys()):
        state = {k.replace("_orig_mod.", "", 1): v for k, v in state.items()}

    model.load_state_dict(state, strict=True)
    model.eval()

    # I recommend leaving this False in your workflow.
    if m_compile:
        try:
            model = torch.compile(model, backend="eager")
        except Exception:
            pass

    return model, best_params, ckpt


### FCN

In [29]:
def load_trained_fcn_dlr3(ckpt_path: str, extra_edges=(), m_compile: bool = False):
    assert os.path.isfile(ckpt_path), f"Checkpoint not found: {ckpt_path}"

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    best_params = ckpt["params"]
    model_width = int(best_params["model_width"])
    model_depth = int(best_params["model_depth"])

    model, in_ch = create_model_fcn(
        width=model_width,
        depth=model_depth,
        extra_edges=extra_edges,
        m_compile=False,
    )

    state = ckpt["model_state_dict"]
    if any(k.startswith("_orig_mod.") for k in state.keys()):
        state = {k.replace("_orig_mod.", "", 1): v for k, v in state.items()}

    model.load_state_dict(state, strict=True)
    model.eval()

    if m_compile:
        try:
            model = torch.compile(model, backend="eager")
        except Exception:
            pass

    return model, in_ch, best_params, ckpt


# Optuna Objectives ============================================================================

In [30]:

def kl_beta(epoch: int, kl_warmup_ep: int, kl_beta_final) -> float:
    if kl_warmup_ep <= 0:
        return kl_beta_final
    return min(1.0, float(epoch / kl_warmup_ep)) * kl_beta_final

### VAE

#### V3

In [31]:
import gc
import torch
from tqdm import tqdm
import optuna

# ============================================================
# VAE with DLR (Optuna objective: SINGLE PROCESS ONLY)
# - CPU or single-GPU only (NO DP/DDP here)
# - Handles optional teacher model (dlr_model) device placement
# - Avoids CPU-compile-then-move issues by compiling AFTER .to(device)
# ============================================================
@timer
def objective_vae3(
    trial: optuna.trial.Trial,
    loss_fn_coeff,
    pbar_interval: int = pbar_interval,
    epochs: int = epochs,
    data_loader_option: int = 0,
    prnt_model_dtype: bool = False,
    data_keep_in_ram: bool = data_keep_in_ram,
    **kargs
):
    print(f"\n=== Starting Trial {trial.number} ===")

    # -----------------------------
    # Normalize coeff dict
    # -----------------------------
    if loss_fn_coeff is None:
        loss_fn_coeff = {}
    a_fixed = loss_fn_coeff.get("a", None)
    b_fixed = loss_fn_coeff.get("b", None)
    c_fixed = loss_fn_coeff.get("c", None)
    d_fixed = loss_fn_coeff.get("d", None)
    e_fixed = loss_fn_coeff.get("e", None)
    dlr_model = loss_fn_coeff.get("dlr_model", None)

    # -----------------------------
    # Tune/assign loss coefficients (a..e)
    # -----------------------------
    a = float(a_fixed) if a_fixed is not None else trial.suggest_float("a", 1.0, 2.0, log=True)
    b = float(b_fixed) if b_fixed is not None else 0.0
    c = float(c_fixed) if c_fixed is not None else trial.suggest_float("c", 1e-4, 1e-3, log=True)
    d = float(d_fixed) if d_fixed is not None else trial.suggest_float("d", 1.0, 2.0, log=True)
    e = float(e_fixed) if e_fixed is not None else trial.suggest_float("e", 1.0, 2.0, log=True)

    # -----------------------------
    # Build loss function (teacher may be inside)
    # -----------------------------
    loss_fn = LossFnCust(a=a, b=b, c=c, d=d, e=e, model=dlr_model)

    # -----------------------------
    # Fixed knobs
    # -----------------------------
    sigma_range = (0.0, 100.0)
    blur_prob = 0.0
    num_workers = int(kargs.get("num_workers", 0))
    extra_edges = ()
    edge_from = "clean"
    blur_kernels = (3, 5)
    blur_sigma_rng = (0.5, 2.0)

    # Dataloader noise controls
    noise_type = str(kargs.get("noise_type", "gaussian")).lower()
    noise_params = dict(kargs.get("noise_params", {}))

    # Runtime device selection
    force_device = str(kargs.get("force_device", "auto")).lower()  # "auto" | "cpu" | "cuda"
    if force_device == "cpu":
        device_t = torch.device("cpu")
    elif force_device in ("cuda", "gpu"):
        device_t = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    else:
        device_t = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # AMP decision must follow device_t
    use_amp_req = bool(kargs.get("use_amp", True))
    use_amp_runtime = bool(use_amp_req) and (device_t.type == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp_runtime)

    # Compile toggle (define locally so no NameError)
    m_compile_train = bool(kargs.get("m_compile_train", False))

    # Safety: init locals used in exception handler
    train_loss = float("nan")
    val_loss = None

    try:
        # -----------------------------
        # Suggested knobs
        # -----------------------------
        batch_size = trial.suggest_categorical("batch_size", [8, 16, 24, 32])
        patch_size = trial.suggest_categorical("patch_size", [64, 128, 256])
        base_lr = trial.suggest_float("base_lr", 1e-4, 5e-3, log=True)
        l2_weight_decay = trial.suggest_float("l2_weight_decay", 1e-6, 1e-2, log=True)
        optimizer_option = trial.suggest_categorical("optimizer_option", [0, 1, 2])
        scheduler_option = trial.suggest_categorical("scheduler_option", [0, 1, 4])

        hidden_dims = trial.suggest_categorical(
            "hidden_dims",
            [
                [16, 32, 64, 128],
                [16, 32, 64, 128, 256],
                [32, 64, 128, 256],
                [64, 128, 256],
            ],
        )
        latent_dim = int(hidden_dims[-1])

        kl_beta_final = trial.suggest_float("kl_beta_final", 1e-4, 1e-1, log=True)
        kl_warmup_ep = trial.suggest_int("kl_warmup_epochs", 0, 10)

        # Loss mixing coefficients
        loss1_coeff = trial.suggest_float("loss1_coeff", 1.0, 2.0, log=True)
        loss2_coeff = trial.suggest_float("loss2_coeff", 0.0, 1.0) if dlr_model is not None else 0.0

        # -----------------------------
        # Datasets & loaders
        # -----------------------------
        train_loader, val_loader = data_loader(
            data_loader_option=data_loader_option,
            batch_size=batch_size,
            patch_size=patch_size,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            num_workers=num_workers,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=200,
            canny_high=255,
            sobel_ksize=9,
            blur_kernels=blur_kernels,
            blur_sigma_rng=blur_sigma_rng,
            keep_in_ram_train=data_keep_in_ram,
            keep_in_ram_val=data_keep_in_ram,
            noise_type=noise_type,
            noise_params=noise_params,
        )

        # -----------------------------
        # Model (single process)
        # -----------------------------
        # VAE.forward() returns: (Yhat, [mu, logvar])
        model = create_model_vae(
            input_channels=3,
            img_size=patch_size,
            latent_dim=latent_dim,
            hidden_dims=hidden_dims,
            m_compile=False,  # compile AFTER .to(device_t)
        )
        model = model.to(device_t)

        if m_compile_train:
            try:
                model = torch.compile(model, backend="eager")
            except Exception:
                pass

        # -----------------------------
        # Ensure teacher model is on same device (if used)
        # -----------------------------
        if getattr(loss_fn, "model", None) is not None:
            try:
                loss_fn.model.to(device_t)
                loss_fn.model.eval()
                for p in loss_fn.model.parameters():
                    p.requires_grad_(False)
            except Exception:
                pass

        # -----------------------------
        # Optimizer & scheduler
        # -----------------------------
        if scheduler_option == 1:
            base_lr, optimizer, lips = create_optimizer(
                model,
                optimizer_option=optimizer_option,
                scheduler_option=1,
                base_lr=base_lr,
                weight_decay=l2_weight_decay,
            )
            scheduler = None
        else:
            base_lr, optimizer, scheduler = create_optimizer(
                model,
                optimizer_option=optimizer_option,
                scheduler_option=scheduler_option,
                base_lr=base_lr,
                weight_decay=l2_weight_decay,
            )
            lips = None

        # -----------------------------
        # Training loop
        # -----------------------------
        best_val = float("inf")
        last_report_value = float("inf")

        for epoch in range(1, epochs + 1):
            model.train()

            if lips is not None:
                lips.update(model, optimizer, epoch=epoch)

            loss3_coeff = kl_beta(epoch, kl_warmup_ep, kl_beta_final)

            show_epoch_info = (epoch % pbar_interval == 0) or (epoch == epochs)
            pbar = tqdm(
                train_loader,
                desc=f"Trial {trial.number} | Epoch {epoch}/{epochs}",
                disable=not show_epoch_info,
                leave=False,
            )

            train_sum = 0.0
            train_cnt = 0

            for i, (X, Y) in enumerate(pbar):
                X = X.to(device_t, non_blocking=True)
                Y = Y.to(device_t, non_blocking=True)

                if i == 0 and prnt_model_dtype:
                    print("Training --------")
                    print("X dtype:", X.dtype, "device:", X.device)
                    print("Y dtype:", Y.dtype, "device:", Y.device)
                    p0 = next(model.parameters())
                    print("student model param dtype:", p0.dtype, "device:", p0.device)
                    if getattr(loss_fn, "model", None) is not None:
                        t0 = next(loss_fn.model.parameters())
                        print("teacher model param dtype:", t0.dtype, "device:", t0.device)

                optimizer.zero_grad(set_to_none=True)

                if use_amp_runtime:
                    with torch.autocast(device_type=device_t.type, dtype=torch.float16, enabled=True):
                        Yhat, [mu, logvar] = model(X)
                        loss = loss_fn(
                            Yhat=Yhat,
                            Y=Y,
                            X=X,
                            loss1_coeff=loss1_coeff,
                            loss2_coeff=loss2_coeff,
                            loss3_coeff=loss3_coeff,
                            mu=mu,
                            logvar=logvar,
                            prnt_model_dtype=prnt_model_dtype,
                            use_amp=True,
                        )
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    Yhat, [mu, logvar] = model(X)
                    loss = loss_fn(
                        Yhat=Yhat,
                        Y=Y,
                        X=X,
                        loss1_coeff=loss1_coeff,
                        loss2_coeff=loss2_coeff,
                        loss3_coeff=loss3_coeff,
                        mu=mu,
                        logvar=logvar,
                        prnt_model_dtype=prnt_model_dtype,
                    )
                    loss.backward()
                    optimizer.step()

                bs = int(Y.shape[0])
                train_sum += float(loss.item()) * bs
                train_cnt += bs

                if show_epoch_info:
                    pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

            train_loss = train_sum / max(1, train_cnt)

            # -----------------------------
            # Validation
            # -----------------------------
            val_loss = None
            if val_loader is not None:
                model.eval()
                val_sum = 0.0
                val_cnt = 0

                with torch.inference_mode():
                    for j, (X, Y) in enumerate(val_loader):
                        X = X.to(device_t, non_blocking=True)
                        Y = Y.to(device_t, non_blocking=True)

                        if j == 0 and prnt_model_dtype:
                            print("Validation --------")
                            print("X dtype:", X.dtype, "device:", X.device)
                            print("Y dtype:", Y.dtype, "device:", Y.device)
                            p0 = next(model.parameters())
                            print("student model param dtype:", p0.dtype, "device:", p0.device)
                            if getattr(loss_fn, "model", None) is not None:
                                t0 = next(loss_fn.model.parameters())
                                print("teacher model param dtype:", t0.dtype, "device:", t0.device)

                        if use_amp_runtime:
                            with torch.autocast(device_type=device_t.type, dtype=torch.float16, enabled=True):
                                Yhat, [mu, logvar] = model(X)
                                vloss = loss_fn(
                                    Yhat=Yhat,
                                    Y=Y,
                                    X=X,
                                    loss1_coeff=loss1_coeff,
                                    loss2_coeff=loss2_coeff,
                                    loss3_coeff=loss3_coeff,
                                    mu=mu,
                                    logvar=logvar,
                                    prnt_model_dtype=prnt_model_dtype,
                                )
                        else:
                            Yhat, [mu, logvar] = model(X)
                            vloss = loss_fn(
                                Yhat=Yhat,
                                Y=Y,
                                X=X,
                                loss1_coeff=loss1_coeff,
                                loss2_coeff=loss2_coeff,
                                loss3_coeff=loss3_coeff,
                                mu=mu,
                                logvar=logvar,
                                prnt_model_dtype=prnt_model_dtype,
                            )

                        bs = int(Y.shape[0])
                        val_sum += float(vloss.item()) * bs
                        val_cnt += bs

                val_loss = val_sum / max(1, val_cnt)
                best_val = min(best_val, val_loss)

            # -----------------------------
            # Scheduler step
            # -----------------------------
            metric_for_sched = val_loss if val_loss is not None else train_loss
            if scheduler is not None:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(metric_for_sched)
                else:
                    scheduler.step()

            # -----------------------------
            # Report to Optuna / prune
            # -----------------------------
            report_value = val_loss if val_loss is not None else train_loss
            last_report_value = float(report_value)

            trial.report(last_report_value, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        print(f"=== Trial {trial.number} successful. Best val: {best_val:.6f} ===\n")
        return best_val if val_loader is not None else last_report_value

    except optuna.TrialPruned:
        raise

    except Exception as e:
        try:
            print(f"train_loss(last): {train_loss}")
        except Exception:
            pass
        try:
            print(f"val_loss(last): {val_loss}")
        except Exception:
            pass

        print(f"Trial {trial.number} failed due to error: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float("inf")

### FCN

#### V3

In [32]:
# ============================================================
# FCN with DLR + TV (Optuna objective: SINGLE PROCESS ONLY)
# - CPU or single-GPU only (NO DP/DDP here)
# - Compiles AFTER .to(device)
# - Handles optional teacher model device placement
# ============================================================
@timer
def objective_fcn3(
    trial: optuna.trial.Trial,
    loss_fn_coeff=None,            # dict or None
    pbar_interval: int = pbar_interval,
    epochs: int = epochs,
    data_loader_option: int = 0,
    tv_coeff=None,
    prnt_model_dtype: bool = False,
    data_keep_in_ram: bool = data_keep_in_ram,
    **kargs
):
    print(f"\n=== Starting FCN Trial {trial.number} ===")

    # -----------------------------
    # Normalize coeff dict
    # -----------------------------
    if loss_fn_coeff is None:
        loss_fn_coeff = {}
    a_fixed = loss_fn_coeff.get("a", None)
    b_fixed = loss_fn_coeff.get("b", None)
    c_fixed = loss_fn_coeff.get("c", None)
    d_fixed = loss_fn_coeff.get("d", None)
    e_fixed = loss_fn_coeff.get("e", None)
    dlr_model = loss_fn_coeff.get("dlr_model", None)

    # -----------------------------
    # Tune/assign loss coefficients (a..e)
    # -----------------------------
    a = float(a_fixed) if a_fixed is not None else trial.suggest_float("a", 1.0, 2.0, log=True)
    b = float(b_fixed) if b_fixed is not None else 0.0
    c = float(c_fixed) if c_fixed is not None else trial.suggest_float("c", 1e-4, 1e-3, log=True)
    d = float(d_fixed) if d_fixed is not None else trial.suggest_float("d", 1.0, 2.0, log=True)
    e = float(e_fixed) if e_fixed is not None else trial.suggest_float("e", 1.0, 2.0, log=True)

    # -----------------------------
    # Build loss function
    # -----------------------------
    loss_fn = LossFnCust(a=a, b=b, c=c, d=d, e=e, model=dlr_model)

    # -----------------------------
    # Fixed knobs
    # -----------------------------
    sigma_range = (0.0, 100.0)
    blur_prob = 0.0
    num_workers = int(kargs.get("num_workers", 0))
    extra_edges = ()
    edge_from = "clean"
    blur_kernels = (3, 5)
    blur_sigma_rng = (0.5, 2.0)

    # Dataloader noise controls
    noise_type = str(kargs.get("noise_type", "gaussian")).lower()
    noise_params = dict(kargs.get("noise_params", {}))

    # Runtime: CPU or single-GPU
    force_device = str(kargs.get("force_device", "auto")).lower()  # "auto" | "cpu" | "cuda"
    if force_device == "cpu":
        device_t = torch.device("cpu")
    elif force_device in ("cuda", "gpu"):
        device_t = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    else:
        device_t = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # AMP decision must follow device_t
    use_amp_req = bool(kargs.get("use_amp", True))
    use_amp_runtime = bool(use_amp_req) and (device_t.type == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp_runtime)

    # Compile toggle (define locally so no NameError)
    m_compile_train = bool(kargs.get("m_compile_train", False))

    # Safety: init locals used in exception handler
    train_loss = float("nan")
    val_loss = None

    try:
        # -----------------------------
        # Suggested knobs (architecture + training)
        # -----------------------------
        batch_size = trial.suggest_categorical("batch_size", [8, 16, 24])
        patch_size = trial.suggest_categorical("patch_size", [64, 128, 256])
        base_lr = trial.suggest_float("base_lr", 1e-4, 5e-3, log=True)
        l2_weight_decay = trial.suggest_float("l2_weight_decay", 1e-6, 1e-2, log=True)
        optimizer_option = trial.suggest_categorical("optimizer_option", [0, 1, 2])
        scheduler_option = trial.suggest_categorical("scheduler_option", [0, 1, 4])
        model_width = trial.suggest_categorical("model_width", [64, 128, 256, 512])
        model_depth = trial.suggest_categorical("model_depth", [5, 10, 15, 20])

        # TV coefficient
        tv_coeff_final = float(tv_coeff) if tv_coeff is not None else trial.suggest_float(
            "tv_coeff", 1e-6, 1e-3, log=True
        )

        # Loss mixing coefficients
        loss1_coeff = trial.suggest_float("loss1_coeff", 1.0, 2.0, log=True)
        loss2_coeff = trial.suggest_float("loss2_coeff", 0.0, 1.0) if dlr_model is not None else 0.0

        # -----------------------------
        # Datasets & loaders
        # -----------------------------
        train_loader, val_loader = data_loader(
            data_loader_option=data_loader_option,
            batch_size=batch_size,
            patch_size=patch_size,
            sigma_range=sigma_range,
            blur_prob=blur_prob,
            num_workers=num_workers,
            extra_edges=extra_edges,
            edge_from=edge_from,
            canny_low=200,
            canny_high=255,
            sobel_ksize=9,
            blur_kernels=blur_kernels,
            blur_sigma_rng=blur_sigma_rng,
            keep_in_ram_train=data_keep_in_ram,
            keep_in_ram_val=data_keep_in_ram,
            noise_type=noise_type,
            noise_params=noise_params,
        )

        # -----------------------------
        # Model
        # -----------------------------
        model, in_ch = create_model_fcn(
            width=model_width,
            depth=model_depth,
            extra_edges=extra_edges,
            m_compile=False,  # compile AFTER .to(device_t)
        )
        model = model.to(device_t)

        if m_compile_train:
            try:
                model = torch.compile(model, backend="eager")
            except Exception:
                pass

        # -----------------------------
        # Ensure teacher model is on same device (if used)
        # -----------------------------
        if getattr(loss_fn, "model", None) is not None:
            try:
                loss_fn.model.to(device_t)
                loss_fn.model.eval()
                for p in loss_fn.model.parameters():
                    p.requires_grad_(False)
            except Exception:
                pass

        # -----------------------------
        # Optimizer & scheduler
        # -----------------------------
        if scheduler_option == 1:
            base_lr, optimizer, lips = create_optimizer(
                model,
                optimizer_option=optimizer_option,
                scheduler_option=1,
                base_lr=base_lr,
                weight_decay=l2_weight_decay,
            )
            scheduler = None
        else:
            base_lr, optimizer, scheduler = create_optimizer(
                model,
                optimizer_option=optimizer_option,
                scheduler_option=scheduler_option,
                base_lr=base_lr,
                weight_decay=l2_weight_decay,
            )
            lips = None

        # -----------------------------
        # Training loop
        # -----------------------------
        best_val = float("inf")
        last_report_value = float("inf")

        for epoch in range(1, epochs + 1):
            model.train()

            if lips is not None:
                lips.update(model, optimizer, epoch=epoch)

            show_epoch_info = (epoch % pbar_interval == 0) or (epoch == epochs)
            pbar = tqdm(
                train_loader,
                desc=f"FCN Trial {trial.number} | Epoch {epoch}/{epochs}",
                disable=not show_epoch_info,
                leave=False,
            )

            train_sum = 0.0
            train_cnt = 0

            for i, (X, Y) in enumerate(pbar):
                X = X.to(device_t, non_blocking=True)
                Y = Y.to(device_t, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                if i == 0 and prnt_model_dtype:
                    p0 = next(model.parameters())
                    print("Training --------")
                    print("X dtype:", X.dtype, "device:", X.device)
                    print("Y dtype:", Y.dtype, "device:", Y.device)
                    print("student model param dtype:", p0.dtype, "device:", p0.device)
                    if getattr(loss_fn, "model", None) is not None:
                        t0 = next(loss_fn.model.parameters())
                        print("teacher model param dtype:", t0.dtype, "device:", t0.device)

                if use_amp_runtime:
                    with torch.autocast(device_type=device_t.type, dtype=torch.float16, enabled=True):
                        Yhat = model(X)
                        base_loss = loss_fn(
                            Yhat=Yhat,
                            Y=Y,
                            X=X,
                            loss1_coeff=loss1_coeff,
                            loss2_coeff=loss2_coeff,
                            prnt_model_dtype=prnt_model_dtype,
                        )
                        tv_loss = total_variation_isotropic(Yhat)
                        loss = base_loss + tv_coeff_final * tv_loss

                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    Yhat = model(X)
                    base_loss = loss_fn(
                        Yhat=Yhat,
                        Y=Y,
                        X=X,
                        loss1_coeff=loss1_coeff,
                        loss2_coeff=loss2_coeff,
                        prnt_model_dtype=prnt_model_dtype,
                    )
                    tv_loss = total_variation_isotropic(Yhat)
                    loss = base_loss + tv_coeff_final * tv_loss

                    loss.backward()
                    optimizer.step()

                bs = int(Y.shape[0])
                train_sum += float(loss.item()) * bs
                train_cnt += bs

                if show_epoch_info:
                    pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

            train_loss = train_sum / max(1, train_cnt)

            # -----------------------------
            # Validation
            # -----------------------------
            val_loss = None
            if val_loader is not None:
                model.eval()
                val_sum = 0.0
                val_cnt = 0

                with torch.inference_mode():
                    for j, (X, Y) in enumerate(val_loader):
                        X = X.to(device_t, non_blocking=True)
                        Y = Y.to(device_t, non_blocking=True)

                        if j == 0 and prnt_model_dtype:
                            p0 = next(model.parameters())
                            print("Validation --------")
                            print("X dtype:", X.dtype, "device:", X.device)
                            print("Y dtype:", Y.dtype, "device:", Y.device)
                            print("student model param dtype:", p0.dtype, "device:", p0.device)
                            if getattr(loss_fn, "model", None) is not None:
                                t0 = next(loss_fn.model.parameters())
                                print("teacher model param dtype:", t0.dtype, "device:", t0.device)

                        if use_amp_runtime:
                            with torch.autocast(device_type=device_t.type, dtype=torch.float16, enabled=True):
                                Yhat = model(X).clamp(0, 1)
                                base_vloss = loss_fn(
                                    Yhat=Yhat,
                                    Y=Y,
                                    X=X,
                                    loss1_coeff=loss1_coeff,
                                    loss2_coeff=loss2_coeff,
                                    prnt_model_dtype=prnt_model_dtype,
                                )
                                tv_vloss = total_variation_isotropic(Yhat)
                                vloss = base_vloss + tv_coeff_final * tv_vloss
                        else:
                            Yhat = model(X).clamp(0, 1)
                            base_vloss = loss_fn(
                                Yhat=Yhat,
                                Y=Y,
                                X=X,
                                loss1_coeff=loss1_coeff,
                                loss2_coeff=loss2_coeff,
                                prnt_model_dtype=prnt_model_dtype,
                            )
                            tv_vloss = total_variation_isotropic(Yhat)
                            vloss = base_vloss + tv_coeff_final * tv_vloss

                        bs = int(Y.shape[0])
                        val_sum += float(vloss.item()) * bs
                        val_cnt += bs

                val_loss = val_sum / max(1, val_cnt)
                best_val = min(best_val, val_loss)

            # -----------------------------
            # Scheduler
            # -----------------------------
            metric_for_sched = val_loss if val_loss is not None else train_loss
            if scheduler is not None:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(metric_for_sched)
                else:
                    scheduler.step()

            # -----------------------------
            # Report to Optuna + prune
            # -----------------------------
            report_value = val_loss if val_loss is not None else train_loss
            last_report_value = float(report_value)

            trial.report(last_report_value, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        print(f"=== FCN Trial {trial.number} successful. Best val: {best_val:.6f} ===\n")
        return best_val if val_loader is not None else last_report_value

    except optuna.TrialPruned:
        raise

    except Exception as e:
        print(f"train_loss(last): {train_loss}")
        print(f"val_loss(last): {val_loss}")
        print(f"FCN Trial {trial.number} failed due to error: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float("inf")


### Optuna Helper Functions

In [33]:
from optuna.trial import TrialState

#=================================
def _ordered_key(d: dict):
    # stable, hashable-ish key for a param dict
    return tuple(sorted(d.items()))

#=================================
def make_retry_failed_callback(max_retries_per_config: int = 1):
    """Create a callback that enqueues a mutated param set when a trial FAILs."""
    seen_retries = {}  # counts retries per original param set

    # helper to nudge params to safer values
    def mutate(params: dict) -> dict:
        newp = dict(params)

        # Safer batch sizes (prefer smaller)
        if "batch_size" in newp:
            ladder = [8, 16, 24]  # your search space
            try:
                idx = ladder.index(int(newp["batch_size"]))
                if idx > 0:
                    newp["batch_size"] = ladder[idx - 1]
            except Exception:
                newp["batch_size"] = 8

        # Safer patch sizes
        if "patch_size" in newp:
            ladder = [64, 128, 256]
            try:
                idx = ladder.index(int(newp["patch_size"]))
                if idx > 0:
                    newp["patch_size"] = ladder[idx - 1]
            except Exception:
                newp["patch_size"] = 64

        # Lower LR (cap to range)
        if "base_lr" in newp:
            newp["base_lr"] = max(1e-5, float(newp["base_lr"]) * 0.5)

        # Prefer AdamW on retry (robust default)
        if "optimizer_option" in newp:
            # 0=AdamW, 1=Adam, 2=SGD (your mapping)
            if int(newp["optimizer_option"]) == 2:
                newp["optimizer_option"] = 0

        # Avoid Lipschitz scheduler on retry; use Plateau instead
        if "scheduler_option" in newp:
            # 0,1,4 in your space (1=Lipschitz, 4=Plateau)
            if int(newp["scheduler_option"]) == 1:
                newp["scheduler_option"] = 4

        return newp

    def callback(study, trial):
        if trial.state != TrialState.FAIL:
            return

        key = _ordered_key(trial.params)
        count = seen_retries.get(key, 0)

        if count >= max_retries_per_config:
            # already retried this config enough times
            return

        seen_retries[key] = count + 1
        new_params = mutate(trial.params)

        print(f"↩️  Enqueue retry #{seen_retries[key]} for failed trial {trial.number}")
        print(f"    Original: {trial.params}")
        print(f"    Mutated : {new_params}")
        study.enqueue_trial(new_params)

    return callback


# ++++++++ run study +++++++++==================

# Start Versions =============

# Optuna DLR with Total Variation Loss regularizer

## VAE *****

### DLR VAE For Fruits

In [34]:
# Once ============
optuna_study_folder = model_folders[0] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
# --- Ensure output folder exists ---
os.makedirs(optuna_study_folder, exist_ok=True)                   # ✅ MANDATORY — creates directory for saving study DB

objective_opt = objective_vae3
data_loader_option = 0 # {0: Fruits, 1: Div2k}
train_dir_default = fruits_train
val_dir_default = fruits_val
#train_dir_default = train_y_dir_default
#val_dir_default = val_y_dir_default


prnt_model_dtype=False
load_dlr_model = False

In [35]:
std_n = 0
db_n = std_n
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,
        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [36]:
if run_optuna_vae_dlr:
        
    
    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [37]:
std_n = 1
db_n = std_n
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,
        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [38]:
if run_optuna_vae_dlr:
        
    
    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [39]:
std_n = 2
db_n = std_n
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,
        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [40]:
if run_optuna_vae_dlr:
        
    
    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [41]:
std_n = 3
db_n = std_n
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,
        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [42]:
if run_optuna_vae_dlr:
        
    
    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [43]:
std_n = 4
db_n = std_n
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,
        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [44]:
if run_optuna_vae_dlr:
        
    
    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [45]:
std_n = 5
db_n = std_n
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,
        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [46]:
if run_optuna_vae_dlr:
        
    
    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


## FCN *****

### DLR FCN for Fruits

In [34]:
# Once ============
optuna_study_folder = model_folders[1]  # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
# --- Ensure output folder exists ---
os.makedirs(optuna_study_folder, exist_ok=True)                   # ✅ MANDATORY — creates directory for saving study DB


objective_opt = objective_fcn3
data_loader_option = 0 # {0: Fruits, 1: Div2k}
train_dir_default = fruits_train
val_dir_default = fruits_val
#train_dir_default = train_y_dir_default
#val_dir_default = val_y_dir_default

tv_coeff = fcn_tv_coeff #  None means trial, or 0.0

prnt_model_dtype=False
load_dlr_model = False

In [35]:
optuna_study_folder

'optuna_studies_fcn_fruits'

In [209]:
std_n = 0
db_n = std_n+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None
dlr_model_ = fcn

loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [210]:
if run_optuna_fcn_dlr:
    
    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        tv_coeff = tv_coeff,
        prnt_model_dtype=prnt_model_dtype,
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-07 12:04:40,909] A new study created in RDB with name: study_charbonnier



=== Starting FCN Trial 0 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 2: SGD
[Init] Spectral L=8.336e-01 | LR set to 3.000e-03


[I 2026-01-07 12:22:10,768] Trial 0 finished with value: 0.06388964962500793 and parameters: {'batch_size': 8, 'patch_size': 256, 'base_lr': 0.004860575429265789, 'l2_weight_decay': 5.391954369912343e-06, 'optimizer_option': 2, 'scheduler_option': 1, 'model_width': 256, 'model_depth': 10, 'tv_coeff': 1.0045597088336484e-06, 'loss1_coeff': 1.04777527406707}. Best is trial 0 with value: 0.06388964962500793.


=== FCN Trial 0 successful. Best val: 0.063890 ===

[TIMER] objective_fcn3 finished in 00:17:29.82 (hh:mm:ss)

=== Starting FCN Trial 1 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-07 12:28:08,168] Trial 1 finished with value: 0.09352968633174896 and parameters: {'batch_size': 16, 'patch_size': 256, 'base_lr': 0.004604154592117724, 'l2_weight_decay': 0.005968580590766215, 'optimizer_option': 0, 'scheduler_option': 0, 'model_width': 64, 'model_depth': 15, 'tv_coeff': 9.73587799466893e-06, 'loss1_coeff': 1.5227791232951031}. Best is trial 0 with value: 0.06388964962500793.


...


=== Starting FCN Trial 48 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-07 15:46:40,679] Trial 48 pruned. 



=== Starting FCN Trial 49 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-07 15:46:45,615] Trial 49 pruned. 


Best Charbonnier params: {'batch_size': 8, 'patch_size': 128, 'base_lr': 0.00028839319474412573, 'l2_weight_decay': 9.84350928884856e-06, 'optimizer_option': 1, 'scheduler_option': 4, 'model_width': 128, 'model_depth': 5, 'tv_coeff': 1.483738690327813e-06, 'loss1_coeff': 1.0131981643014132}
Best Charbonnier value: 0.015670576849235937
Study saved to: sqlite:///optuna_studies_fcn_fruits/study_charbonnier.db
{'batch_size': 8, 'patch_size': 128, 'base_lr': 0.00028839319474412573, 'l2_weight_decay': 9.84350928884856e-06, 'optimizer_option': 1, 'scheduler_option': 4, 'model_width': 128, 'model_depth': 5, 'tv_coeff': 1.483738690327813e-06, 'loss1_coeff': 1.0131981643014132}


In [36]:
std_n = 1
db_n = std_n+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None

loss_fn_coeff = get_loss_fn_coeff(std_n, fcn)

In [ ]:
if run_optuna_fcn_dlr:
    
    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        tv_coeff = tv_coeff,
        prnt_model_dtype=prnt_model_dtype,
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-09 11:02:47,026] A new study created in RDB with name: study_ms_ssim



=== Starting FCN Trial 0 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 2: SGD


[I 2026-01-09 11:08:14,464] Trial 0 finished with value: 0.2308206409215927 and parameters: {'batch_size': 24, 'patch_size': 256, 'base_lr': 0.0003366353762964515, 'l2_weight_decay': 9.163305680526845e-06, 'optimizer_option': 2, 'scheduler_option': 0, 'model_width': 128, 'model_depth': 5, 'tv_coeff': 0.00014377086571141517, 'loss1_coeff': 1.0237334930972524}. Best is trial 0 with value: 0.2308206409215927.


=== FCN Trial 0 successful. Best val: 0.230821 ===

[TIMER] objective_fcn3 finished in 00:05:27.40 (hh:mm:ss)

=== Starting FCN Trial 1 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 0: AdamW
[Init] Spectral L=8.256e-01 | LR set to 3.000e-03


In [36]:
std_n = 2
db_n = std_n+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None

loss_fn_coeff = get_loss_fn_coeff(std_n, fcn)

In [37]:
if run_optuna_fcn_dlr:
    
    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        tv_coeff = tv_coeff,
        prnt_model_dtype=prnt_model_dtype,
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-11 12:53:18,049] A new study created in RDB with name: study_psnr



=== Starting FCN Trial 0 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-11 12:54:46,250] Trial 0 finished with value: -41.30207061767578 and parameters: {'batch_size': 16, 'patch_size': 128, 'base_lr': 0.00030148757587382077, 'l2_weight_decay': 0.0005330963748315548, 'optimizer_option': 0, 'scheduler_option': 4, 'model_width': 64, 'model_depth': 5, 'tv_coeff': 1.3144058105435743e-06, 'loss1_coeff': 1.3872742603510857}. Best is trial 0 with value: -41.30207061767578.


=== FCN Trial 0 successful. Best val: -41.302071 ===

[TIMER] objective_fcn3 finished in 00:01:28.16 (hh:mm:ss)

=== Starting FCN Trial 1 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-11 12:56:15,803] Trial 1 finished with value: -51.11772918701172 and parameters: {'batch_size': 16, 'patch_size': 128, 'base_lr': 0.0001345481433872227, 'l2_weight_decay': 0.0005596681435211841, 'optimizer_option': 1, 'scheduler_option': 0, 'model_width': 128, 'model_depth': 5, 'tv_coeff': 7.951272280895152e-05, 'loss1_coeff': 1.718167409829284}. Best is trial 1 with value: -51.11772918701172.


...


=== Starting FCN Trial 48 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-11 13:28:00,771] Trial 48 pruned. 



=== Starting FCN Trial 49 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-11 13:28:03,151] Trial 49 pruned. 


Best Charbonnier params: {'batch_size': 8, 'patch_size': 64, 'base_lr': 0.000205270767488497, 'l2_weight_decay': 4.376648617684922e-05, 'optimizer_option': 1, 'scheduler_option': 0, 'model_width': 128, 'model_depth': 5, 'tv_coeff': 0.0003575176763260585, 'loss1_coeff': 1.999258047852223}
Best Charbonnier value: -66.95445544903095
Study saved to: sqlite:///optuna_studies_fcn_fruits/study_psnr.db
{'batch_size': 8, 'patch_size': 64, 'base_lr': 0.000205270767488497, 'l2_weight_decay': 4.376648617684922e-05, 'optimizer_option': 1, 'scheduler_option': 0, 'model_width': 128, 'model_depth': 5, 'tv_coeff': 0.0003575176763260585, 'loss1_coeff': 1.999258047852223}


In [38]:
std_n = 3
db_n = std_n+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None

loss_fn_coeff = get_loss_fn_coeff(std_n, fcn)

In [39]:
if run_optuna_fcn_dlr:
    
    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        tv_coeff = tv_coeff,
        prnt_model_dtype=prnt_model_dtype,
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-11 13:28:03,501] A new study created in RDB with name: study_mse



=== Starting FCN Trial 0 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-11 13:31:35,887] Trial 0 finished with value: 0.01768316887319088 and parameters: {'batch_size': 24, 'patch_size': 128, 'base_lr': 0.0004280820402419135, 'l2_weight_decay': 0.0007748033957549193, 'optimizer_option': 1, 'scheduler_option': 4, 'model_width': 128, 'model_depth': 20, 'tv_coeff': 9.445172986916807e-06, 'loss1_coeff': 1.2931451415171245}. Best is trial 0 with value: 0.01768316887319088.


=== FCN Trial 0 successful. Best val: 0.017683 ===

[TIMER] objective_fcn3 finished in 00:03:32.36 (hh:mm:ss)

=== Starting FCN Trial 1 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 2: SGD


[I 2026-01-11 13:54:42,382] Trial 1 finished with value: 0.012254530563950539 and parameters: {'batch_size': 24, 'patch_size': 256, 'base_lr': 0.00029259147034735506, 'l2_weight_decay': 0.0005793185556255263, 'optimizer_option': 2, 'scheduler_option': 0, 'model_width': 512, 'model_depth': 5, 'tv_coeff': 1.5308020910320258e-06, 'loss1_coeff': 1.2966929294910645}. Best is trial 1 with value: 0.012254530563950539.


...


=== Starting FCN Trial 48 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-11 16:07:46,117] Trial 48 pruned. 



=== Starting FCN Trial 49 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 2: SGD


[I 2026-01-11 16:07:55,382] Trial 49 pruned. 


Best Charbonnier params: {'batch_size': 24, 'patch_size': 128, 'base_lr': 0.0005760569174265085, 'l2_weight_decay': 2.271245807267635e-05, 'optimizer_option': 0, 'scheduler_option': 0, 'model_width': 512, 'model_depth': 5, 'tv_coeff': 8.423410403318882e-05, 'loss1_coeff': 1.0377152445639615}
Best Charbonnier value: 0.0007552559254691005
Study saved to: sqlite:///optuna_studies_fcn_fruits/study_mse.db
{'batch_size': 24, 'patch_size': 128, 'base_lr': 0.0005760569174265085, 'l2_weight_decay': 2.271245807267635e-05, 'optimizer_option': 0, 'scheduler_option': 0, 'model_width': 512, 'model_depth': 5, 'tv_coeff': 8.423410403318882e-05, 'loss1_coeff': 1.0377152445639615}


In [40]:
std_n = 4
db_n = std_n+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None

loss_fn_coeff = get_loss_fn_coeff(std_n, fcn)

In [ ]:
if run_optuna_fcn_dlr:
    
    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        tv_coeff = tv_coeff,
        prnt_model_dtype=prnt_model_dtype,
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-11 16:07:55,711] A new study created in RDB with name: study_charbonnier_ms_ssim



=== Starting FCN Trial 0 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-11 16:09:27,609] Trial 0 finished with value: 0.19156478803891402 and parameters: {'a': 1.483274453517211, 'c': 0.0008408210952094616, 'batch_size': 8, 'patch_size': 64, 'base_lr': 0.001329607732214361, 'l2_weight_decay': 0.008006906596710836, 'optimizer_option': 1, 'scheduler_option': 4, 'model_width': 64, 'model_depth': 10, 'tv_coeff': 4.463736070261022e-05, 'loss1_coeff': 1.2851338244345638}. Best is trial 0 with value: 0.19156478803891402.


=== FCN Trial 0 successful. Best val: 0.191565 ===

[TIMER] objective_fcn3 finished in 00:01:31.87 (hh:mm:ss)

=== Starting FCN Trial 1 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 0: AdamW
[Init] Spectral L=9.612e-01 | LR set to 3.000e-03


[I 2026-01-11 16:11:33,478] Trial 1 finished with value: 0.10062865167856216 and parameters: {'a': 1.1528193176934047, 'c': 0.0003336639553614229, 'batch_size': 16, 'patch_size': 64, 'base_lr': 0.00018650659750697724, 'l2_weight_decay': 0.004911835967978918, 'optimizer_option': 0, 'scheduler_option': 1, 'model_width': 512, 'model_depth': 5, 'tv_coeff': 0.0004854617148808011, 'loss1_coeff': 1.3344110496504724}. Best is trial 1 with value: 0.10062865167856216.


=== FCN Trial 1 successful. Best val: 0.100629 ===

[TIMER] objective_fcn3 finished in 00:02:05.83 (hh:mm:ss)

=== Starting FCN Trial 2 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 2: SGD


In [36]:
std_n = 5
db_n = std_n+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = dlr_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None

loss_fn_coeff = get_loss_fn_coeff(std_n, fcn)

In [ ]:
if run_optuna_fcn_dlr:
    
    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        tv_coeff = tv_coeff,
        prnt_model_dtype=prnt_model_dtype,
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-13 11:55:42,918] A new study created in RDB with name: study_mse_ms_ssim



=== Starting FCN Trial 0 ===
[CleanImageFolder] Cached 240 images into RAM.
[CleanImageFolder] Cached 13 images into RAM.
[Optimizer] Using option 1: Adam


# Train DLR V3

## VAE on Fruits

In [61]:
# Once ============
optuna_study_folder = model_folders[0] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
data_loader_option = 0 # {0: Fruits, 1: Div2k}
train_dir_default = fruits_train
val_dir_default = fruits_val
#train_dir_default = train_y_dir_default
#val_dir_default = val_y_dir_default

train_epochs = train_epochs
load_dlr_model = False

In [62]:
##

In [63]:
if run_train_vae_dlr:
    
    
    std_n = 0
    db_n  = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if load_dlr_model:
        ckpt_path = best_models[std_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,

            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [64]:
if run_train_vae_dlr:
    
    
    std_n = 1
    db_n  = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if load_dlr_model:
        ckpt_path = best_models[std_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,

            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [65]:
if run_train_vae_dlr:
    
    
    std_n = 2
    db_n  = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if load_dlr_model:
        ckpt_path = best_models[std_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,

            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [66]:
if run_train_vae_dlr:
    
    
    std_n = 3
    db_n  = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if load_dlr_model:
        ckpt_path = best_models[std_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,

            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [67]:
if run_train_vae_dlr:
    
    
    std_n = 4
    db_n  = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if load_dlr_model:
        ckpt_path = best_models[std_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,

            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [68]:
if run_train_vae_dlr:
    
    
    std_n = 5
    db_n  = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if load_dlr_model:
        ckpt_path = best_models[std_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,

            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

## FCN on Fruits

In [34]:
# Once ============
optuna_study_folder = model_folders[1] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
data_loader_option = 0 # {0: Fruits, 1: Div2k}
train_dir_default = fruits_train
val_dir_default = fruits_val
#train_dir_default = train_y_dir_default
#val_dir_default = val_y_dir_default

train_epochs = train_epochs
load_dlr_model = False

In [70]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================
if run_train_fcn_dlr:
    
    std_n = 0
    db_n  = std_n + 6
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=fcn_tv_coeff,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt_fcn[1]

In [71]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================
if run_train_fcn_dlr:
    
    std_n = 1
    db_n  = std_n + 6
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=fcn_tv_coeff,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt_fcn[1]

In [72]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================
if run_train_fcn_dlr:
    
    std_n = 2
    db_n  = std_n + 6
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=fcn_tv_coeff,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt_fcn[1]

In [73]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================
if run_train_fcn_dlr:
    
    std_n = 3
    db_n  = std_n + 6
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=fcn_tv_coeff,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt_fcn[1]

In [37]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================
if run_train_fcn_dlr:
    
    std_n = 4
    db_n  = std_n + 6
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=fcn_tv_coeff,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt_fcn[1]

In [38]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================
if run_train_fcn_dlr:
    
    std_n = 5
    db_n  = std_n + 6
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=fcn_tv_coeff,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt_fcn[1]

# Optuna Main Model

## VAE *****

### Without DLR For Div2k

In [76]:
# Once ============
optuna_study_folder = model_folders[2] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
# --- Ensure output folder exists ---
os.makedirs(optuna_study_folder, exist_ok=True)                   # ✅ MANDATORY — creates directory for saving study DB

objective_opt = objective_vae3
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

prnt_model_dtype=False
load_dlr_model = False

In [77]:
std_n = 0
db_n = std_n+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_charbonnier/best.pt


In [78]:
if run_optuna_vae_main:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [79]:
std_n = 1
db_n = std_n+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_ms_ssim/best.pt


In [80]:
if run_optuna_vae_main:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [81]:
std_n = 2
db_n = std_n+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_psnr/best.pt


In [82]:
if run_optuna_vae_main:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [83]:
std_n = 3
db_n = std_n+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_mse/best.pt


In [84]:
if run_optuna_vae_main:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [85]:
std_n = 4
db_n = std_n+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_charbonnier_ms_ssim/best.pt


In [86]:
if run_optuna_vae_main:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [87]:
std_n = 5
db_n = std_n+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_mse_ms_ssim/best.pt


In [88]:
if run_optuna_vae_main:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


### With DLR For Div2k

In [89]:
# Once ============
optuna_study_folder = model_folders[4] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
# --- Ensure output folder exists ---
os.makedirs(optuna_study_folder, exist_ok=True)                   # ✅ MANDATORY — creates directory for saving study DB

objective_opt = objective_vae3
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

prnt_model_dtype=False
load_dlr_model = True

In [90]:
std_n = 0
db_n = std_n+12+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_charbonnier/best.pt


In [91]:
if run_optuna_vae_main_dlr:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-05 12:45:39,008] A new study created in RDB with name: study_charbonnier
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [32, 64, 128, 256] which is of type list.
  warnings.warn(message


=== Starting Trial 0 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distri

=== Trial 0 successful. Best val: 19370.412031 ===

[TIMER] objective_vae3 finished in 00:16:01.09 (hh:mm:ss)

=== Starting Trial 1 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-05 13:06:26,468] Trial 1 finished with value: 149224.255 and parameters: {'batch_size': 24, 'patch_size': 128, 'base_lr': 0.0031239259266664246, 'l2_weight_decay': 0.0001256837074006591, 'optimizer_option': 0, 'scheduler_option': 0, 'hidden_dims': [64, 128, 256], 'kl_beta_final': 0.0034778605284734366, 'kl_warmup_epochs': 4, 'loss1_coeff': 1.5578328224602298, 'loss2_coeff': 0.8748640228970318}. Best is trial 0 with value: 19370.41203125.


...

[I 2026-01-05 16:20:11,951] Trial 29 pruned. 



=== Starting Trial 30 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-05 16:33:30,425] Trial 30 finished with value: 0.1255773377418518 and parameters: {'batch_size': 16, 'patch_size': 256, 'base_lr': 0.00013656931210670936, 'l2_weight_decay': 2.156531563414698e-05, 'optimizer_option': 1, 'scheduler_option': 4, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.0005590290282796093, 'kl_warmup_epochs': 5, 'loss1_coeff': 1.0660357035240842, 'loss2_coeff': 0.07646054859981091}. Best is trial 25 with value: 0.12125649124383926.


=== Trial 30 successful. Best val: 0.125577 ===

[TIMER] objective_vae3 finished in 00:13:18.45 (hh:mm:ss)

=== Starting Trial 31 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


Trial 31 | Epoch 50/100:  22%|██████▊                        | 11/50 [00:01<00:06,  5.61it/s, loss=0.1040, lr=2.27e-06]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [92]:
std_n = 1
db_n = std_n+12+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_ms_ssim/best.pt


In [93]:
if run_optuna_vae_main_dlr:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-05 19:13:50,518] A new study created in RDB with name: study_ms_ssim



=== Starting Trial 0 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distri

=== Trial 0 successful. Best val: 384992.915000 ===

[TIMER] objective_vae3 finished in 00:15:25.81 (hh:mm:ss)

=== Starting Trial 1 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-05 19:31:55,524] Trial 1 finished with value: 1.2475773620605468 and parameters: {'batch_size': 32, 'patch_size': 64, 'base_lr': 0.004524499911837428, 'l2_weight_decay': 1.9034243422894083e-05, 'optimizer_option': 1, 'scheduler_option': 4, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.07478913265177968, 'kl_warmup_epochs': 10, 'loss1_coeff': 1.3884101356235221, 'loss2_coeff': 0.23904897659300828}. Best is trial 1 with value: 1.2475773620605468.


...


=== Starting Trial 48 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW
[Init] Spectral L=3.036e+00 | LR set to 3.000e-03


[I 2026-01-05 20:53:25,306] Trial 48 pruned. 



=== Starting Trial 49 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 2: SGD


[I 2026-01-05 20:54:09,515] Trial 49 pruned. 


Best Charbonnier params: {'batch_size': 24, 'patch_size': 256, 'base_lr': 0.00017477819838922107, 'l2_weight_decay': 3.6665738380493595e-06, 'optimizer_option': 2, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.00011484811395439416, 'kl_warmup_epochs': 0, 'loss1_coeff': 1.0011272934195645, 'loss2_coeff': 0.03024185114725586}
Best Charbonnier value: 0.2802977168560028
Study saved to: sqlite:///optuna_studies_vae_div2k_dlr/study_ms_ssim.db
{'batch_size': 24, 'patch_size': 256, 'base_lr': 0.00017477819838922107, 'l2_weight_decay': 3.6665738380493595e-06, 'optimizer_option': 2, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.00011484811395439416, 'kl_warmup_epochs': 0, 'loss1_coeff': 1.0011272934195645, 'loss2_coeff': 0.03024185114725586}


In [94]:
std_n = 2
db_n = std_n+12+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_psnr/best.pt


In [95]:
if run_optuna_vae_main_dlr:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-05 20:54:10,094] A new study created in RDB with name: study_psnr



=== Starting Trial 0 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distri

=== Trial 0 successful. Best val: -44.398282 ===

[TIMER] objective_vae3 finished in 00:05:12.41 (hh:mm:ss)

=== Starting Trial 1 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-05 21:12:44,870] Trial 1 finished with value: 77783.0653125 and parameters: {'batch_size': 16, 'patch_size': 256, 'base_lr': 0.0009644034311030605, 'l2_weight_decay': 0.00036888861133138465, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.005040717300454068, 'kl_warmup_epochs': 9, 'loss1_coeff': 1.5437419190873864, 'loss2_coeff': 0.5770976163054166}. Best is trial 0 with value: -44.39828216552734.


...


=== Starting Trial 48 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 2: SGD


[I 2026-01-05 22:39:28,633] Trial 48 pruned. 



=== Starting Trial 49 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-05 22:39:35,253] Trial 49 pruned. 


Best Charbonnier params: {'batch_size': 8, 'patch_size': 64, 'base_lr': 0.001220307124984542, 'l2_weight_decay': 1.0257840680539607e-05, 'optimizer_option': 1, 'scheduler_option': 0, 'hidden_dims': [64, 128, 256], 'kl_beta_final': 0.00015063660533295133, 'kl_warmup_epochs': 3, 'loss1_coeff': 1.9926926706074082, 'loss2_coeff': 0.9959654721767154}
Best Charbonnier value: -53.13966552734375
Study saved to: sqlite:///optuna_studies_vae_div2k_dlr/study_psnr.db
{'batch_size': 8, 'patch_size': 64, 'base_lr': 0.001220307124984542, 'l2_weight_decay': 1.0257840680539607e-05, 'optimizer_option': 1, 'scheduler_option': 0, 'hidden_dims': [64, 128, 256], 'kl_beta_final': 0.00015063660533295133, 'kl_warmup_epochs': 3, 'loss1_coeff': 1.9926926706074082, 'loss2_coeff': 0.9959654721767154}


In [96]:
std_n = 3
db_n = std_n+12+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_mse/best.pt


In [97]:
if run_optuna_vae_main_dlr:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-05 22:39:35,738] A new study created in RDB with name: study_mse



=== Starting Trial 0 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW
[Init] Spectral L=3.029e+00 | LR set to 3.000e-03


C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distri

=== Trial 0 successful. Best val: 9038.416797 ===

[TIMER] objective_vae3 finished in 00:04:18.66 (hh:mm:ss)

=== Starting Trial 1 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-05 22:48:42,352] Trial 1 finished with value: 0.06324818044900894 and parameters: {'batch_size': 16, 'patch_size': 128, 'base_lr': 0.0001878160803377596, 'l2_weight_decay': 2.0032617418692182e-05, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [64, 128, 256], 'kl_beta_final': 0.001163071972763976, 'kl_warmup_epochs': 8, 'loss1_coeff': 1.8437983618611458, 'loss2_coeff': 0.35336116016105956}. Best is trial 1 with value: 0.06324818044900894.


...

[I 2026-01-06 01:05:49,930] Trial 19 finished with value: 0.03302880093455315 and parameters: {'batch_size': 8, 'patch_size': 256, 'base_lr': 0.00019883109190764437, 'l2_weight_decay': 0.00880531729009614, 'optimizer_option': 1, 'scheduler_option': 4, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.00038160447753443155, 'kl_warmup_epochs': 0, 'loss1_coeff': 1.066783099554256, 'loss2_coeff': 0.4317094400888079}. Best is trial 11 with value: 0.027175487726926805.


=== Trial 19 successful. Best val: 0.033029 ===

[TIMER] objective_vae3 finished in 00:13:52.63 (hh:mm:ss)

=== Starting Trial 20 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 2: SGD


[I 2026-01-06 01:05:55,767] Trial 20 pruned. 



=== Starting Trial 21 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


Trial 21 | Epoch 50/100:  14%|████▏                         | 14/100 [00:01<00:06, 12.44it/s, loss=0.0153, lr=1.38e-04]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [98]:
std_n = 4
db_n = std_n+12+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_charbonnier_ms_ssim/best.pt


In [99]:
if run_optuna_vae_main_dlr:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-06 03:43:36,059] A new study created in RDB with name: study_charbonnier_ms_ssim



=== Starting Trial 0 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [32, 64, 128, 256] which is of type list.
  warnings.warn(message)
C:\Users\kec994\AppData\Local\anaconda3\envs\div2k\Lib\site-packages\optuna\distri

=== Trial 0 successful. Best val: 0.628073 ===

[TIMER] objective_vae3 finished in 00:02:31.63 (hh:mm:ss)

=== Starting Trial 1 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW
[Init] Spectral L=2.513e+00 | LR set to 3.000e-03


[I 2026-01-06 03:51:36,151] Trial 1 finished with value: 11.699693145751953 and parameters: {'a': 1.3538864761714169, 'c': 0.00010563334136624461, 'batch_size': 16, 'patch_size': 128, 'base_lr': 0.0005188180498808486, 'l2_weight_decay': 4.599937793003845e-06, 'optimizer_option': 0, 'scheduler_option': 1, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.001757480412264309, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.5384229228233772, 'loss2_coeff': 0.299341904814041}. Best is trial 0 with value: 0.6280725598335266.


...


=== Starting Trial 48 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-06 07:10:08,986] Trial 48 pruned. 



=== Starting Trial 49 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-06 07:10:20,629] Trial 49 pruned. 


Best Charbonnier params: {'a': 1.0436265273619474, 'c': 0.0002903419002096374, 'batch_size': 32, 'patch_size': 256, 'base_lr': 0.0002204587779902954, 'l2_weight_decay': 0.0009636628020501404, 'optimizer_option': 0, 'scheduler_option': 0, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.00025444530748831975, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.0020228129513342, 'loss2_coeff': 0.11201644506926087}
Best Charbonnier value: 0.1555773138999939
Study saved to: sqlite:///optuna_studies_vae_div2k_dlr/study_charbonnier_ms_ssim.db
{'a': 1.0436265273619474, 'c': 0.0002903419002096374, 'batch_size': 32, 'patch_size': 256, 'base_lr': 0.0002204587779902954, 'l2_weight_decay': 0.0009636628020501404, 'optimizer_option': 0, 'scheduler_option': 0, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.00025444530748831975, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.0020228129513342, 'loss2_coeff': 0.11201644506926087}


In [100]:
std_n = 5
db_n = std_n+12+12
dlr_n = std_n

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.



if load_dlr_model:    
    ckpt_path = best_models[dlr_n]
    vae, best_params_loaded, ckpt = load_trained_vae_dlr3(
        ckpt_path=ckpt_path,

        m_compile=m_compile_train,   # or True if you want compile after load
    )
    # Inference example
    vae.eval();
else:
    vae=None
dlr_model_ = vae
loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

print(f'Best DLR: {best_models[dlr_n]}')

Best DLR: optuna_studies_vae_fruits/study_mse_ms_ssim/best.pt


In [101]:
if run_optuna_vae_main_dlr:

    # objective_vae
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


[I 2026-01-06 07:10:21,146] A new study created in RDB with name: study_mse_ms_ssim



=== Starting Trial 0 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-06 07:26:54,062] Trial 0 finished with value: 18474.828125 and parameters: {'c': 0.00021355630894135015, 'e': 1.0928851882507513, 'batch_size': 8, 'patch_size': 256, 'base_lr': 0.000127966317702959, 'l2_weight_decay': 0.0007684307115517053, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [64, 128, 256], 'kl_beta_final': 0.001695005262752781, 'kl_warmup_epochs': 0, 'loss1_coeff': 1.6271781132806082, 'loss2_coeff': 0.03829291072583019}. Best is trial 0 with value: 18474.828125.


=== Trial 0 successful. Best val: 18474.828125 ===

[TIMER] objective_vae3 finished in 00:16:32.88 (hh:mm:ss)

=== Starting Trial 1 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 1: Adam


[I 2026-01-06 07:40:55,237] Trial 1 finished with value: 6.639351310729981 and parameters: {'c': 0.0005800119465735356, 'e': 1.8180174089207017, 'batch_size': 16, 'patch_size': 256, 'base_lr': 0.0016848610987359762, 'l2_weight_decay': 9.415393585266677e-06, 'optimizer_option': 1, 'scheduler_option': 0, 'hidden_dims': [16, 32, 64, 128, 256], 'kl_beta_final': 0.07350827617625588, 'kl_warmup_epochs': 2, 'loss1_coeff': 1.9869254276325743, 'loss2_coeff': 0.858260507132912}. Best is trial 1 with value: 6.639351310729981.


...

=== Trial 46 successful. Best val: 0.105610 ===

[TIMER] objective_vae3 finished in 00:13:46.29 (hh:mm:ss)

=== Starting Trial 47 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-06 09:19:58,826] Trial 47 pruned. 



=== Starting Trial 48 ===
[CleanImageFolder] Cached 800 images into RAM.
[CleanImageFolder] Cached 100 images into RAM.
[Optimizer] Using option 0: AdamW


[I 2026-01-06 09:33:52,248] Trial 48 finished with value: 0.08971289187669754 and parameters: {'c': 0.0002386493943344956, 'e': 1.291908115452928, 'batch_size': 16, 'patch_size': 256, 'base_lr': 0.00012478721746687202, 'l2_weight_decay': 0.0031304924727895995, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.0001352666447795807, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.7554441026749303, 'loss2_coeff': 0.005604487677530745}. Best is trial 48 with value: 0.08971289187669754.
[I 2026-01-06 09:47:49,267] Trial 49 finished with value: 0.09489645451307296 and parameters: {'c': 0.00022076599758224716, 'e': 1.2823016356787207, 'batch_size': 16, 'patch_size': 256, 'base_lr': 0.00012469222511647668, 'l2_weight_decay': 0.003110960852443143, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.00013403081187698577, 'kl_warmup_epochs': 6, 'loss1_coeff': 1.7566634099499359, 'loss2_coeff': 0.0978574424406

=== Trial 49 successful. Best val: 0.094896 ===

[TIMER] objective_vae3 finished in 00:13:56.99 (hh:mm:ss)
Best Charbonnier params: {'c': 0.0002386493943344956, 'e': 1.291908115452928, 'batch_size': 16, 'patch_size': 256, 'base_lr': 0.00012478721746687202, 'l2_weight_decay': 0.0031304924727895995, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.0001352666447795807, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.7554441026749303, 'loss2_coeff': 0.005604487677530745}
Best Charbonnier value: 0.08971289187669754
Study saved to: sqlite:///optuna_studies_vae_div2k_dlr/study_mse_ms_ssim.db
{'c': 0.0002386493943344956, 'e': 1.291908115452928, 'batch_size': 16, 'patch_size': 256, 'base_lr': 0.00012478721746687202, 'l2_weight_decay': 0.0031304924727895995, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.0001352666447795807, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.7554441026749303, 'loss2_coeff': 0.00

## FCN *****

### Without DLR For Div2k

In [102]:
# Once ============
optuna_study_folder = model_folders[3]  # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
# --- Ensure output folder exists ---
os.makedirs(optuna_study_folder, exist_ok=True)                   # ✅ MANDATORY — creates directory for saving study DB


objective_opt = objective_fcn3
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

tv_coeff = fcn_tv_coeff #  None means trial, or 0.0

prnt_model_dtype=False
load_dlr_model = False

In [103]:
std_n = 0
db_n = std_n+12+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None
dlr_model_ = fcn

loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [104]:
if run_optuna_fcn_main:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [105]:
std_n = 1
db_n = std_n+12+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None
dlr_model_ = fcn

loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [106]:
if run_optuna_fcn_main:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [107]:
std_n = 2
db_n = std_n+12+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None
dlr_model_ = fcn

loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [108]:
if run_optuna_fcn_main:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [109]:
std_n = 3
db_n = std_n+12+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None
dlr_model_ = fcn

loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [110]:
if run_optuna_fcn_main:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [111]:
std_n = 4
db_n = std_n+12+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None
dlr_model_ = fcn

loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [112]:
if run_optuna_fcn_main:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [113]:
std_n = 5
db_n = std_n+12+6
dlr_n = std_n+6

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the same extra_edges you used when training (if any)
extra_edges = ()  # or ("sobel", "canny"), etc.

if load_dlr_model:
    ckpt_path = best_models[dlr_n]
    fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
        ckpt_path=ckpt_path,

        extra_edges=extra_edges,
        m_compile=False,   # set True if you want compile after loading
    )
    # Inference example
    fcn.eval();
    
else:
    fcn=None
dlr_model_ = fcn

loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [114]:
if run_optuna_fcn_main:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


### With DLR For Div2k

In [115]:
# Once ============
optuna_study_folder = model_folders[5]  # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
# --- Ensure output folder exists ---
os.makedirs(optuna_study_folder, exist_ok=True)                   # ✅ MANDATORY — creates directory for saving study DB


objective_opt = objective_fcn3
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

tv_coeff = fcn_tv_coeff #  None means trial, or 0.0

prnt_model_dtype=False
load_dlr_model = True

In [116]:
if run_optuna_fcn_main_dlr:
    
    std_n = 0
    db_n = std_n+12+6+12
    dlr_n = std_n+6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Use the same extra_edges you used when training (if any)
    extra_edges = ()  # or ("sobel", "canny"), etc.
    
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,
    
            extra_edges=extra_edges,
            m_compile=False,   # set True if you want compile after loading
        )
        # Inference example
        fcn.eval();
        
    else:
        fcn=None
    dlr_model_ = fcn
    
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [117]:
if run_optuna_fcn_main_dlr:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [118]:
if run_optuna_fcn_main_dlr:
    
    std_n = 1
    db_n = std_n+12+6+12
    dlr_n = std_n+6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Use the same extra_edges you used when training (if any)
    extra_edges = ()  # or ("sobel", "canny"), etc.
    
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,
    
            extra_edges=extra_edges,
            m_compile=False,   # set True if you want compile after loading
        )
        # Inference example
        fcn.eval();
        
    else:
        fcn=None
    dlr_model_ = fcn
    
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [119]:
if run_optuna_fcn_main_dlr:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [120]:
if run_optuna_fcn_main_dlr:
    
    std_n = 2
    db_n = std_n+12+6+12
    dlr_n = std_n+6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Use the same extra_edges you used when training (if any)
    extra_edges = ()  # or ("sobel", "canny"), etc.
    
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,
    
            extra_edges=extra_edges,
            m_compile=False,   # set True if you want compile after loading
        )
        # Inference example
        fcn.eval();
        
    else:
        fcn=None
    dlr_model_ = fcn
    
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [121]:
if run_optuna_fcn_main_dlr:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [122]:
if run_optuna_fcn_main_dlr:
    
    std_n = 3
    db_n = std_n+12+6+12
    dlr_n = std_n+6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Use the same extra_edges you used when training (if any)
    extra_edges = ()  # or ("sobel", "canny"), etc.
    
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,
    
            extra_edges=extra_edges,
            m_compile=False,   # set True if you want compile after loading
        )
        # Inference example
        fcn.eval();
        
    else:
        fcn=None
    dlr_model_ = fcn
    
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [123]:
if run_optuna_fcn_main_dlr:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [124]:
if run_optuna_fcn_main_dlr:
    
    std_n = 4
    db_n = std_n+12+6+12
    dlr_n = std_n+6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Use the same extra_edges you used when training (if any)
    extra_edges = ()  # or ("sobel", "canny"), etc.
    
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,
    
            extra_edges=extra_edges,
            m_compile=False,   # set True if you want compile after loading
        )
        # Inference example
        fcn.eval();
        
    else:
        fcn=None
    dlr_model_ = fcn
    
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [125]:
if run_optuna_fcn_main_dlr:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


In [126]:
if run_optuna_fcn_main_dlr:
    
    std_n = 5
    db_n = std_n+12+6+12
    dlr_n = std_n+6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Use the same extra_edges you used when training (if any)
    extra_edges = ()  # or ("sobel", "canny"), etc.
    
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,
    
            extra_edges=extra_edges,
            m_compile=False,   # set True if you want compile after loading
        )
        # Inference example
        fcn.eval();
        
    else:
        fcn=None
    dlr_model_ = fcn
    
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)

In [127]:
if run_optuna_fcn_main_dlr:

    # objective_fcn
    import os                                                      # ✅ MANDATORY — needed for folder creation
    import optuna                                                  # ✅ MANDATORY — core optimization library
    from functools import partial                                  # ✅ MANDATORY — to partially bind objective args
    
    
    study_name=study_names[std_n]
    # --- Define persistent storage path ---
    storage_path = model_w_dlr_optuna_db[db_n]
    
    
    # --- Create (or load) the study ---
    study1 = optuna.create_study(
        direction="minimize",                                      # ✅ MANDATORY — minimize loss
        study_name=study_name,                            # ⚙️ OPTIONAL — name label for reuse/resume
        storage=storage_path,                                      # ✅ MANDATORY — connect to persistent DB
        load_if_exists=True,                                       # ⚙️ OPTIONAL — resume if study already exists
    )
    
    
    
    
    # --- Define the objective function ---
    objective1 = partial(
        objective_opt,                                                 # ✅ MANDATORY — user-defined training function
        loss_fn_coeff=loss_fn_coeff,
        pbar_interval=50,                                          # ⚙️ OPTIONAL — how often to show progress bar
        data_loader_option = data_loader_option,
        prnt_model_dtype=prnt_model_dtype,
    
    )
    
    
    # --- Create retry callback ---
    retry_cb = make_retry_failed_callback(max_retries_per_config=1) # ⚙️ OPTIONAL (but HIGHLY RECOMMENDED)
                                                                   # retries failed trials automatically with safer params
    
    
    # --- Run the optimization ---
    study1.optimize(
        objective1,                                                # ✅ MANDATORY — optimization target
        n_trials=n_trials,                                         # ✅ MANDATORY — total number of trials to run
        callbacks=[retry_cb],                                      # ⚙️ OPTIONAL — handle failed trials gracefully
        catch=(Exception,),                                        # ⚙️ OPTIONAL — prevents stopping on exceptions
    )
    
    
    # --- Print and persist results ---
    print("Best Charbonnier params:", study1.best_params)          # ✅ MANDATORY — show best hyperparameters
    print("Best Charbonnier value:", study1.best_value)            # ✅ MANDATORY — show best loss value
    print(f"Study saved to: {storage_path}")                       # ✅ MANDATORY — confirm where study is stored
    
    
    
    
    
    study1 = optuna.load_study(
        study_name=study_name,
        storage=storage_path,
    )
    print(study1.best_params)


# Train Model with DLR V3

## VAE without DLR on Div2k

In [128]:
# Once ============
optuna_study_folder = model_folders[2] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

train_epochs = train_epochs

In [129]:
if run_train_vae_main:
    
    std_n = 0
    db_n  = std_n+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [130]:
if run_train_vae_main:
    
    std_n = 1
    db_n  = std_n+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [131]:
if run_train_vae_main:
    
    std_n = 2
    db_n  = std_n+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [132]:
if run_train_vae_main:
    
    std_n = 3
    db_n  = std_n+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [133]:
if run_train_vae_main:
    
    std_n = 4
    db_n  = std_n+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

In [134]:
if run_train_vae_main:
    
    std_n = 5
    db_n  = std_n+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

## VAE with DLR on Div2k

In [192]:
# Once ============
optuna_study_folder = model_folders[4] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

train_epochs = train_epochs
load_dlr_model = True

In [193]:
if run_train_vae_main_dlr:
    
    std_n = 0
    db_n  = std_n+12+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

Best params: {'batch_size': 16, 'patch_size': 256, 'base_lr': 0.00012423238583380295, 'l2_weight_decay': 1.0439716430178904e-06, 'optimizer_option': 1, 'scheduler_option': 4, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.00041195142194944644, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.0005747100302107, 'loss2_coeff': 0.00021116916598141154}
[Optimizer] Using option 1: Adam

=== Final training with best params ===
  batch_size: 16
  patch_size: 256
  base_lr: 0.00012423238583380295
  l2_weight_decay: 1.0439716430178904e-06
  optimizer_option: 1
  scheduler_option: 4
  hidden_dims: [16, 32, 64, 128]
  kl_beta_final: 0.00041195142194944644
  kl_warmup_epochs: 7
  loss1_coeff: 1.0005747100302107
  loss2_coeff: 0.00021116916598141154
  mode: cuda | is_ddp=False rank=0/1 device=cuda:0



[FINAL VAE] Epoch 500/500: 100%|█████████████████████████████| 50/50 [00:09<00:00,  5.26it/s, loss=0.0933, lr=1.00e-06]



=== Final training complete ===
Best metric: 0.094006
Best checkpoint saved at: optuna_studies_vae_div2k_dlr\study_charbonnier\best.pt
Final model saved to optuna_studies_vae_div2k_dlr\study_charbonnier\final.pt
Training history saved to optuna_studies_vae_div2k_dlr\study_charbonnier\history.pt
[TIMER] train_dlr_vae3 finished in 01:28:08.70 (hh:mm:ss)
Final best checkpoint: ('optuna_studies_vae_div2k_dlr\\study_charbonnier\\best.pt', 5288.6990885999985)


In [194]:
if run_train_vae_main_dlr:
    
    std_n = 1
    db_n  = std_n+12+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

Best params: {'batch_size': 24, 'patch_size': 256, 'base_lr': 0.00017477819838922107, 'l2_weight_decay': 3.6665738380493595e-06, 'optimizer_option': 2, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.00011484811395439416, 'kl_warmup_epochs': 0, 'loss1_coeff': 1.0011272934195645, 'loss2_coeff': 0.03024185114725586}
[Optimizer] Using option 2: SGD

=== Final training with best params ===
  batch_size: 24
  patch_size: 256
  base_lr: 0.00017477819838922107
  l2_weight_decay: 3.6665738380493595e-06
  optimizer_option: 2
  scheduler_option: 4
  hidden_dims: [32, 64, 128, 256]
  kl_beta_final: 0.00011484811395439416
  kl_warmup_epochs: 0
  loss1_coeff: 1.0011272934195645
  loss2_coeff: 0.03024185114725586
  mode: cuda | is_ddp=False rank=0/1 device=cuda:0



[FINAL VAE] Epoch 500/500: 100%|█████████████████████████████| 33/33 [00:09<00:00,  3.57it/s, loss=0.2773, lr=1.00e-06]



=== Final training complete ===
Best metric: 0.214190
Best checkpoint saved at: optuna_studies_vae_div2k_dlr\study_ms_ssim\best.pt
Final model saved to optuna_studies_vae_div2k_dlr\study_ms_ssim\final.pt
Training history saved to optuna_studies_vae_div2k_dlr\study_ms_ssim\history.pt
[TIMER] train_dlr_vae3 finished in 01:25:20.28 (hh:mm:ss)
Final best checkpoint: ('optuna_studies_vae_div2k_dlr\\study_ms_ssim\\best.pt', 5120.28003970001)


In [195]:
if run_train_vae_main_dlr:
    
    std_n = 2
    db_n  = std_n+12+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

Best params: {'batch_size': 8, 'patch_size': 64, 'base_lr': 0.001220307124984542, 'l2_weight_decay': 1.0257840680539607e-05, 'optimizer_option': 1, 'scheduler_option': 0, 'hidden_dims': [64, 128, 256], 'kl_beta_final': 0.00015063660533295133, 'kl_warmup_epochs': 3, 'loss1_coeff': 1.9926926706074082, 'loss2_coeff': 0.9959654721767154}
[Optimizer] Using option 1: Adam

=== Final training with best params ===
  batch_size: 8
  patch_size: 64
  base_lr: 0.001220307124984542
  l2_weight_decay: 1.0257840680539607e-05
  optimizer_option: 1
  scheduler_option: 0
  hidden_dims: [64, 128, 256]
  kl_beta_final: 0.00015063660533295133
  kl_warmup_epochs: 3
  loss1_coeff: 1.9926926706074082
  loss2_coeff: 0.9959654721767154
  mode: cuda | is_ddp=False rank=0/1 device=cuda:0



[FINAL VAE] Epoch 500/500: 100%|█████████████████████████| 100/100 [00:05<00:00, 17.65it/s, loss=-32.6546, lr=1.22e-03]



=== Final training complete ===
Best metric: -36.634197
Best checkpoint saved at: optuna_studies_vae_div2k_dlr\study_psnr\best.pt
Final model saved to optuna_studies_vae_div2k_dlr\study_psnr\final.pt
Training history saved to optuna_studies_vae_div2k_dlr\study_psnr\history.pt
[TIMER] train_dlr_vae3 finished in 00:47:18.27 (hh:mm:ss)
Final best checkpoint: ('optuna_studies_vae_div2k_dlr\\study_psnr\\best.pt', 2838.2711532999965)


In [196]:
if run_train_vae_main_dlr:
    
    std_n = 3
    db_n  = std_n+12+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

Best params: {'batch_size': 8, 'patch_size': 256, 'base_lr': 0.00016711147647156888, 'l2_weight_decay': 0.005822212096029208, 'optimizer_option': 1, 'scheduler_option': 0, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.0007495173135080864, 'kl_warmup_epochs': 3, 'loss1_coeff': 1.0069800768173307, 'loss2_coeff': 0.010961439826814502}
[Optimizer] Using option 1: Adam

=== Final training with best params ===
  batch_size: 8
  patch_size: 256
  base_lr: 0.00016711147647156888
  l2_weight_decay: 0.005822212096029208
  optimizer_option: 1
  scheduler_option: 0
  hidden_dims: [16, 32, 64, 128]
  kl_beta_final: 0.0007495173135080864
  kl_warmup_epochs: 3
  loss1_coeff: 1.0069800768173307
  loss2_coeff: 0.010961439826814502
  mode: cuda | is_ddp=False rank=0/1 device=cuda:0



[FINAL VAE] Epoch 500/500: 100%|███████████████████████████| 100/100 [00:07<00:00, 12.78it/s, loss=0.0223, lr=1.67e-04]



=== Final training complete ===
Best metric: 0.018855
Best checkpoint saved at: optuna_studies_vae_div2k_dlr\study_mse\best.pt
Final model saved to optuna_studies_vae_div2k_dlr\study_mse\final.pt
Training history saved to optuna_studies_vae_div2k_dlr\study_mse\history.pt
[TIMER] train_dlr_vae3 finished in 01:27:14.19 (hh:mm:ss)
Final best checkpoint: ('optuna_studies_vae_div2k_dlr\\study_mse\\best.pt', 5234.192876999994)


In [197]:
if run_train_vae_main_dlr:
    
    std_n = 4
    db_n  = std_n+12+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

Best params: {'a': 1.0436265273619474, 'c': 0.0002903419002096374, 'batch_size': 32, 'patch_size': 256, 'base_lr': 0.0002204587779902954, 'l2_weight_decay': 0.0009636628020501404, 'optimizer_option': 0, 'scheduler_option': 0, 'hidden_dims': [16, 32, 64, 128], 'kl_beta_final': 0.00025444530748831975, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.0020228129513342, 'loss2_coeff': 0.11201644506926087}
[Optimizer] Using option 0: AdamW

=== Final training with best params ===
  a: 1.0436265273619474
  c: 0.0002903419002096374
  batch_size: 32
  patch_size: 256
  base_lr: 0.0002204587779902954
  l2_weight_decay: 0.0009636628020501404
  optimizer_option: 0
  scheduler_option: 0
  hidden_dims: [16, 32, 64, 128]
  kl_beta_final: 0.00025444530748831975
  kl_warmup_epochs: 7
  loss1_coeff: 1.0020228129513342
  loss2_coeff: 0.11201644506926087
  mode: cuda | is_ddp=False rank=0/1 device=cuda:0



[FINAL VAE] Epoch 500/500: 100%|█████████████████████████████| 25/25 [00:08<00:00,  2.94it/s, loss=0.1180, lr=2.20e-04]



=== Final training complete ===
Best metric: 0.097873
Best checkpoint saved at: optuna_studies_vae_div2k_dlr\study_charbonnier_ms_ssim\best.pt
Final model saved to optuna_studies_vae_div2k_dlr\study_charbonnier_ms_ssim\final.pt
Training history saved to optuna_studies_vae_div2k_dlr\study_charbonnier_ms_ssim\history.pt
[TIMER] train_dlr_vae3 finished in 01:19:08.71 (hh:mm:ss)
Final best checkpoint: ('optuna_studies_vae_div2k_dlr\\study_charbonnier_ms_ssim\\best.pt', 4748.707758899982)


In [198]:
if run_train_vae_main_dlr:
    
    std_n = 5
    db_n  = std_n+12+12
    dlr_n = std_n
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # If you used a teacher DLR model during Optuna:
    if data_loader_option == 1:
        ckpt_path = best_models[dlr_n]
        vae_teacher, best_params_loaded, ckpt = load_trained_vae_dlr3(
            ckpt_path=ckpt_path,
            device=device,
            m_compile=m_compile_train,
        )
        vae_teacher.eval()
    else:
        vae_teacher = None
    
    # Recreate the SAME loss_fn_coeff you used in objective_vae3
    loss_fn_coeff = get_loss_fn_coeff(std_n, vae_teacher)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ Study not found. Check study_name or path.")
        raise
    
    best_params = study.best_params
    print("Best params:", best_params)
    
    # Rebuild the final loss using best_params + loss_fn_coeff
    final_loss_fn = build_final_loss_fn3(best_params, loss_fn_coeff)
    
    best_model_folder = study_name
    
    #==================================
    best_ckpt = train_dlr_vae3(
        best_params=best_params,
        loss_fn=final_loss_fn,
        epochs=train_epochs,
        data_loader_option=data_loader_option,
        save_dir=os.path.join(optuna_study_folder, best_model_folder),
        pbar_interval=50,
    )
    print("Final best checkpoint:", best_ckpt)
    
    train_time[best_ckpt[0]]=best_ckpt[1]

Best params: {'c': 0.0002386493943344956, 'e': 1.291908115452928, 'batch_size': 16, 'patch_size': 256, 'base_lr': 0.00012478721746687202, 'l2_weight_decay': 0.0031304924727895995, 'optimizer_option': 0, 'scheduler_option': 4, 'hidden_dims': [32, 64, 128, 256], 'kl_beta_final': 0.0001352666447795807, 'kl_warmup_epochs': 7, 'loss1_coeff': 1.7554441026749303, 'loss2_coeff': 0.005604487677530745}
[Optimizer] Using option 0: AdamW

=== Final training with best params ===
  c: 0.0002386493943344956
  e: 1.291908115452928
  batch_size: 16
  patch_size: 256
  base_lr: 0.00012478721746687202
  l2_weight_decay: 0.0031304924727895995
  optimizer_option: 0
  scheduler_option: 4
  hidden_dims: [32, 64, 128, 256]
  kl_beta_final: 0.0001352666447795807
  kl_warmup_epochs: 7
  loss1_coeff: 1.7554441026749303
  loss2_coeff: 0.005604487677530745
  mode: cuda | is_ddp=False rank=0/1 device=cuda:0



[FINAL VAE] Epoch 500/500: 100%|█████████████████████████████| 50/50 [00:10<00:00,  4.83it/s, loss=0.0553, lr=1.00e-06]



=== Final training complete ===
Best metric: 0.043097
Best checkpoint saved at: optuna_studies_vae_div2k_dlr\study_mse_ms_ssim\best.pt
Final model saved to optuna_studies_vae_div2k_dlr\study_mse_ms_ssim\final.pt
Training history saved to optuna_studies_vae_div2k_dlr\study_mse_ms_ssim\history.pt
[TIMER] train_dlr_vae3 finished in 01:36:03.49 (hh:mm:ss)
Final best checkpoint: ('optuna_studies_vae_div2k_dlr\\study_mse_ms_ssim\\best.pt', 5763.489188300009)


In [199]:
print('Training VAE Completed')

Training VAE Completed


## FCN Without DLR on Div2k

In [143]:
# Once ============
optuna_study_folder = model_folders[3] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

train_epochs = train_epochs
load_dlr_model = False

In [144]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main:
    
    std_n = 0
    db_n  = std_n + 18
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [145]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main:
    
    std_n = 1
    db_n  = std_n + 18
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [146]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main:
    
    std_n = 2
    db_n  = std_n + 18
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [147]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main:
    
    std_n = 3
    db_n  = std_n + 18
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [148]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main:
    
    std_n = 4
    db_n  = std_n + 18
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [149]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main:
    
    std_n = 5
    db_n  = std_n + 18
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

## FCN With DLR on Div2k

In [150]:
# Once ============
optuna_study_folder = model_folders[5] # VAE = [0,2], FCN = [1,3], fruits = [0, 1], div2k  = [2,3] 
data_loader_option = 0 # {0: Fruits, 1: Div2k, 0 but div2k}
#train_dir_default = fruits_train
#val_dir_default = fruits_val
train_dir_default = train_y_dir_default
val_dir_default = val_y_dir_default

train_epochs = train_epochs
load_dlr_model = True

In [151]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main_dlr:
    
    std_n = 0
    db_n  = std_n + 18 + 12
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [152]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main_dlr:
    
    std_n = 1
    db_n  = std_n + 18 + 12
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [153]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main_dlr:
    
    std_n = 2
    db_n  = std_n + 18 + 12
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [154]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main_dlr:
    
    std_n = 3
    db_n  = std_n + 18 + 12
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [155]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main_dlr:
    
    std_n = 4
    db_n  = std_n + 18 + 12
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

In [156]:
# ============================================
# Final FCN training with best Optuna parameters
# ============================================

if run_train_fcn_main_dlr:
    
    std_n = 5
    db_n  = std_n + 18 + 12
    dlr_n = std_n + 6
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extra_edges = ()  # must match what you used during Optuna
    
    # --- Rebuild the teacher DLR model exactly as in the Optuna setup ---
    if load_dlr_model:
        ckpt_path = best_models[dlr_n]
        fcn_teacher, in_ch, best_params_fcn_loaded, ckpt_fcn = load_trained_fcn_dlr3(
            ckpt_path=ckpt_path,

            extra_edges=extra_edges,
            m_compile=False,
        )
        fcn_teacher.eval()
        fcn_teacher = fcn_teacher.to(device)   # <-- critical
        fcn_teacher.eval()
        for p in fcn_teacher.parameters():
            p.requires_grad_(False)
            
    else:
        fcn_teacher = None
    
    dlr_model_ = fcn_teacher
    loss_fn_coeff = get_loss_fn_coeff(std_n, dlr_model_)
    
    study_name   = study_names[std_n]
    storage_path = model_w_dlr_optuna_db[db_n]
    best_model_folder = study_name  # or any name you like
    
    try:
        study_fcn = optuna.load_study(study_name=study_name, storage=storage_path)
    except KeyError:
        print("❌ FCN study not found. Check study_name or storage_path.")
        study_fcn = None
    
    if study_fcn is not None:
        best_params_fcn = study_fcn.best_params
        print("Best FCN params:", best_params_fcn)
    
        # Rebuild loss exactly as in objective_fcn3
        final_loss_fn_fcn = build_final_loss_fn3(best_params_fcn, loss_fn_coeff)
    
        # Run final FCN training (using train_dlr_fcn3, not train_final_fcn)
        best_ckpt_fcn = train_dlr_fcn3(
            best_params=best_params_fcn,
            loss_fn=final_loss_fn_fcn,
            epochs=train_epochs,
            data_loader_option=data_loader_option,
            save_dir=os.path.join(optuna_study_folder, best_model_folder),
            pbar_interval=50,
            tv_coeff=None,          # will use best_params["tv_coeff"] if present
            prnt_model_dtype=False,
        )
    
        print("Final FCN best checkpoint:", best_ckpt_fcn)
    
        train_time[best_ckpt[0]]=best_ckpt[1]

# Benchmarking

In [200]:
print(train_time)

{'optuna_studies_vae_div2k_dlr\\study_charbonnier\\best.pt': 5288.6990885999985, 'optuna_studies_vae_div2k_dlr\\study_ms_ssim\\best.pt': 5120.28003970001, 'optuna_studies_vae_div2k_dlr\\study_psnr\\best.pt': 2838.2711532999965, 'optuna_studies_vae_div2k_dlr\\study_mse\\best.pt': 5234.192876999994, 'optuna_studies_vae_div2k_dlr\\study_charbonnier_ms_ssim\\best.pt': 4748.707758899982, 'optuna_studies_vae_div2k_dlr\\study_mse_ms_ssim\\best.pt': 5763.489188300009}


In [201]:
import pickle
import os

def save_pickle(obj, path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"[Saved] {path}")

# Example: train_time is a list
save_pickle(train_time, train_time_pkl)


[Saved] train_time_vae_dlr.pkl
